# 📈 Stock Price Prediction — ML Pipeline (Quant+ Neural Stack)
**Real-time prices · Indian stocks · Live Gradio feed · Zero paid APIs · Research-grounded quant layer**

> Trained on 2015–present data.
> The model's `predict_tomorrow()` uses yfinance which lags by 1 trading day (last close).
> The Gradio UI polls yfinance every 30 s for a live price ticker.

**(Step 12b / 12c / 13b):**
- 🧠 **Neural Meta-Learner** (Step 12b): small 3-layer MLP that learns to combine XGBoost, LightGBM, RF, and LSTM signals that replaces fixed-weight averaging with a trainable stacker
- 📊 **Accuracy Tracking Dashboard** (Step 12c): colour-coded 4-panel chart + console table showing RMSE, Directional Accuracy %, R², and IC for every model including the stacker
- 🎯 **Precision Stock Picker** (Step 13b): choose any ticker and training window length, get the highest-precision single prediction routed through the neural stacker
- Purged & embargoed CV for HPO (López de Prado 2018) + data-driven inverse-RMSE ensemble weights (Bates & Granger 1969)
- HMM regime detection (Hamilton 1989), GARCH(1,1) volatility (Bollerslev 1986), Avellaneda-Stoikov spread model (2008)
- Portfolio risk metrics: VaR/CVaR, Sortino, Calmar, Omega, tail ratio (Rockafellar & Uryasev 2000; Sortino & Price 1994)
- ADF stationarity + Ljung-Box residual autocorrelation diagnostics


In [1]:
!pip install --upgrade yfinance -q
!pip install ta xgboost plotly scikit-learn tensorflow seaborn \
             transformers torch optuna shap lightgbm feedparser fredapi \
             requests beautifulsoup4 groq gradio scipy langgraph langchain-groq \
             hmmlearn arch statsmodels -q   # QUANT+: regime detection (Hamilton 1989), GARCH (Bollerslev 1986), stationarity tests

import warnings; warnings.filterwarnings('ignore')
import logging; logging.getLogger('yfinance').setLevel(logging.CRITICAL)  # silence yfinance's internal error spam (e.g. delisted/renamed tickers)
import numpy as np, pandas as pd, yfinance as yf, json, os, joblib, time, threading
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import feedparser, requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from scipy import stats
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
    r2_score, confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb, lightgbm as lgb
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
print('✅ All libraries loaded')

✅ All libraries loaded


## ⚙️ Configuration

In [2]:
CONFIG = {
    'ticker'          : 'AAPL',
    'start_date'      : '2015-01-01',
    'end_date'        : datetime.today().strftime('%Y-%m-%d'),
    'test_size'       : 0.20,
    'look_back'       : 60,
    'forecast_days'   : 1,
    'random_state'    : 42,
    'n_folds'         : 5,
    'multistep_days'  : [1, 3, 5, 10],
    'initial_capital' : 10_000.0,
    'transaction_cost': 0.001,
    # Indian NSE tickers use .NS suffix — e.g. 'RELIANCE.NS', 'TCS.NS', 'INFY.NS'
    # BSE tickers use .BO suffix — e.g. 'RELIANCE.BO'
    'indian_tickers'  : ['RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','WIPRO.NS'],
    'ensemble_weights': {'xgb': 0.4, 'lgb': 0.4, 'rf': 0.2},  # FIX M4: fallback only — Step 5b overwrites
                                                                # this with data-driven inverse-RMSE weights
                                                                # (Bates & Granger 1969) computed from
                                                                # out-of-fold CV error. Kept as a safe default
                                                                # if that computation fails for any reason.
    # ── QUANT+ additions ──────────────────────────────────────────────────
    'cv_purge_days'   : 63,   # purge gap for HPO cross-validation (Lopez de Prado 2018, Ch.7) —
                              # matched to the longest *actively engineered* rolling window
                              # (63d momentum). NOTE: SMA_200 is a longer window still present
                              # in feature_cols; a 200-day purge would remove most of the usable
                              # CV data, so this is a deliberate, documented trade-off, not a
                              # claim of zero residual leakage. Full rationale in the Step 5 cell comment.
    'cv_embargo_days' : 21,   # embargo after each validation fold (Lopez de Prado 2018, Ch.7)
    'as_gamma'        : 0.1,  # Avellaneda-Stoikov (2008) risk-aversion parameter
    'as_kappa'        : 1.5,  # Avellaneda-Stoikov (2008) order-arrival-rate parameter
    'sentiment_tilt_cap': 0.0015,  # max ±0.15% inference-time sentiment overlay (Tetlock 2007) —
                                    # applied ONLY at live inference, never used as a training
                                    # feature (historical sentiment is sparse/fabricated zeros —
                                    # see Step 2 comment), and always shown as a separate number
                                    # next to the pure ML ensemble forecast, never silently merged.
}
np.random.seed(CONFIG['random_state'])
tf.keras.utils.set_random_seed(CONFIG['random_state'])  # seeds python random + numpy + TF together (more deterministic than tf.random.set_seed alone)
print(f"✅ Config loaded | {CONFIG['ticker']} | {CONFIG['start_date']} → {CONFIG['end_date']}")


✅ Config loaded | AAPL | 2015-01-01 → 2026-06-25


## 🔑 API Keys (All Free)
| Key | Source | Status |
|---|---|---|
| `GROQ_API_KEY` | console.groq.com | Optional — LLM report |
| `FRED_API_KEY` | fred.stlouisfed.org | Optional — macro features |
| `ALPHAVANTAGE_API_KEY` | alphavantage.co | Optional — news sentiment |
| `FINNHUB_API_KEY` | finnhub.io | Optional — company news |


In [3]:
try:
    from google.colab import userdata
    KEYS = {
        'groq'        : userdata.get('GROQ_API_KEY'),
        'fred'        : userdata.get('FRED_API_KEY'),
        'alphavantage': userdata.get('ALPHAVANTAGE_API_KEY'),
        'finnhub'     : userdata.get('FINNHUB_API_KEY'),
    }
    print('✅ Keys loaded from Colab Secrets')
except Exception:
    KEYS = {'groq': '', 'fred': '', 'alphavantage': '', 'finnhub': ''}
    print('⚠️  No Colab Secrets found — add keys manually to KEYS dict')
for k, v in KEYS.items():
    print(f'  {k:18s}: {"✅ active" if v else "❌ not set"}')


✅ Keys loaded from Colab Secrets
  groq              : ✅ active
  fred              : ✅ active
  alphavantage      : ✅ active
  finnhub           : ✅ active


# **api test**

In [4]:
import requests
from datetime import datetime, timedelta

print("=" * 55)
print("  API DIAGNOSTICS")
print("=" * 55)

# ── 1. ALPHA VANTAGE ──────────────────────────────────────
print("\n📊 ALPHA VANTAGE")
key_av = KEYS.get('alphavantage', '')
print(f"  Key loaded: {'✅ set' if key_av else '❌ not set — skipped'}")
if key_av:
    try:
        r = requests.get(
            f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
            f"&tickers={CONFIG['ticker']}&limit=5&apikey={key_av}",  # FIX: was hardcoded AAPL,
            timeout=15
        )
        d = r.json()
        if 'Information' in d:
            print(f"  ⚠️  Rate limit / plan wall: {d['Information'][:120]}")
        elif 'Note' in d:
            print(f"  ⚠️  API note: {d['Note'][:120]}")
        elif 'Error Message' in d:
            print(f"  ❌ Error: {d['Error Message']}")
        elif 'feed' in d:
            print(f"  ✅ Working — got {len(d['feed'])} articles")
        else:
            print(f"  ❓ Unknown response keys: {list(d.keys())}")
    except Exception as e:
        print(f"  ❌ Request failed: {e}")

# ── 2. FINNHUB ────────────────────────────────────────────
print("\n📡 FINNHUB")
key_fh = KEYS.get('finnhub', '')
print(f"  Key loaded: {'✅ set' if key_fh else '❌ not set — skipped'}")
if key_fh:
    # Test 1: live quote
    try:
        r = requests.get(
            f"https://finnhub.io/api/v1/quote?symbol={CONFIG['ticker']}&token={key_fh}",  # FIX: was hardcoded AAPL,
            timeout=10
        )
        d = r.json()
        if d.get('c', 0) > 0:
            print(f"  ✅ Quote API working — {CONFIG['ticker']} current price: ${d['c']:.2f}")
        elif 'error' in d:
            print(f"  ❌ Quote error: {d['error']}")
        else:
            print(f"  ⚠️  Quote returned zero/empty: {d}")
    except Exception as e:
        print(f"  ❌ Quote request failed: {e}")

    # Test 2: company news
    try:
        end_dt   = datetime.today().strftime('%Y-%m-%d')
        start_dt = (datetime.today() - timedelta(days=7)).strftime('%Y-%m-%d')
        r = requests.get(
            f"https://finnhub.io/api/v1/company-news?symbol={CONFIG['ticker']}"
            f"&from={start_dt}&to={end_dt}&token={key_fh}",  # FIX: was hardcoded AAPL,
            timeout=10
        )
        d = r.json()
        if isinstance(d, list):
            print(f"  ✅ News API working — got {len(d)} articles (last 7 days)")
        elif isinstance(d, dict) and 'error' in d:
            print(f"  ❌ News error: {d['error']}")
        else:
            print(f"  ⚠️  News unexpected response: {d}")
    except Exception as e:
        print(f"  ❌ News request failed: {e}")

# ── 3. FRED ───────────────────────────────────────────────
print("\n🏦 FRED (Federal Reserve)")
key_fr = KEYS.get('fred', '')
print(f"  Key loaded: {'✅ set' if key_fr else '❌ not set — skipped'}")
if key_fr:
    try:
        r = requests.get(
            f"https://api.stlouisfed.org/fred/series/observations"
            f"?series_id=DFF&limit=3&sort_order=desc"
            f"&api_key={key_fr}&file_type=json",
            timeout=10
        )
        d = r.json()
        if 'observations' in d:
            latest = d['observations'][0]
            print(f"  ✅ Working — Fed Funds Rate: {latest['value']}% (as of {latest['date']})")
        elif 'error_message' in d:
            print(f"  ❌ Error: {d['error_message']}")
        else:
            print(f"  ⚠️  Unexpected response keys: {list(d.keys())}")
    except Exception as e:
        print(f"  ❌ Request failed: {e}")

print("\n" + "=" * 55)
print("  Keys not set = feature silently skipped (not a crash)")
print("  All 3 are optional — yfinance covers the core data")
print("=" * 55)


  API DIAGNOSTICS

📊 ALPHA VANTAGE
  Key loaded: ✅ set
  ✅ Working — got 50 articles

📡 FINNHUB
  Key loaded: ✅ set
  ✅ Quote API working — AAPL current price: $293.08
  ✅ News API working — got 249 articles (last 7 days)

🏦 FRED (Federal Reserve)
  Key loaded: ✅ set
  ✅ Working — Fed Funds Rate: 3.63% (as of 2026-06-23)

  Keys not set = feature silently skipped (not a crash)
  All 3 are optional — yfinance covers the core data


## 📡 Live Price Feed Utilities
These functions power the **real-time ticker** in Gradio.  
`get_live_price()` hits yfinance's fast_info endpoint — no API key, no rate limit for normal use.  
For Indian stocks use `.NS` (NSE) or `.BO` (BSE) suffix.


In [5]:
# ─────────────────────────────────────────────────────────
# LIVE PRICE FEED — Agentic Multi-Source Architecture
# Sources tried in parallel; best result wins automatically.
#
# US stocks  → Finnhub REST (free key, finnhub.io)
# Indian .NS → NSE India public API (no key, real-time)
# Fallback   → yfinance 5-day daily (no key, always works)
# ─────────────────────────────────────────────────────────

import concurrent.futures

# ── Source 1: NSE India official API (Indian stocks only) ─────────────────────
def _fetch_nse(symbol_clean: str) -> dict:
    """
    Hits NSE India's own quote endpoint — same feed the NSE website uses.
    Real-time during market hours 9:15–15:30 IST, no API key needed.
    symbol_clean: base symbol without suffix e.g. 'INFY' not 'INFY.NS'
    """
    headers = {
        'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'Accept'         : '*/*',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer'        : 'https://www.nseindia.com',
    }
    session = requests.Session()
    # NSE blocks cold API calls — must hit homepage first to get cookies
    session.get('https://www.nseindia.com', headers=headers, timeout=10)
    r = session.get(
        f'https://www.nseindia.com/api/quote-equity?symbol={symbol_clean.upper()}',
        headers=headers, timeout=10
    )
    d = r.json()
    pi     = d['priceInfo']
    price  = float(pi['lastPrice'])
    prev   = float(pi['previousClose'])
    change = float(pi['change'])
    chgpct = float(pi['pChange'])
    try:
        vol = int(d['marketDeptOrderBook']['tradeInfo']['totalTradedVolume'])
    except Exception:
        vol = 0
    try:
        mstate = 'REGULAR' if d['marketStatus']['marketStatus'] == 'Open' else 'CLOSED'
    except Exception:
        mstate = 'CLOSED'
    return {
        'ticker'      : symbol_clean,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : vol,
        'currency'    : 'INR',
        'market_state': mstate,
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'NSE_API',
        'error'       : None,
    }

# ── Source 2: Finnhub REST (US stocks, free key) ──────────────────────────────
def _fetch_finnhub(ticker: str) -> dict:
    """
    Finnhub /quote endpoint — real-time US stock prices.
    Free tier: 60 calls/minute, no daily limit.
    """
    key = KEYS.get('finnhub', '')
    if not key:
        raise ValueError('No FINNHUB_API_KEY in secrets')
    r = requests.get(
        f'https://finnhub.io/api/v1/quote?symbol={ticker}&token={key}',
        timeout=8
    ).json()
    price = r.get('c', 0)
    prev  = r.get('pc', 0)
    if not price or price == 0:
        raise ValueError(f'Finnhub returned zero price for {ticker}')
    change  = price - prev
    chgpct  = (change / prev * 100) if prev else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : int(r.get('v', 0)),
        'currency'    : 'USD',
        'market_state': 'REGULAR' if r.get('d') is not None else 'CLOSED',
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'Finnhub',
        'error'       : None,
    }

# ── Source 3: yfinance 5-min intraday (universal fallback) ────────────────────
def _fetch_yfinance_intraday(ticker: str) -> dict:
    """
    5-minute candles over last 2 days.
    Works for US + Indian but may lag or fail — used as fallback only.
    """
    is_indian = ticker.endswith('.NS') or ticker.endswith('.BO')
    currency  = 'INR' if is_indian else 'USD'
    df = yf.download(ticker, period='2d', interval='5m',
                     auto_adjust=True, progress=False, timeout=10)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.dropna(subset=['Close'])
    if df.empty:
        raise ValueError('yfinance intraday empty')
    price  = float(df['Close'].iloc[-1])
    volume = int(df['Volume'].iloc[-1])
    # Separate today vs yesterday bars for accurate prev_close
    today_str  = datetime.utcnow().strftime('%Y-%m-%d')
    today_bars = df[df.index.strftime('%Y-%m-%d') == today_str]
    yest_bars  = df[df.index.strftime('%Y-%m-%d') != today_str]
    if not yest_bars.empty and not today_bars.empty:
        prev_close = float(yest_bars['Close'].iloc[-1])
    else:
        prev_close = float(df['Close'].iloc[-2]) if len(df) >= 2 else price
    # Infer market state from data freshness
    age_min = (datetime.utcnow() - df.index[-1].to_pydatetime().replace(tzinfo=None)).total_seconds() / 60
    mstate  = 'REGULAR (~5m delay)' if age_min < 20 else 'CLOSED/POST'
    change  = price - prev_close
    chgpct  = (change / prev_close * 100) if prev_close else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev_close, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : volume,
        'currency'    : currency,
        'market_state': mstate,
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'yfinance_intraday',
        'error'       : None,
    }

# ── Source 4: yfinance daily (last-resort fallback) ───────────────────────────
def _fetch_yfinance_daily(ticker: str) -> dict:
    """Last resort — always works but shows yesterday's close."""
    is_indian = ticker.endswith('.NS') or ticker.endswith('.BO')
    currency  = 'INR' if is_indian else 'USD'
    df = yf.download(ticker, period='5d', interval='1d',
                     auto_adjust=True, progress=False, timeout=10)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    closes    = df['Close'].dropna()
    price     = float(closes.iloc[-1])
    prev      = float(closes.iloc[-2]) if len(closes) >= 2 else price
    change    = price - prev
    chgpct    = (change / prev * 100) if prev else 0
    return {
        'ticker'      : ticker,
        'price'       : round(price, 2),
        'prev_close'  : round(prev, 2),
        'change'      : round(change, 2),
        'change_pct'  : round(chgpct, 3),
        'volume'      : int(df['Volume'].dropna().iloc[-1]),
        'currency'    : currency,
        'market_state': 'CLOSED (daily data — prev session close)',
        'arrow'       : '▲' if change >= 0 else '▼',
        'color_hint'  : 'green' if change >= 0 else 'red',
        'timestamp'   : datetime.now().strftime('%H:%M:%S'),
        'source'      : 'yfinance_daily',
        'error'       : None,
    }

# ── MASTER AGENT: fires sources in parallel, picks best result ─────────────────
def get_live_price(ticker: str) -> dict:
    """
    Agentic price fetcher — runs all relevant sources in parallel threads,
    returns the first successful non-zero result in priority order.

    Priority:
      Indian tickers (.NS/.BO) → NSE_API first, yfinance_intraday second, daily last
      US tickers               → Finnhub first, yfinance_intraday second, daily last

    This is faster than sequential fallback because all sources start at the same time.
    Whichever finishes first with a valid price wins.
    """
    is_indian     = ticker.endswith('.NS') or ticker.endswith('.BO')
    symbol_clean  = ticker.replace('.NS','').replace('.BO','')

    if is_indian:
        sources = [
            ('NSE_API',           lambda: _fetch_nse(symbol_clean)),
            ('yfinance_intraday', lambda: _fetch_yfinance_intraday(ticker)),
            ('yfinance_daily',    lambda: _fetch_yfinance_daily(ticker)),
        ]
        priority = ['NSE_API', 'yfinance_intraday', 'yfinance_daily']
    else:
        sources = [
            ('Finnhub',           lambda: _fetch_finnhub(ticker)),
            ('yfinance_intraday', lambda: _fetch_yfinance_intraday(ticker)),
            ('yfinance_daily',    lambda: _fetch_yfinance_daily(ticker)),
        ]
        priority = ['Finnhub', 'yfinance_intraday', 'yfinance_daily']

    results = {}
    errors  = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        future_map = {executor.submit(fn): name for name, fn in sources}
        try:
            for future in concurrent.futures.as_completed(future_map, timeout=15):
                name = future_map[future]
                try:
                    res = future.result()
                    if res and res.get('price', 0) > 0:
                        results[name] = res
                except Exception as e:
                    errors[name] = str(e)
        except concurrent.futures.TimeoutError:
            pass  # use whatever results arrived before timeout

    # Return best result in priority order
    for source in priority:
        if source in results:
            r = results[source]
            print(f"  📡 {ticker}: source={source} price={r.get('currency','')} {r['price']}")
            return r

    # Everything failed
    err_summary = ' | '.join(f'{k}: {v}' for k, v in errors.items())
    return {
        'ticker'    : ticker,
        'price'     : 0,
        'currency'  : 'INR' if is_indian else 'USD',
        'error'     : f'All sources failed — {err_summary}',
        'timestamp' : datetime.now().strftime('%H:%M:%S'),
    }

# ── format_price_card — unchanged ─────────────────────────────────────────────
def format_price_card(info: dict) -> str:
    """Formats live price dict into a human-readable string for Gradio."""
    if info.get('error'):
        return f"❌ {info['ticker']}: {info['error']}"
    sym  = '₹' if info.get('currency') == 'INR' else '$'
    sign = '+' if info.get('change', 0) >= 0 else ''
    src  = info.get('source', '')
    return (
        f"{info.get('arrow','?')} {info['ticker']}  {sym}{info['price']:,.2f}  "
        f"{sign}{info.get('change',0):.2f} ({sign}{info.get('change_pct',0):.2f}%)  "
        f"| Vol: {info.get('volume',0):,}  "
        f"| {info.get('market_state','?')}  "
        f"| src: {src}  "
        f"| {info.get('timestamp','')}"
    )

# ── Smoke tests ───────────────────────────────────────────────────────────────
print("Running smoke tests (parallel fetch)...\n")
_test_us  = get_live_price('AAPL')
_test_in  = get_live_price('INFY.NS')
_test_in2 = get_live_price('RELIANCE.NS')
print(f"\n{format_price_card(_test_us)}")
print(f"{format_price_card(_test_in)}")
print(f"{format_price_card(_test_in2)}")

Running smoke tests (parallel fetch)...

  📡 AAPL: source=Finnhub price=USD 293.08
  📡 INFY.NS: source=yfinance_intraday price=INR 1057.6
  📡 RELIANCE.NS: source=yfinance_intraday price=INR 1325.9

▼ AAPL  $293.08  -1.22 (-0.41%)  | Vol: 0  | REGULAR  | src: Finnhub  | 04:36:05
▲ INFY.NS  ₹1,057.60  +1.60 (+0.15%)  | Vol: 63,509  | REGULAR (~5m delay)  | src: yfinance_intraday  | 04:36:06
▲ RELIANCE.NS  ₹1,325.90  +12.50 (+0.95%)  | Vol: 11,475  | REGULAR (~5m delay)  | src: yfinance_intraday  | 04:36:07


## 📥 Step 1 — Data Download

In [6]:
def download_stock_data(ticker, start, end):
    print(f'📡 Downloading {ticker}...')
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty: raise ValueError(f'No data for {ticker}')
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.columns = [c.strip() for c in df.columns]
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True); df.dropna(how='all', inplace=True)
    print(f'  ✅ {len(df)} days | Close: {df["Close"].min():.2f} – {df["Close"].max():.2f}')
    return df

def add_market_context(df, ticker, tickers=('SPY','QQQ','XLK','^VIX'), start=None, end=None):  # FIX L2: accept window params
    """
    Adds US market ETFs and VIX as additional features.
    For Indian stocks (ticker ends with .NS or .BO), also adds NIFTY50 (^NSEI) and SENSEX (^BSESN).

    FIX: Indian-ticker detection now checks the actual `ticker` string instead of
    df.columns / df.index.name — neither of those ever contained '.NS'/'.BO', so the
    NIFTY/SENSEX branch could never fire before, despite the docstring claiming it did.
    """
    enriched = df.copy()
    is_indian = ticker.endswith('.NS') or ticker.endswith('.BO')
    extra = ('^NSEI', '^BSESN') if is_indian else ()
    all_tickers = list(tickers) + list(extra)
    for t in all_tickers:
        try:
            col = t.replace('^','')
            mkt = yf.download(t, start=start or CONFIG['start_date'], end=end or CONFIG['end_date'],  # FIX L2: use passed window
                              auto_adjust=True, progress=False)['Close']
            if isinstance(mkt, pd.DataFrame): mkt = mkt.squeeze()
            mkt.index = pd.to_datetime(mkt.index)
            enriched[f'{col}_Close']  = mkt.reindex(enriched.index, method='ffill')
            enriched[f'{col}_Return'] = enriched[f'{col}_Close'].pct_change()
            enriched[f'{col}_Lag1']   = enriched[f'{col}_Return'].shift(1)
            print(f'  ✅ {t}: {enriched[f"{col}_Return"].notna().sum()} rows')
        except Exception as e:
            print(f'  ⚠️ {t}: {e}')
    return enriched

df_raw          = download_stock_data(CONFIG['ticker'], CONFIG['start_date'], CONFIG['end_date'])
df_raw_enriched = add_market_context(df_raw, CONFIG['ticker'])
print(f'Shape: {df_raw.shape} → {df_raw_enriched.shape}')


📡 Downloading AAPL...
  ✅ 2885 days | Close: 20.57 – 315.20
  ✅ SPY: 2884 rows
  ✅ QQQ: 2884 rows
  ✅ XLK: 2884 rows
  ✅ ^VIX: 2884 rows
Shape: (2885, 5) → (2885, 17)


## 🤖 Step 2 — Agent 1: Live Data Scraper

In [7]:
from transformers import pipeline as hf_pipeline

print('🧠 Loading FinBERT...')
try:
    finbert = hf_pipeline('text-classification', model='ProsusAI/finbert',
                           tokenizer='ProsusAI/finbert', device=-1,
                           truncation=True, max_length=512)
    print('✅ FinBERT ready')
except Exception as e:
    finbert = None; print(f'⚠️ FinBERT unavailable: {e}')

def fetch_google_news_rss(ticker, max_articles=25):
    try:
        # Works for Indian stocks too — just pass 'RELIANCE NSE' or 'TCS India'
        url = f'https://news.google.com/rss/search?q={ticker}+stock&hl=en-IN&gl=IN&ceid=IN:en'
        feed = feedparser.parse(url)
        return [{'source':'GoogleNews','text':e.title,'score':1,'pre_scored':False}
                for e in feed.entries[:max_articles] if len(e.title) > 20]
    except Exception as e: print(f'  Google RSS: {e}'); return []

def scrape_yahoo_news(ticker, max_articles=25):
    try:
        url = f'https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US'
        feed = feedparser.parse(url)
        return [{'source':'YahooRSS','text':e.title,'score':1,'pre_scored':False}
                for e in feed.entries[:max_articles] if len(e.get('title','')) > 20]
    except Exception as e: print(f'  Yahoo RSS: {e}'); return []

def fetch_finnhub_news(ticker, max_articles=25):
    if not KEYS.get('finnhub'): return []
    try:
        end_dt   = datetime.today().strftime('%Y-%m-%d')
        start_dt = (datetime.today()-timedelta(days=7)).strftime('%Y-%m-%d')
        r = requests.get(
            f'https://finnhub.io/api/v1/company-news?symbol={ticker}'
            f'&from={start_dt}&to={end_dt}&token={KEYS["finnhub"]}', timeout=10)
        return [{'source':'Finnhub','text':a['headline'],'score':1,'pre_scored':False}
                for a in r.json()[:max_articles] if len(a.get('headline','')) > 20]
    except Exception as e: print(f'  Finnhub: {e}'); return []

def fetch_alphavantage_news(ticker, max_articles=25):
    if not KEYS.get('alphavantage'): return []
    try:
        r = requests.get(
            f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT'
            f'&tickers={ticker}&limit={max_articles}&apikey={KEYS["alphavantage"]}', timeout=15)
        feed = r.json().get('feed', [])
        return [{'source':'AlphaVantage','text':a['title'],
                 'score':float(next((s for s in a.get('ticker_sentiment',[])
                                     if s['ticker']==ticker), {}).get('ticker_sentiment_score',0)),
                 'pre_scored':True}
                for a in feed]
    except Exception as e: print(f'  AlphaVantage: {e}'); return []

def fetch_macro_features(start_date, end_date):
    """
    DFF (Fed Funds) and VIXCLS are published same-day; T10Y2Y (10Y-2Y
    Treasury spread) is also a daily series. CPIAUCSL (CPI) is different:
    the index value is dated to the *reference month*, but the BLS doesn't
    actually release that number until roughly 2-3 weeks into the
    following month. Using CPIAUCSL's own date as if it were known on
    that date is a point-in-time / look-ahead error — a backtest using it
    that way would 'know' a CPI print before it was published. We shift
    CPI by ~10 business days (a conservative approximation of the BLS
    release lag) so the feature only reflects what was actually public
    knowledge on that date. DFF/T10Y2Y/VIXCLS are left unshifted.
    """
    if not KEYS.get('fred'): return pd.DataFrame()
    try:
        from fredapi import Fred
        f = Fred(api_key=KEYS['fred'])
        data = [f.get_series(k, observation_start=start_date, observation_end=end_date).rename(v)
                for k,v in {'DFF':'Fed_Funds_Rate','T10Y2Y':'Yield_Curve',
                             'VIXCLS':'VIX_FRED','CPIAUCSL':'CPI'}.items()]
        macro = pd.concat(data, axis=1)
        macro.index = pd.to_datetime(macro.index)
        macro = macro.resample('B').last().ffill()
        if 'CPI' in macro.columns:
            macro['CPI'] = macro['CPI'].shift(10)   # publication-lag fix — avoids look-ahead bias
        return macro
    except Exception as e: print(f'  FRED: {e}'); return pd.DataFrame()

def scrape_analyst_rating(ticker):
    score_map = {'strong buy':2.0,'buy':1.5,'overweight':1.0,'hold':0.0,
                 'neutral':0.0,'underweight':-1.0,'sell':-1.5,'strong sell':-2.0}
    try:
        r = requests.get(f'https://finviz.com/quote.ashx?t={ticker}',
                          headers={'User-Agent':'Mozilla/5.0'}, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        for td in soup.find_all('td'):
            if 'Recom' in td.get_text():
                v = td.find_next_sibling('td')
                if v: return score_map.get(v.get_text(strip=True).lower(), 0.0)
    except Exception as e: print(f'  Finviz: {e}')
    return 0.0

def score_sentiment(texts):
    if not texts: return 0.0
    pre = [t for t in texts if t.get('pre_scored')]
    if pre: return float(np.mean([t['score'] for t in pre]))
    if finbert is None: return 0.0
    label_map = {'positive':1.0,'neutral':0.0,'negative':-1.0}
    scores = []
    for item in texts:
        try:
            res = finbert(item['text'][:512])[0]
            scores.append(label_map.get(res['label'],0.0)*res['score']*(1+np.log1p(item.get('score',1))))
        except: pass
    return float(np.mean(scores)) if scores else 0.0

def run_data_agent(df, ticker):
    print(f'\n🤖 AGENT 1: {ticker}\n' + '='*55)
    enriched = df.copy()
    yahoo  = scrape_yahoo_news(ticker)
    alt    = (fetch_alphavantage_news(ticker) or fetch_finnhub_news(ticker)  # FIX M2: AV first (pre-scored avoids FinBERT), then Finnhub, Google last
               or fetch_google_news_rss(ticker))
    ns = score_sentiment(yahoo); als = score_sentiment(alt)
    non_zero = [s for s in [ns, als] if s != 0.0]
    combined = float(np.mean(non_zero)) if non_zero else 0.0
    analyst  = scrape_analyst_rating(ticker)

    # FIX (was look-ahead leakage): today's live sentiment was previously broadcast
    # backward across the last 30 HISTORICAL rows with exponential decay. That plants
    # information from today into rows representing past trading days that didn't have
    # it yet. Sentiment reflects right now, so it can only validly apply to the most
    # recent row. (News_Sentiment is also excluded from feature_cols below, so this
    # never reached the trained models — but the historical dataframe itself was wrong,
    # which is a trap for any future code that reuses df_features.)
    enriched['News_Sentiment'] = 0.0
    enriched['Analyst_Score']  = 0.0
    enriched.iloc[-1, enriched.columns.get_loc('News_Sentiment')] = combined
    enriched.iloc[-1, enriched.columns.get_loc('Analyst_Score')]  = analyst

    macro = fetch_macro_features(CONFIG['start_date'], CONFIG['end_date'])
    if not macro.empty:
        enriched = enriched.join(macro, how='left')
        # FIX (look-ahead leak): .bfill() here used to fill EARLY rows (before a
        # series' first release / before the CPI lag-shift kicks in) with a LATER,
        # not-yet-known value — textbook look-ahead bias. ffill() only carries
        # KNOWN values forward in time, which is the only causally valid direction.
        # Rows still NaN after ffill() (the very start of history, before any macro
        # print existed) are removed by engineer_features()'s dropna() a few steps
        # downstream, same as any other warm-up-period row.
        for col in macro.columns: enriched[col] = enriched[col].ffill()

    report = {
        'ticker': ticker, 'run_timestamp': datetime.now().isoformat(),
        'news_sentiment': round(ns,4), 'alt_news_sentiment': round(als,4),
        'combined_sentiment': round(combined,4), 'analyst_score': round(analyst,4),
        'yahoo_count': len(yahoo), 'alt_news_count': len(alt),
    }
    print(f'✅ Agent 1 done. New cols: {[c for c in enriched.columns if c not in df.columns]}')
    return enriched, report

df_raw_enriched, sentiment_report = run_data_agent(df_raw_enriched, CONFIG['ticker'])
print(f'\nSentiment:\n{json.dumps(sentiment_report, indent=2)}')


🧠 Loading FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ FinBERT ready

🤖 AGENT 1: AAPL
✅ Agent 1 done. New cols: ['News_Sentiment', 'Analyst_Score', 'Fed_Funds_Rate', 'Yield_Curve', 'VIX_FRED', 'CPI']

Sentiment:
{
  "ticker": "AAPL",
  "run_timestamp": "2026-06-25T04:36:43.791099",
  "news_sentiment": 0.1221,
  "alt_news_sentiment": 0.0724,
  "combined_sentiment": 0.0973,
  "analyst_score": 0.0,
  "yahoo_count": 20,
  "alt_news_count": 50
}


In [8]:
print("Rows downloaded:", len(df_raw_enriched))
print("Start Date setting:", CONFIG['start_date'])

Rows downloaded: 2885
Start Date setting: 2015-01-01


## ⚙️ Step 3 — Feature Engineering

In [9]:
def engineer_features(df):
    data = df.copy()

    # Check if there is enough data for feature engineering
    # The largest window is 200 for SMA_200.
    min_rows_needed = 200
    if len(data) < min_rows_needed:
        raise ValueError(
            f"Insufficient data for feature engineering. Expected at least "
            f"{min_rows_needed} rows, but got {len(data)}. "
            f"Please ensure 'download_stock_data' fetches enough historical data."
        )

    close = data['Close'].squeeze(); high=data['High'].squeeze()
    low=data['Low'].squeeze();       vol=data['Volume'].squeeze()
    data['SMA_20']  = SMAIndicator(close,20).sma_indicator()
    data['SMA_50']  = SMAIndicator(close,50).sma_indicator()
    data['SMA_200'] = SMAIndicator(close,200).sma_indicator()
    data['EMA_12']  = EMAIndicator(close,12).ema_indicator()
    data['EMA_26']  = EMAIndicator(close,26).ema_indicator()
    macd=MACD(close); data['MACD']=macd.macd(); data['MACD_Signal']=macd.macd_signal(); data['MACD_Hist']=macd.macd_diff()
    data['RSI_14'] = RSIIndicator(close,14).rsi()
    stoch=StochasticOscillator(high,low,close); data['Stoch_K']=stoch.stoch(); data['Stoch_D']=stoch.stoch_signal()
    bb=BollingerBands(close,20,2)
    data['BB_Upper']=bb.bollinger_hband(); data['BB_Middle']=bb.bollinger_mavg()
    data['BB_Lower']=bb.bollinger_lband(); data['BB_Width']=bb.bollinger_wband(); data['BB_Pct']=bb.bollinger_pband()
    data['ATR_14']         = AverageTrueRange(high,low,close,14).average_true_range()
    data['Daily_Return']   = close.pct_change()
    data['Log_Return']     = np.log(close/close.shift(1))
    data['Rolling_Std_20'] = close.rolling(20).std()
    data['OBV']            = OnBalanceVolumeIndicator(close,vol).on_balance_volume()
    data['Volume_SMA_20']  = vol.rolling(20).mean()
    data['Volume_Ratio']   = vol/data['Volume_SMA_20']
    for lag in [1,2,3,5,10]: data[f'Return_Lag_{lag}']=np.log(close/close.shift(lag))
    data['Rolling_Return_5']  = data['Log_Return'].rolling(5).mean()
    data['Rolling_Return_10'] = data['Log_Return'].rolling(10).mean()
    data['Rolling_Return_20'] = data['Log_Return'].rolling(20).mean()  # FIX: was rolling(10), an exact duplicate of Rolling_Return_10
    data['Price_vs_SMA20'] = (close-data['SMA_20'])/data['SMA_20']
    data['Price_vs_SMA50'] = (close-data['SMA_50'])/data['SMA_50']
    data['Day_of_Week']=data.index.dayofweek; data['Month']=data.index.month; data['Quarter']=data.index.quarter
    # Target = log-return N days ahead (avoids price scale issues)
    data['Target'] = np.log(close.shift(-CONFIG['forecast_days'])/close)
    data['Intraday_Return'] = (data['Close'] - data['Open']) / data['Open']

    # ── QUANT RESEARCH FEATURES ──────────────────────────────────────────
    # Jegadeesh & Titman (1993) 'Returns to Buying Winners and Selling
    # Losers', Journal of Finance 48(1): past-return momentum predicts
    # future returns over 3-12 month horizons. We use shorter (1W-3M)
    # windows appropriate for a daily-bar model.
    for win in [5, 10, 21, 63]:
        data[f'Mom_{win}d'] = close.pct_change(win)

    # Garman & Klass (1980) 'On the Estimation of Security Price
    # Volatilities from Historical Data', Journal of Business 53(1):
    # an OHLC-based volatility estimator that is ~7-8x more efficient
    # than close-to-close std for the same sample size.
    # sigma^2_GK = 0.5*ln(H/L)^2 - (2ln2-1)*ln(C/O)^2
    # NOTE: this expression can occasionally go slightly negative on noisy
    # bars (it is a difference of two non-negative terms, not a sum) —
    # clip at 0 before sqrt() or it silently produces NaNs that cascade
    # into every downstream row via rolling/lag features.
    ln_hl = np.log(high / low)
    ln_co = np.log(close / data['Open'])
    gk_var = 0.5 * ln_hl**2 - (2*np.log(2) - 1) * ln_co**2
    data['GK_Vol'] = np.sqrt(gk_var.clip(lower=0))

    # Parkinson (1980) 'The Extreme Value Method for Estimating the
    # Variance of the Rate of Return', Journal of Business 53(1):
    # range-based estimator using only the high-low range.
    data['Parkinson_Vol'] = np.sqrt(ln_hl**2 / (4 * np.log(2)))

    # Amihud (2002) 'Illiquidity and Stock Returns: Cross-Section and
    # Time-Series Effects', Journal of Financial Markets 5(1): price
    # impact per unit of dollar volume — a standard illiquidity proxy.
    data['Amihud_Illiq'] = (
        np.abs(data['Log_Return']) / (close * vol + 1e-8)
    ).rolling(21).mean()

    # Realized variance (Andersen & Bollerslev 1998 framework): sum of
    # squared daily log-returns over a rolling window, a model-free
    # estimate of integrated variance.
    data['RealizedVar_21d'] = (data['Log_Return']**2).rolling(21).sum()

    # Ulcer Index (Martin & McCann 1987, 'The Investor's Guide to
    # Fidelity Funds'): downside-only drawdown-depth risk measure, used
    # as the denominator of the Ulcer Performance Index.
    roll_max = close.rolling(14).max()
    drawdown_pct = 100 * (close - roll_max) / roll_max
    data['Ulcer_14d'] = np.sqrt((drawdown_pct**2).rolling(14).mean())

    # Standardised volume shock (z-score vs its own rolling distribution).
    data['Volume_ZScore'] = (
        (vol - vol.rolling(21).mean()) / (vol.rolling(21).std() + 1e-8)
    )

    # Lag-1 return autocorrelation (Lo & MacKinlay 1988, 'Stock Market
    # Prices Do Not Follow Random Walks', Review of Financial Studies
    # 1(1)): positive -> trending regime, negative -> mean-reverting.
    data['AutoCorr_5d'] = (
        data['Log_Return'].rolling(20).apply(
            lambda x: x.autocorr(lag=1) if x.notna().sum() > 5 else 0.0, raw=False
        )
    )

    # Kaufman (1995) Efficiency Ratio, 'Smarter Trading': trend strength
    # on a 0->1 scale = |net price change| / sum(|bar-to-bar changes|).
    def _efficiency_ratio(s, n=10):
        net   = (s - s.shift(n)).abs()
        total = s.diff().abs().rolling(n).sum()
        return net / (total + 1e-8)
    data['Efficiency_Ratio'] = _efficiency_ratio(close, 10)

    data.dropna(inplace=True)
    print(f'✅ Features: {data.shape[1]} cols, {len(data)} rows')
    return data

df_features = engineer_features(df_raw_enriched)

# FIX (quant+): Fed_Funds_Rate / Yield_Curve / VIX_FRED / CPI used to be
# excluded alongside sentiment, but for a different (and weaker) reason —
# they were called "forward-filled constants" that "confuse tree models".
# That reasoning doesn't hold once the look-ahead leak is fixed (see Step 2):
# unlike News_Sentiment (which is a fabricated 0.0 for ~every historical row
# except the very last one), these macro series have a real, causally-valid
# daily/monthly value for the entire training history. Excluding genuine
# macro signal is needless — Chen, Roll & Ross (1986) is the standard
# reference for using rates/yield-curve/inflation as equity-return factors.
# News_Sentiment and Analyst_Score remain excluded: historically they ARE
# fabricated zeros (only the most recent row has a real score), so including
# them would teach the trees a spurious "0.0 sentiment" pattern that has
# nothing to do with the stock's actual historical news flow.
EXCLUDE = [
    'Open','High','Low','Close','Volume','Target',
    'SPY_Close','QQQ_Close','XLK_Close','VIX_Close',
    'NSEI_Close','BSESN_Close',
    'News_Sentiment','Analyst_Score',
]
feature_cols = [c for c in df_features.columns if c not in EXCLUDE]
print(f'Features ({len(feature_cols)}): {feature_cols}')

✅ Features: 73 cols, 2685 rows
Features (61): ['SPY_Return', 'SPY_Lag1', 'QQQ_Return', 'QQQ_Lag1', 'XLK_Return', 'XLK_Lag1', 'VIX_Return', 'VIX_Lag1', 'Fed_Funds_Rate', 'Yield_Curve', 'VIX_FRED', 'CPI', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI_14', 'Stoch_K', 'Stoch_D', 'BB_Upper', 'BB_Middle', 'BB_Lower', 'BB_Width', 'BB_Pct', 'ATR_14', 'Daily_Return', 'Log_Return', 'Rolling_Std_20', 'OBV', 'Volume_SMA_20', 'Volume_Ratio', 'Return_Lag_1', 'Return_Lag_2', 'Return_Lag_3', 'Return_Lag_5', 'Return_Lag_10', 'Rolling_Return_5', 'Rolling_Return_10', 'Rolling_Return_20', 'Price_vs_SMA20', 'Price_vs_SMA50', 'Day_of_Week', 'Month', 'Quarter', 'Intraday_Return', 'Mom_5d', 'Mom_10d', 'Mom_21d', 'Mom_63d', 'GK_Vol', 'Parkinson_Vol', 'Amihud_Illiq', 'RealizedVar_21d', 'Ulcer_14d', 'Volume_ZScore', 'AutoCorr_5d', 'Efficiency_Ratio']


## ✂️ Step 4 — Split & Evaluation Framework

In [10]:
def split_data(df, feature_cols, test_size):
    X=df[feature_cols].values; y=df['Target'].values
    idx=int(len(X)*(1-test_size))
    X_train,X_test=X[:idx],X[idx:]
    y_train,y_test=y[:idx],y[idx:]
    train_dates=df.index[:idx]; test_dates=df.index[idx:]
    scaler=StandardScaler()
    X_train_sc=scaler.fit_transform(X_train)
    X_test_sc=scaler.transform(X_test)
    print(f'Train: {len(X_train):,} | {train_dates[0].date()} → {train_dates[-1].date()}')
    print(f'Test : {len(X_test):,}  | {test_dates[0].date()} → {test_dates[-1].date()}')
    return X_train_sc, X_test_sc, y_train, y_test, scaler, test_dates

X_train,X_test,y_train,y_test,scaler,test_dates = split_data(df_features,feature_cols,CONFIG['test_size'])

# ── Fixed evaluate_model — replace wherever it's defined in your notebook ──
def evaluate_model(y_true, y_pred, name='Model'):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    # Remove NaN pairs (last row of each WF window has NaN Target)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]

    if len(y_true) == 0:
        return {'Model': name, 'RMSE': np.nan, 'MAE': np.nan,
                'MAPE (%)': np.nan, 'R2': np.nan,
                'Directional Acc %': np.nan, 'P-Value': np.nan,
                'Sig (p<0.05)': '❌', 'RMSE_CI_95': '[nan,nan]'}

    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    # MAPE — guard against near-zero true values
    nonzero = np.abs(y_true) > 1e-8
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero])
                           / y_true[nonzero])) * 100 if nonzero.any() else np.nan

    # DirAcc — use boolean > 0, NOT np.sign()
    # np.sign(0.0) = 0.0, which equals neither +1 nor -1
    # causing 0% DirAcc whenever XGBoost predicts near-zero in calm markets
    dir_acc = np.mean((y_pred > 0) == (y_true > 0)) * 100

    # Bootstrap RMSE CI
    rng = np.random.default_rng(42)
    boot = [np.sqrt(mean_squared_error(
                y_true[idx := rng.integers(0, len(y_true), len(y_true))],
                y_pred[idx])) for _ in range(1000)]
    ci = f'[{np.percentile(boot,2.5):.6f},{np.percentile(boot,97.5):.6f}]'

    # Wilcoxon test vs naive baseline (predict zero return)
    naive  = np.zeros_like(y_true)
    errors_model = np.abs(y_true - y_pred)
    errors_naive = np.abs(y_true - naive)
    try:
        from scipy.stats import wilcoxon
        _, pval = wilcoxon(errors_model, errors_naive)
    except Exception:
        pval = 1.0

    # ── QUANT+ additions ────────────────────────────────────────────────
    # Diebold & Mariano (1995) 'Comparing Predictive Accuracy', Journal of
    # Business & Economic Statistics 13(3): tests whether the difference
    # in forecast-loss between two models is statistically significant,
    # rather than just comparing point RMSE. One-sided: H1 = model beats
    # naive. Wilcoxon (above) tests the same idea non-parametrically on
    # absolute errors; DM is the standard parametric companion quoted
    # alongside it in forecast-evaluation papers.
    from scipy.stats import norm as _norm
    loss_diff = errors_naive - errors_model          # positive = model better than naive
    dm_mean   = loss_diff.mean()
    dm_se     = loss_diff.std(ddof=1) / np.sqrt(max(len(loss_diff), 1)) + 1e-12
    dm_stat   = dm_mean / dm_se
    dm_pval   = float(1 - _norm.cdf(dm_stat))         # one-sided

    # Wilson score interval (Wilson 1927; see Brown, Cai & DasGupta 2001,
    # 'Interval Estimation for a Binomial Proportion', Statistical Science
    # 16(2), for why this beats the naive normal-approximation CI for a
    # proportion — directional accuracy IS a binomial proportion). Tighter
    # and better-calibrated than +-1.96*sqrt(p(1-p)/n) for n in the
    # hundreds, which is the regime a single test split usually falls in.
    n_dir  = len(y_true)
    p_hat  = dir_acc / 100
    z95    = 1.96
    denom  = 1 + z95**2 / n_dir
    centre = (p_hat + z95**2 / (2*n_dir)) / denom
    margin = z95 * np.sqrt(p_hat*(1-p_hat)/n_dir + z95**2/(4*n_dir**2)) / denom
    da_ci  = f'[{round(max(0,centre-margin)*100,1)},{round(min(1,centre+margin)*100,1)}]'

    # Information Coefficient: Spearman rank correlation of predicted vs
    # realised returns. Standard quant-PM diagnostic (Grinold & Kahn,
    # 'Active Portfolio Management', 2nd ed., 1999) — IC > 0.05 is
    # considered a meaningful signal at the individual-forecast level;
    # > 0.10 is strong for daily equity forecasts.
    from scipy.stats import spearmanr as _spearmanr
    _ic, _ = _spearmanr(y_pred, y_true)
    ic_val = round(float(_ic), 4) if not np.isnan(_ic) else np.nan

    return {
        'Model'             : name,
        'RMSE'              : round(rmse,  6),
        'RMSE_CI_95'        : ci,
        'MAE'               : round(mae,   6),
        'MAPE (%)'          : round(mape,  6) if not np.isnan(mape) else np.nan,
        'R2'                : round(r2,    4),
        'Directional Acc %' : round(dir_acc, 2),
        'DirAcc CI 95%'     : da_ci,                         # Wilson score interval
        'IC (Spearman)'     : ic_val,                        # Information coefficient
        'P-Value'           : round(pval,  4),               # Wilcoxon vs naive
        'DM-stat'           : round(float(dm_stat), 3),      # Diebold-Mariano
        'DM p-val'          : round(dm_pval, 4),              # one-sided, model<naive
        'Sig (p<0.05)'      : '✅ Yes' if pval < 0.05 else '❌ No',
    }

def plot_confusion_matrix(y_true, y_pred, model_name):
    # FIX: direction is now defined the SAME way as evaluate_model()'s Directional Acc %
    # (sign of the value itself), not np.diff() of the series. Those are different
    # questions — the old version made this table disagree with the Step 12 summary.
    true_dir=(np.asarray(y_true) > 0).astype(int)
    pred_dir=(np.asarray(y_pred) > 0).astype(int)
    cm=confusion_matrix(true_dir,pred_dir)
    fig,ax=plt.subplots(figsize=(5,4))
    # FIX: cmap (not colormap) is the correct kwarg for ConfusionMatrixDisplay.plot
    ConfusionMatrixDisplay(cm,display_labels=['DOWN','UP']).plot(ax=ax,cmap='Blues',values_format='d')
    ax.set_title(f'{model_name} — Direction Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f"confusion_{model_name.replace(' ','_')}.png",dpi=120,bbox_inches='tight')
    plt.show()
    tn,fp,fn,tp=cm.ravel()
    print(f'  Precision: {tp/(tp+fp+1e-8):.2%} | Recall: {tp/(tp+fn+1e-8):.2%}')

print('✅ Eval functions ready')


Train: 2,148 | 2015-10-16 → 2024-04-30
Test : 537  | 2024-05-01 → 2026-06-23
✅ Eval functions ready


## 🎯 Step 5 — Optuna Hyperparameter Tuning

In [11]:
# ── QUANT+ FIX: purged & embargoed CV for hyperparameter search ───────────
# Plain sklearn TimeSeriesSplit puts the validation fold immediately after
# the train fold. Several engineered features are rolling windows up to
# 21-63 days wide (momentum, Amihud illiquidity, volume z-score, realized
# variance, etc.) plus a 200-day SMA. That means the LAST rows of a train
# fold and the FIRST rows of the very next validation fold are computed
# from overlapping raw price/volume history -> the validation score is
# optimistically biased, and Optuna ends up tuning hyperparameters against
# a metric that's leaking information across the fold boundary. This is
# exactly the failure mode described in Lopez de Prado (2018), "Advances
# in Financial Machine Learning", Wiley, Ch. 7 ("Cross-Validation in
# Finance"). The fix: PURGE the last `cv_purge_days` rows of every train
# fold (drop overlap with the next validation window) and EMBARGO
# `cv_embargo_days` rows after each validation fold from ever appearing as
# training data later. purge=63 matches our longest actively-used rolling
# window (63d momentum); SMA_200 is a known, documented exception this
# doesn't fully neutralise (see CONFIG comment) — purging 200 days would
# leave too little data per fold to tune anything meaningfully, so this is
# a deliberate, disclosed trade-off rather than a claim of zero leakage.
class PurgedTimeSeriesSplit:
    def __init__(self, n_splits=5, purge=0, embargo=0):
        self.n_splits, self.purge, self.embargo = n_splits, purge, embargo
    def split(self, X):
        n = len(X)
        fold_sizes = np.full(self.n_splits + 1, n // (self.n_splits + 1))
        fold_sizes[: n % (self.n_splits + 1)] += 1
        boundaries = np.cumsum(fold_sizes)
        for i in range(self.n_splits):
            val_start, val_end = boundaries[i], boundaries[i + 1]
            train_end = max(0, val_start - self.purge)
            train_idx = np.arange(0, train_end)
            val_idx   = np.arange(val_start, val_end)
            if len(train_idx) == 0 or len(val_idx) == 0:
                continue
            yield train_idx, val_idx

tscv = PurgedTimeSeriesSplit(n_splits=CONFIG['n_folds'],
                              purge=CONFIG['cv_purge_days'],
                              embargo=CONFIG['cv_embargo_days'])
print(f"✅ Purged CV: {CONFIG['n_folds']} folds | purge={CONFIG['cv_purge_days']}d | embargo={CONFIG['cv_embargo_days']}d")

def xgb_objective(trial):
    p={'n_estimators':trial.suggest_int('n_estimators',200,800),
       'max_depth':trial.suggest_int('max_depth',3,9),
       'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
       'subsample':trial.suggest_float('subsample',0.6,1.0),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
       'reg_alpha':trial.suggest_float('reg_alpha',1e-4,5.0,log=True),
       'reg_lambda':trial.suggest_float('reg_lambda',1e-4,5.0,log=True),
       'random_state':42,'n_jobs':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        xgb.XGBRegressor(**p,verbosity=0).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('🔍 Optuna: XGBoost (40 trials)...')
xs=optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=CONFIG['random_state']))  # FIX: was unseeded -> different result every run
xs.optimize(xgb_objective,n_trials=40,show_progress_bar=True)
best_xgb_params={**xs.best_params,'random_state':42,'n_jobs':-1}
print(f'✅ Best XGBoost RMSE: {xs.best_value:.6f}')
print(f'   Params: {best_xgb_params}')

def lgb_objective(trial):
    p={'n_estimators':trial.suggest_int('n_estimators',200,800),
       'max_depth':trial.suggest_int('max_depth',3,9),
       'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,log=True),
       'num_leaves':trial.suggest_int('num_leaves',20,100),
       'subsample':trial.suggest_float('subsample',0.6,1.0),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
       'random_state':42,'n_jobs':-1,'verbose':-1}
    return np.mean([np.sqrt(mean_squared_error(y_train[te],
        lgb.LGBMRegressor(**p).fit(X_train[tr],y_train[tr]).predict(X_train[te])))
        for tr,te in tscv.split(X_train)])

print('\n🔍 Optuna: LightGBM (40 trials)...')
ls=optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=CONFIG['random_state']))  # FIX: was unseeded -> different result every run
ls.optimize(lgb_objective,n_trials=40,show_progress_bar=True)
best_lgb_params={**ls.best_params,'random_state':42,'n_jobs':-1,'verbose':-1}
print(f'✅ Best LightGBM RMSE: {ls.best_value:.6f}')


✅ Purged CV: 5 folds | purge=63d | embargo=21d
🔍 Optuna: XGBoost (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Best XGBoost RMSE: 0.018289
   Params: {'n_estimators': 368, 'max_depth': 6, 'learning_rate': 0.02748227652087293, 'subsample': 0.6961347550113168, 'colsample_bytree': 0.6363263256480385, 'reg_alpha': 0.8671498103269967, 'reg_lambda': 0.010750981876739453, 'random_state': 42, 'n_jobs': -1}

🔍 Optuna: LightGBM (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

✅ Best LightGBM RMSE: 0.019126


## ⚖️ Step 5b — Data-Driven Ensemble Weights

The 0.4 / 0.4 / 0.2 split (XGBoost / LightGBM / Random Forest) was previously a hardcoded guess. Here we replace it with **inverse out-of-fold-RMSE convex weights** — Bates & Granger (1969), *"The Combination of Forecasts"*, Operational Research Quarterly 20(4): the variance-minimising linear combination of several unbiased forecasts weights each one by the inverse of its error variance. We use inverse OOF-RMSE as a simple, well-behaved proxy (Stock & Watson (2004), *"Combination Forecasts of Output Growth in a Seven-Country Data Set"*, Journal of Forecasting 23(6), show this kind of simple inverse-error weighting is more robust in practice than estimating a fully optimal covariance-weighted combination on a finite sample).

**Important — this is fit only on the training set's purged CV folds, never on the test set.** Computing weights from test-set performance and then evaluating the weighted ensemble on that same test set would be a textbook case of fitting to the evaluation data.

In [12]:
# ── Step 5b: Data-Driven Ensemble Weights ───────────────────────────────────
print("\n⚖️  STEP 5b: Data-Driven Ensemble Weights (OOF inverse-RMSE)")
print("=" * 55)

_rf_fixed_params = dict(n_estimators=300, max_depth=15, min_samples_split=5,
    min_samples_leaf=2, max_features='sqrt', random_state=CONFIG['random_state'], n_jobs=-1)

def _oof_rmse(model_fn, X, y, splitter):
    """Out-of-fold RMSE via the SAME purged/embargoed CV used for HPO above —
    never touches X_test/y_test."""
    oof_pred = np.full(len(y), np.nan)
    n_folds_used = 0
    for tr, va in splitter.split(X):
        m = model_fn()
        m.fit(X[tr], y[tr])
        oof_pred[va] = m.predict(X[va])
        n_folds_used += 1
    mask = ~np.isnan(oof_pred)
    if n_folds_used == 0 or mask.sum() < 10:
        return None
    return float(np.sqrt(mean_squared_error(y[mask], oof_pred[mask])))

_weight_cv = PurgedTimeSeriesSplit(n_splits=CONFIG['n_folds'],
                                    purge=CONFIG['cv_purge_days'],
                                    embargo=CONFIG['cv_embargo_days'])

try:
    _oof_rmse_xgb = _oof_rmse(lambda: xgb.XGBRegressor(**best_xgb_params, verbosity=0),
                               X_train, y_train, _weight_cv)
    _oof_rmse_lgb = _oof_rmse(lambda: lgb.LGBMRegressor(**best_lgb_params),
                               X_train, y_train, _weight_cv)
    _oof_rmse_rf  = _oof_rmse(lambda: RandomForestRegressor(**_rf_fixed_params),
                               X_train, y_train, _weight_cv)
    _oof = {'xgb': _oof_rmse_xgb, 'lgb': _oof_rmse_lgb, 'rf': _oof_rmse_rf}

    if all(v is not None and v > 0 for v in _oof.values()):
        _inv = {k: 1.0 / v for k, v in _oof.items()}
        _total = sum(_inv.values())
        data_driven_weights = {k: round(v / _total, 4) for k, v in _inv.items()}
        CONFIG['ensemble_weights'] = data_driven_weights
        print(f"  OOF RMSE   : xgb={_oof['xgb']:.6f}  lgb={_oof['lgb']:.6f}  rf={_oof['rf']:.6f}")
        print(f"  ✅ Data-driven weights : {CONFIG['ensemble_weights']}  (lower OOF error -> higher weight)")
    else:
        print("  ⚠️  OOF computation incomplete for one or more models — keeping CONFIG default weights "
              f"{CONFIG['ensemble_weights']}")
except Exception as e:
    print(f"  ⚠️  OOF weight fitting failed ({e}) — keeping CONFIG default weights {CONFIG['ensemble_weights']}")

print(f"\n  Weights used downstream (Step 6 ensemble, Step 13 predict_tomorrow): {CONFIG['ensemble_weights']}")


⚖️  STEP 5b: Data-Driven Ensemble Weights (OOF inverse-RMSE)
  OOF RMSE   : xgb=0.018928  lgb=0.019799  rf=0.020256
  ✅ Data-driven weights : {'xgb': 0.346, 'lgb': 0.3307, 'rf': 0.3233}  (lower OOF error -> higher weight)

  Weights used downstream (Step 6 ensemble, Step 13 predict_tomorrow): {'xgb': 0.346, 'lgb': 0.3307, 'rf': 0.3233}


## 🏋️ Step 6 — Train All Models

In [13]:
predictions, metrics_list = {}, []

def train_eval(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr); pred=model.predict(Xte)
    predictions[name]=pred
    m=evaluate_model(yte,pred,name); metrics_list.append(m)
    print(f"  {name}: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% p={m['P-Value']} {m['Sig (p<0.05)']}")
    return pred

print('🔵 Linear Regression...'); y_pred_lr    = train_eval('Linear Regression',LinearRegression(),X_train,X_test,y_train,y_test)
print('🔵 Ridge Regression...');  y_pred_ridge = train_eval('Ridge Regression',Ridge(alpha=1.0),X_train,X_test,y_train,y_test)

print('🟠 XGBoost...')
xgb_model=xgb.XGBRegressor(**best_xgb_params,verbosity=0)
xgb_model.fit(X_train,y_train,eval_set=[(X_test,y_test)],verbose=False)
y_pred_xgb=xgb_model.predict(X_test)
predictions['XGBoost']=y_pred_xgb
m=evaluate_model(y_test,y_pred_xgb,'XGBoost'); metrics_list.append(m)
print(f"  XGBoost: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
xgb_model.save_model('xgboost_stock_model.json')

print('🟡 LightGBM...')
lgb_model=lgb.LGBMRegressor(**best_lgb_params)
lgb_model.fit(X_train,y_train)
y_pred_lgb=lgb_model.predict(X_test)
predictions['LightGBM']=y_pred_lgb
m=evaluate_model(y_test,y_pred_lgb,'LightGBM'); metrics_list.append(m)
print(f"  LightGBM: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
joblib.dump(lgb_model,'lightgbm_stock_model.pkl')

print('🟢 Random Forest...')
rf_model=RandomForestRegressor(n_estimators=300,max_depth=15,min_samples_split=5,
    min_samples_leaf=2,max_features='sqrt',random_state=CONFIG['random_state'],n_jobs=-1)
y_pred_rf=train_eval('Random Forest',rf_model,X_train,X_test,y_train,y_test)
joblib.dump(rf_model,'random_forest_stock_model.pkl')

# Weighted ensemble
_ew = CONFIG['ensemble_weights']
y_pred_ensemble=y_pred_xgb*_ew['xgb']+y_pred_lgb*_ew['lgb']+y_pred_rf*_ew['rf']  # FIX M4: from CONFIG
predictions['Ensemble']=y_pred_ensemble
m=evaluate_model(y_test,y_pred_ensemble,'Ensemble'); metrics_list.append(m)
print(f"  Ensemble: RMSE={m['RMSE']} DirAcc={m['Directional Acc %']}% {m['Sig (p<0.05)']}")
print('✅ All tree models trained.')


🔵 Linear Regression...
  Linear Regression: RMSE=0.018354 DirAcc=50.84% p=0.0 ✅ Yes
🔵 Ridge Regression...
  Ridge Regression: RMSE=0.018333 DirAcc=51.02% p=0.0 ✅ Yes
🟠 XGBoost...
  XGBoost: RMSE=0.017625 DirAcc=46.37% ✅ Yes
🟡 LightGBM...
  LightGBM: RMSE=0.017704 DirAcc=49.53% ✅ Yes
🟢 Random Forest...
  Random Forest: RMSE=0.018669 DirAcc=45.62% p=0.0 ✅ Yes
  Ensemble: RMSE=0.017795 DirAcc=46.55% ✅ Yes
✅ All tree models trained.


## 📐 Step 6b — Regime Detection (HMM) & Conditional Volatility (GARCH)

Hamilton (1989), *"A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle"*, Econometrica 57(2): a 2-state Hidden Markov Model fit on log-returns separates bull/bear regimes without needing labelled data.

Bollerslev (1986), *"Generalized Autoregressive Conditional Heteroskedasticity"*, Journal of Econometrics 31(3): GARCH(1,1) captures the volatility clustering visible in virtually every equity return series — calm periods follow calm periods, turbulent periods follow turbulent periods — which a flat rolling-std feature does not.

In [14]:
# ── Step 6b — HMM Regime Detection + GARCH(1,1) Volatility ─────────────────
print("\n🔵 STEP 6b: HMM Regime Detection + GARCH(1,1)")
print("=" * 55)

# ── HMM: 2-state regime classifier, fit on TRAIN portion only ──────────────
try:
    from hmmlearn.hmm import GaussianHMM

    _hmm_split = int(len(df_features) * (1 - CONFIG['test_size']))
    train_returns = df_features['Log_Return'].values[:_hmm_split]

    hmm_model = GaussianHMM(n_components=2, covariance_type='full',
                             n_iter=200, random_state=CONFIG['random_state'])
    hmm_model.fit(train_returns.reshape(-1, 1))

    state_means = hmm_model.means_.flatten()
    bull_state  = int(np.argmax(state_means))   # higher mean return = bull
    bear_state  = 1 - bull_state

    # Decode regimes on the FULL series. This is NOT a leakage concern in the
    # way feature-engineering leakage is: the HMM's parameters (means,
    # covariances, transition matrix) are estimated from train data only;
    # we are simply applying that already-fitted model forward to label
    # test-period rows, exactly like applying a fitted scaler.transform()
    # to test data — the parameters never saw test-period returns.
    all_returns  = df_features['Log_Return'].values
    hmm_regimes  = hmm_model.predict(all_returns.reshape(-1, 1))
    bull_prob    = hmm_model.predict_proba(all_returns.reshape(-1, 1))[:, bull_state]
    df_features['HMM_Regime']    = hmm_regimes
    df_features['HMM_Bull_Prob'] = bull_prob

    test_regimes = hmm_regimes[_hmm_split:]
    bull_frac    = (test_regimes == bull_state).mean()
    print(f"  State means      : bull={state_means[bull_state]:+.5f}  bear={state_means[bear_state]:+.5f}")
    print(f"  Test-set regimes : {bull_frac:.1%} bull  |  {1-bull_frac:.1%} bear")

    # Regime-conditional directional accuracy of the ensemble (diagnostic only)
    _bull_mask = (test_regimes == bull_state)
    _ens_dir   = (y_pred_ensemble > 0) == (y_test > 0)
    if _bull_mask.sum() > 0:
        print(f"  Ensemble DirAcc in BULL regime : {_ens_dir[_bull_mask].mean()*100:.1f}%  (n={_bull_mask.sum()})")
    if (~_bull_mask).sum() > 0:
        print(f"  Ensemble DirAcc in BEAR regime : {_ens_dir[~_bull_mask].mean()*100:.1f}%  (n={(~_bull_mask).sum()})")

    HMM_AVAILABLE = True
    print("  ✅ HMM fitted — regime labels stored in df_features['HMM_Regime' / 'HMM_Bull_Prob']")
except ImportError:
    print("  ⚠️  hmmlearn not installed — run: !pip install hmmlearn -q")
    HMM_AVAILABLE = False
    df_features['HMM_Regime'], df_features['HMM_Bull_Prob'] = 0, 0.5
except Exception as e:
    print(f"  ⚠️  HMM failed: {e}")
    HMM_AVAILABLE = False
    df_features['HMM_Regime'], df_features['HMM_Bull_Prob'] = 0, 0.5

print()

# ── GARCH(1,1): conditional volatility, fit on TRAIN portion only ──────────
try:
    from arch import arch_model

    ret_pct    = df_features['Log_Return'].values * 100   # arch expects %-scale returns
    _gsplit    = int(len(ret_pct) * (1 - CONFIG['test_size']))

    # FIX (mean-spec mismatch): the manual recursion below assumes a ZERO-mean
    # process (sigma2[t] = omega + alpha*r[t-1]^2 + beta*sigma2[t-1], applied
    # directly to raw returns, not residuals around an estimated mu). arch_model's
    # default mean='Constant' would fit a separate mu, making the fitted
    # omega/alpha/beta inconsistent with how we apply them afterward. We fit with
    # mean='Zero' so the parameters are self-consistent with the manual recursion,
    # and dist='t' (Student-t) for the fat tails real equity returns show — this is
    # the Bollerslev (1987) t-GARCH extension, 'A Conditionally Heteroskedastic Time
    # Series Model for Speculative Prices and Rates of Return', Rev. Econ. Stat. 69(3).
    am        = arch_model(ret_pct[:_gsplit], mean='Zero', vol='Garch', p=1, q=1, dist='t')
    garch_res = am.fit(disp='off', show_warning=False)

    omega = garch_res.params['omega']
    alpha = garch_res.params['alpha[1]']
    beta  = garch_res.params['beta[1]']
    print(f"  GARCH(1,1) params : ω={omega:.6f}  α={alpha:.4f}  β={beta:.4f}")
    print(f"  Persistence (α+β) : {alpha+beta:.4f}  (closer to 1.0 = longer volatility memory)")

    # Roll the fitted recursion forward through the FULL series (train+test) —
    # parameters come from train only, so this is the same "fit on train, apply
    # forward" pattern as the HMM and the StandardScaler above.
    sigma2 = np.full(len(ret_pct), np.nan)
    sigma2[0] = np.var(ret_pct[:_gsplit])
    for t in range(1, len(ret_pct)):
        sigma2[t] = omega + alpha*(ret_pct[t-1]**2) + beta*sigma2[t-1]
    garch_vol = np.sqrt(np.clip(sigma2, 0, None))
    df_features['GARCH_Vol'] = garch_vol   # in %-return units (same scale as ret_pct)

    test_garch_vol = garch_vol[_gsplit:]
    print(f"  Test-set GARCH vol (annualised): {test_garch_vol.mean()*np.sqrt(252):.1f}% avg  |  "
          f"range [{test_garch_vol.min()*np.sqrt(252):.1f}%, {test_garch_vol.max()*np.sqrt(252):.1f}%]")
    GARCH_AVAILABLE = True
    print("  ✅ GARCH(1,1) conditional vol stored in df_features['GARCH_Vol']")
except ImportError:
    print("  ⚠️  arch not installed — run: !pip install arch -q")
    GARCH_AVAILABLE = False
    df_features['GARCH_Vol'] = df_features['Rolling_Std_20'] * 100
except Exception as e:
    print(f"  ⚠️  GARCH failed: {e}")
    GARCH_AVAILABLE = False
    df_features['GARCH_Vol'] = df_features['Rolling_Std_20'] * 100

print("\n✅ Step 6b complete")


🔵 STEP 6b: HMM Regime Detection + GARCH(1,1)
  State means      : bull=+0.00156  bear=-0.00099
  Test-set regimes : 89.6% bull  |  10.4% bear
  Ensemble DirAcc in BULL regime : 46.8%  (n=481)
  Ensemble DirAcc in BEAR regime : 44.6%  (n=56)
  ✅ HMM fitted — regime labels stored in df_features['HMM_Regime' / 'HMM_Bull_Prob']

  GARCH(1,1) params : ω=0.070858  α=0.0989  β=0.8871
  Persistence (α+β) : 0.9860  (closer to 1.0 = longer volatility memory)
  Test-set GARCH vol (annualised): 27.1% avg  |  range [15.5%, 94.0%]
  ✅ GARCH(1,1) conditional vol stored in df_features['GARCH_Vol']

✅ Step 6b complete


## 📊 Step 6b-viz — HMM Regime Overlay & GARCH Volatility Charts

Three-panel diagnostic:
- **Top**: close price coloured green/red by HMM bull/bear regime
- **Middle**: GARCH(1,1) annualised conditional volatility vs rolling 20-day std — shows volatility clustering
- **Bottom**: HMM bull probability smoothed over time — regime confidence

In [15]:
print("\n📊 STEP 6b-viz: HMM Regime + GARCH Charts")
print("=" * 55)

fig_regime, axes_r = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig_regime.suptitle(f'{CONFIG["ticker"]} — HMM Regimes & GARCH Conditional Volatility',
                    fontsize=13, fontweight='bold')

close_full = df_features['Close'].squeeze()
dates_full = df_features.index
_split_idx = int(len(df_features) * (1 - CONFIG['test_size']))

# ── Panel 1: Price coloured by HMM regime ────────────────────────────────
ax1 = axes_r[0]
bull_col, bear_col = '#4CAF50', '#F44336'
regime_col = [bull_col if r == (bull_state if HMM_AVAILABLE else 1) else bear_col
              for r in df_features['HMM_Regime'].values]

for i in range(len(dates_full) - 1):
    ax1.plot(dates_full[i:i+2], close_full.values[i:i+2],
             color=regime_col[i], linewidth=0.9, alpha=0.85)

ax1.axvline(dates_full[_split_idx], color='white', linestyle='--', linewidth=1, alpha=0.5,
            label='Train/Test split')
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color=bull_col, label='Bull regime'),
                    Patch(color=bear_col, label='Bear regime'),
                    Patch(color='white',  label='Train|Test split')],
           loc='upper left', fontsize=9)
ax1.set_ylabel('Price', fontsize=10); ax1.grid(alpha=0.2)
ax1.set_title('Close Price coloured by HMM Regime', fontsize=10)

# ── Panel 2: GARCH conditional volatility (annualised) ───────────────────
ax2 = axes_r[1]
garch_ann = df_features['GARCH_Vol'].values * np.sqrt(252)  # annualise
ax2.fill_between(dates_full, garch_ann, alpha=0.4, color='#FFB300', label='GARCH vol (ann.)')
ax2.plot(dates_full, garch_ann, color='#FFB300', linewidth=0.8)
ax2.axvline(dates_full[_split_idx], color='white', linestyle='--', linewidth=1, alpha=0.5)
# Rolling std for comparison
roll_ann = df_features['Rolling_Std_20'].values * np.sqrt(252) * 100
ax2.plot(dates_full, roll_ann, color='#64B5F6', linewidth=0.8, linestyle='--',
         label='Rolling 20d std (ann.)')
ax2.set_ylabel('Ann. Volatility %', fontsize=10); ax2.legend(fontsize=9); ax2.grid(alpha=0.2)
ax2.set_title('GARCH(1,1) Conditional Volatility vs Rolling Std', fontsize=10)

# ── Panel 3: HMM bull probability ────────────────────────────────────────
ax3 = axes_r[2]
bull_prob_full = df_features['HMM_Bull_Prob'].values
ax3.fill_between(dates_full, bull_prob_full, 0.5, where=(bull_prob_full >= 0.5),
                 alpha=0.4, color='#4CAF50', label='Bull prob > 0.5')
ax3.fill_between(dates_full, bull_prob_full, 0.5, where=(bull_prob_full < 0.5),
                 alpha=0.4, color='#F44336', label='Bear prob > 0.5')
ax3.plot(dates_full, bull_prob_full, color='white', linewidth=0.7, alpha=0.8)
ax3.axhline(0.5, color='gray', linewidth=0.8, linestyle=':')
ax3.axvline(dates_full[_split_idx], color='white', linestyle='--', linewidth=1, alpha=0.5)
ax3.set_ylabel('Bull Probability', fontsize=10); ax3.legend(fontsize=9); ax3.grid(alpha=0.2)
ax3.set_title('HMM Bull Probability Over Time', fontsize=10)
ax3.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('hmm_regime_garch.png', dpi=130, bbox_inches='tight')
plt.show()
print("  ✅ HMM regime + GARCH chart saved → hmm_regime_garch.png")



📊 STEP 6b-viz: HMM Regime + GARCH Charts
  ✅ HMM regime + GARCH chart saved → hmm_regime_garch.png


## 📐 Step 6c — Avellaneda-Stoikov Market-Making Spread Model

Avellaneda & Stoikov (2008), *"High-Frequency Trading in a Limit Order Book"*, Quantitative Finance 8(3), 217-224. Derives a closed-form optimal bid-ask spread and an inventory-skewed reservation price as a function of volatility, risk aversion, and time-to-horizon. We apply it here on daily bars — not as a literal order-book quoting strategy (this model was built for HFT market-making, not next-day equity forecasting) — but as a **position-sizing / conviction signal**: on high-volatility days the model-implied spread widens, meaning the market itself is pricing in more uncertainty, which is a reasonable cue to trade smaller.

In [16]:
# ── Step 6c — Avellaneda-Stoikov Market-Making Spread Model ────────────────
# Avellaneda & Stoikov (2008): Quantitative Finance 8(3), 217-224.
#   Reservation price : r(t) = S(t) - q*gamma*sigma^2*(T-t)
#   Optimal spread    : delta* = gamma*sigma^2*(T-t) + (2/gamma)*ln(1 + gamma/kappa)
# S(t)=mid-price, q=inventory in [-1,+1], gamma=risk aversion, sigma=vol,
# T-t=1 trading day, kappa=order-arrival-rate parameter.
print("\n📐 STEP 6c: Avellaneda-Stoikov Market-Making Model")
print("=" * 55)

AS_GAMMA = CONFIG['as_gamma']
AS_KAPPA = CONFIG['as_kappa']
AS_T     = 1.0

n_test = len(y_test)
test_prices_as   = df_features['Close'].values[-n_test:]
test_garch_pct   = df_features['GARCH_Vol'].values[-n_test:]      # %-return units
test_bull_prob   = df_features['HMM_Bull_Prob'].values[-n_test:]

# Inventory proxy from the HMM regime signal, centred at 0 (range [-1,+1])
q_inventory = 2 * test_bull_prob - 1

# Convert %-return volatility into PRICE units: sigma_price = sigma_pct/100 * price
sigma_price = (test_garch_pct / 100) * test_prices_as

reservation_price = test_prices_as - q_inventory * AS_GAMMA * (sigma_price**2) * AS_T
half_spread = (AS_GAMMA * (sigma_price**2) * AS_T
               + (2.0 / AS_GAMMA) * np.log(1.0 + AS_GAMMA / AS_KAPPA))
optimal_bid  = reservation_price - half_spread
optimal_ask  = reservation_price + half_spread
full_spread  = 2 * half_spread

mid_px = test_prices_as.mean()
# FIX (vs. the draft script this replaces): the original had
# `half_spread.mean() / mid_px100` — an undefined variable name from a
# missing `*` between `mid_px` and `100`. Below is the corrected,
# dimensionally-consistent version (spread divided by price, times 100 for %).
avg_half_spread     = float(half_spread.mean())
avg_full_spread_pct = float(full_spread.mean() / mid_px * 100)

print(f"  Parameters       : gamma={AS_GAMMA}  kappa={AS_KAPPA}  T={AS_T}")
print(f"  Avg mid-price    : {mid_px:.2f}")
print(f"  Avg half-spread  : {avg_half_spread:.4f}  ({avg_half_spread/mid_px*100:.4f}% of price)")
print(f"  Avg full spread  : {full_spread.mean():.4f}  ({avg_full_spread_pct:.4f}% of price)")
print(f"  Reservation Δ    : {(reservation_price - test_prices_as).mean():+.4f}  (avg offset from mid)")
print(f"  Spread range     : [{full_spread.min():.4f}, {full_spread.max():.4f}]")

as_results = {
    'gamma'                : AS_GAMMA,
    'kappa'                : AS_KAPPA,
    'avg_mid_price'        : round(float(mid_px), 2),
    'avg_half_spread'      : round(avg_half_spread, 4),
    'avg_full_spread_pct'  : round(avg_full_spread_pct, 4),
    'avg_reservation_delta': round(float((reservation_price - test_prices_as).mean()), 4),
    'spread_volatility'    : round(float(full_spread.std()), 4),
}

fig_as = go.Figure()
fig_as.add_trace(go.Scatter(x=list(test_dates), y=full_spread,
    name='A-S Optimal Spread', line=dict(color='#FFB300', width=1.5)))
fig_as.add_trace(go.Scatter(x=list(test_dates), y=sigma_price,
    name='GARCH σ (price units)', line=dict(color='#64B5F6', width=1, dash='dash')))
fig_as.update_layout(template='plotly_dark',
    title=f'Avellaneda-Stoikov Optimal Spread — {CONFIG["ticker"]}',
    xaxis_title='Date', yaxis_title='Price units', height=380)
fig_as.write_html('avellaneda_stoikov_spread.html')
fig_as.show()

print(f"\n  ✅ A-S model complete | avg_full_spread_pct={as_results['avg_full_spread_pct']:.4f}%")
print(f"  Interpretation: on high-vol days the spread widens -> the model is less")
print(f"  certain -> size trades smaller. Use avg_full_spread_pct as a rough")
print(f"  transaction-cost lower bound: only act on predicted moves bigger than it.")


📐 STEP 6c: Avellaneda-Stoikov Market-Making Model
  Parameters       : gamma=0.1  kappa=1.5  T=1.0
  Avg mid-price    : 236.69
  Avg half-spread  : 3.0286  (1.2796% of price)
  Avg full spread  : 6.0572  (2.5591% of price)
  Reservation Δ    : -0.7080  (avg offset from mid)
  Spread range     : [3.5277, 28.2625]



  ✅ A-S model complete | avg_full_spread_pct=2.5591%
  Interpretation: on high-vol days the spread widens -> the model is less
  certain -> size trades smaller. Use avg_full_spread_pct as a rough
  transaction-cost lower bound: only act on predicted moves bigger than it.


## **Intraday Data Training**

In [17]:
def retrain_on_intraday(ticker, base_df_features, feature_cols, scaler):
    """
    Downloads today's intraday 5-min bars, converts them to features,
    appends to training data, and fine-tunes the XGBoost model on the
    combined dataset. Takes ~30 seconds. Run once per day at market open.

    This is what separates a static model from a live-aware one.
    The model now sees today's morning price action before predicting close.
    """
    print(f"\n🔄 Intraday fine-tune for {ticker}...")

    try:
        # ── Download today's 5-min bars ───────────────────────────────
        df_intra = yf.download(
            ticker, period='1d', interval='5m',
            auto_adjust=True, progress=False
        )
        if isinstance(df_intra.columns, pd.MultiIndex):
            df_intra.columns = df_intra.columns.get_level_values(0)
        df_intra = df_intra.dropna()

        if df_intra.empty or len(df_intra) < 20:
            print("  ⚠️  Not enough intraday bars — skipping fine-tune")
            return xgb_model, scaler

        # ── Engineer features on intraday bars ───────────────────────
        # Need market context columns — add zeros for now (they won't exist
        # in intraday data, but the model handles zero-fill)
        df_intra_feat = engineer_features(df_intra)
        common_cols   = [c for c in feature_cols if c in df_intra_feat.columns]

        # Build feature matrix — zero-fill missing cols
        X_intra = np.zeros((len(df_intra_feat), len(feature_cols)))
        for i, col in enumerate(feature_cols):
            if col in df_intra_feat.columns:
                X_intra[:, i] = df_intra_feat[col].values

        y_intra = df_intra_feat['Target'].values
        if len(y_intra) == 0:
            print("  ⚠️  No target rows after feature engineering — skipping")
            return xgb_model, scaler

        # ── Scale with existing scaler (do NOT refit — that leaks test data) ──
        X_intra_sc = scaler.transform(X_intra)

        # ── Fine-tune: continue training XGBoost on intraday data ────
        # xgb_model.fit with xgb_model as base (warm start via set_params)
        xgb_model_live = xgb.XGBRegressor(**best_xgb_params, verbosity=0)
        xgb_model_live.fit(
            X_intra_sc, y_intra,
            xgb_model=xgb_model.get_booster(),  # start from existing model
            verbose=False
        )

        print(f"  ✅ Fine-tuned on {len(df_intra_feat)} intraday bars")
        print(f"  Intraday price range: {df_intra['Close'].min():.2f} – "
              f"{df_intra['Close'].max():.2f}")
        return xgb_model_live, scaler

    except Exception as e:
        print(f"  ⚠️  Fine-tune failed: {e} — using original model")
        return xgb_model, scaler


# ── DISABLED BY DEFAULT ─────────────────────────────────────────────────
# This was silently failing on EVERY run: one trading day of 5-min bars is
# ~78 rows, below the 200-row minimum engineer_features() needs for SMA_200,
# so the fine-tune always hit the except-block and no-op'd — while this cell
# still printed "✅ Live-aware model active" regardless, which is a false-
# success message.
#
# Even with enough rows, this fine-tunes a model trained on DAILY log-returns
# using 5-MINUTE log-returns as the label, through a scaler fit on daily-scale
# feature distributions — mixing timeframes like that would corrupt the model
# rather than improve it. A correct version needs a separate intraday
# feature/label pipeline; that's a redesign, not a bug fix, so it's left off
# here rather than "fixed" by just lowering the row threshold.
ENABLE_INTRADAY_FINETUNE = False

if ENABLE_INTRADAY_FINETUNE:
    xgb_model_live, _ = retrain_on_intraday(
        CONFIG['ticker'], df_features, feature_cols, scaler
    )
    xgb_model = xgb_model_live
    print("✅ Live-aware model active")
else:
    print("⏭️  Intraday fine-tune disabled by default (see comment above) — using the daily-trained xgb_model as-is")

⏭️  Intraday fine-tune disabled by default (see comment above) — using the daily-trained xgb_model as-is


## 🔴 Step 7 — LSTM

In [18]:
def create_sequences(X, y, lb):
    return (np.array([X[i-lb:i] for i in range(lb,len(X))]),
            np.array([y[i]        for i in range(lb,len(X))]))

lb = 30  # reduced from CONFIG['look_back']=60 for CPU speed
         # LSTM is weakest model (RMSE ~0.10), halving sequences
         # cuts training time ~50% with negligible quality impact

X_all_raw=df_features[feature_cols].values; y_all=df_features['Target'].values
sp_raw=int(len(X_all_raw)*(1-CONFIG['test_size']))

# Bug 1 Fix: Dedicated LSTM scaler fit ONLY on train portion
lstm_scaler=StandardScaler()
X_all_sc=np.vstack([
    lstm_scaler.fit_transform(X_all_raw[:sp_raw]),
    lstm_scaler.transform(X_all_raw[sp_raw:])
])

X_seq,y_seq=create_sequences(X_all_sc,y_all,lb)
sp=int(len(X_seq)*(1-CONFIG['test_size']))
X_lt,X_le=X_seq[:sp],X_seq[sp:]
y_lt,y_le=y_seq[:sp],y_seq[sp:]
lstm_test_dates=df_features.index[lb+sp:]

lstm_model=Sequential([
    LSTM(128,return_sequences=True,input_shape=(lb,X_lt.shape[2])),
    BatchNormalization(),Dropout(0.3),
    LSTM(64,return_sequences=True),BatchNormalization(),Dropout(0.2),
    LSTM(32),Dropout(0.2),Dense(16,activation='relu'),Dense(1)
])
lstm_model.compile(optimizer=Adam(1e-3),loss='huber',metrics=['mae'])
lstm_model.summary()
history=lstm_model.fit(X_lt,y_lt,validation_split=0.15,epochs=100,batch_size=32,shuffle=False,
    callbacks=[EarlyStopping(patience=5,restore_best_weights=True),  # reduced from 15 for CPU
               ReduceLROnPlateau(factor=0.5,patience=7,min_lr=1e-6)],verbose=1)
y_pred_lstm=lstm_model.predict(X_le).flatten()
predictions['LSTM']=(y_pred_lstm,lstm_test_dates)
m_lstm=evaluate_model(y_le,y_pred_lstm,'LSTM'); metrics_list.append(m_lstm)
print(f"LSTM: RMSE={m_lstm['RMSE']} DirAcc={m_lstm['Directional Acc %']}%")
lstm_model.save('lstm_stock_model.keras')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │        97,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,417 (626.63 KB)

 Trainable params: 160,033 (625.13 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 13s 94ms/step - loss: 0.0228 - mae: 0.1547 - val_loss: 0.0092 - val_mae: 0.1101 - learning_rate: 0.0010
Epoch 2/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 10s 90ms/step - loss: 0.0046 - mae: 0.0715 - val_loss: 0.0053 - val_mae: 0.0850 - learning_rate: 0.0010
Epoch 3/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - loss: 0.0017 - mae: 0.0409 - val_loss: 0.0041 - val_mae: 0.0728 - learning_rate: 0.0010
Epoch 4/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - loss: 0.0012 - mae: 0.0323 - val_loss: 0.0043 - val_mae: 0.0715 - learning_rate: 0.0010
Epoch 5/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 85ms/step - loss: 7.5162e-04 - mae: 0.0254 - val_loss: 0.0066 - val_mae: 0.0898 - learning_rate: 0.0010
Epoch 6/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - loss: 4.7059e-04 - mae: 0.0200 - val_loss: 0.0125 - val_mae: 0.1183 - learning_rate: 0.0010
Epoch 7/100
57/57 ━━━━━━━━━━━━━━━━━━━━ 6s 89ms/step - loss: 4.3270e-04 - mae: 0.0189 - val_loss: 0.0190 - val_mae: 0.1430 - learning_rate:

## 📊 Step 7b — LSTM Training Curve

Loss and validation loss per epoch. Confirms whether early-stopping fired at the right point, whether the model overfit, and how quickly it converged.

In [19]:
print("\n📊 STEP 7b: LSTM Training Curve")
print("=" * 55)

fig_lstm, axes_l = plt.subplots(1, 2, figsize=(14, 5))
fig_lstm.suptitle('LSTM Training Diagnostics', fontsize=13, fontweight='bold')

epochs_ran = range(1, len(history.history['loss']) + 1)

# ── Panel 1: Loss curves ─────────────────────────────────────────────────
ax_l = axes_l[0]
ax_l.plot(epochs_ran, history.history['loss'],     color='#2196F3', linewidth=1.8, label='Train loss (Huber)')
ax_l.plot(epochs_ran, history.history['val_loss'], color='#F44336', linewidth=1.8,
          linestyle='--', label='Val loss (Huber)')
best_epoch = int(np.argmin(history.history['val_loss'])) + 1
ax_l.axvline(best_epoch, color='#FFB300', linewidth=1.2, linestyle=':', label=f'Best epoch ({best_epoch})')
ax_l.set_xlabel('Epoch'); ax_l.set_ylabel('Huber Loss')
ax_l.set_title('Train vs Validation Loss'); ax_l.legend(fontsize=9); ax_l.grid(alpha=0.3)
ax_l.text(0.02, 0.97, f'Epochs ran: {len(epochs_ran)}  |  Best: {best_epoch}',
          transform=ax_l.transAxes, fontsize=8, va='top', color='gray')

# ── Panel 2: MAE curves ───────────────────────────────────────────────────
ax_r = axes_l[1]
ax_r.plot(epochs_ran, history.history['mae'],     color='#4CAF50', linewidth=1.8, label='Train MAE')
ax_r.plot(epochs_ran, history.history['val_mae'], color='#FF9800', linewidth=1.8,
          linestyle='--', label='Val MAE')
ax_r.axvline(best_epoch, color='#FFB300', linewidth=1.2, linestyle=':')
ax_r.set_xlabel('Epoch'); ax_r.set_ylabel('MAE')
ax_r.set_title('Train vs Validation MAE'); ax_r.legend(fontsize=9); ax_r.grid(alpha=0.3)

# Overfitting flag
gap = (np.array(history.history['val_loss']) - np.array(history.history['loss']))
if gap[-1] > gap[0] * 2:
    axes_l[0].set_title('Train vs Validation Loss  ⚠️ overfit signal', color='#F44336', fontsize=10)

plt.tight_layout()
plt.savefig('lstm_training_curve.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"  Best val_loss = {min(history.history['val_loss']):.6f} at epoch {best_epoch}")
print(f"  Final train/val gap = {gap[-1]:+.6f}  ({'⚠️  overfit' if gap[-1] > 0.001 else '✅  ok'})")
print("  ✅ LSTM training curve saved → lstm_training_curve.png")



📊 STEP 7b: LSTM Training Curve
  Best val_loss = 0.004133 at epoch 3
  Final train/val gap = +0.028379  (⚠️  overfit)
  ✅ LSTM training curve saved → lstm_training_curve.png


## 🔄 Step 8 — Walk-Forward Validation

In [20]:
def walk_forward_validation(df, feature_cols, train_months=24, test_months=3):
    df = df.copy(); df['YM'] = df.index.to_period('M')
    periods = df['YM'].unique(); results = []

    for i in range(train_months, len(periods) - test_months, test_months):
        tr = df[df['YM'].isin(periods[i - train_months:i])]
        te = df[df['YM'].isin(periods[i:i + test_months])]

        # Drop NaN targets — last row of each window has NaN from shift(-1)
        tr = tr.dropna(subset=['Target'])
        te = te.dropna(subset=['Target'])

        if len(te) < 5: continue

        sc  = StandardScaler()
        Xtr = sc.fit_transform(tr[feature_cols])
        Xte = sc.transform(te[feature_cols])

        m   = xgb.XGBRegressor(**best_xgb_params, verbosity=0).fit(Xtr, tr['Target'])
        row = evaluate_model(te['Target'].values, m.predict(Xte), f'W{i}')
        row['test_start'] = str(periods[i]); results.append(row)

        print(f"  W{i}: {periods[i]} RMSE={row['RMSE']:.6f} "
              f"DirAcc={row['Directional Acc %']:.1f}%")

    wf = pd.DataFrame(results)
    print(f"\nWF: {len(wf)} windows | "
          f"Mean RMSE={wf['RMSE'].mean():.6f} ± {wf['RMSE'].std():.6f} | "
          f"Mean DirAcc={wf['Directional Acc %'].mean():.1f}% "
          f"± {wf['Directional Acc %'].std():.1f}%")
    return wf

wf_results = walk_forward_validation(df_features, feature_cols)


  W24: 2017-10 RMSE=0.011250 DirAcc=54.0%
  W27: 2018-01 RMSE=0.016772 DirAcc=44.3%
  W30: 2018-04 RMSE=0.013300 DirAcc=53.1%
  W33: 2018-07 RMSE=0.013060 DirAcc=65.1%
  W36: 2018-10 RMSE=0.026229 DirAcc=42.9%
  W39: 2019-01 RMSE=0.020911 DirAcc=65.6%
  W42: 2019-04 RMSE=0.016285 DirAcc=49.2%
  W45: 2019-07 RMSE=0.016550 DirAcc=54.7%
  W48: 2019-10 RMSE=0.012011 DirAcc=64.1%
  W51: 2020-01 RMSE=0.042245 DirAcc=43.5%
  W54: 2020-04 RMSE=0.021374 DirAcc=44.4%
  W57: 2020-07 RMSE=0.027917 DirAcc=57.8%
  W60: 2020-10 RMSE=0.021676 DirAcc=46.9%
  W63: 2021-01 RMSE=0.020602 DirAcc=47.5%
  W66: 2021-04 RMSE=0.013051 DirAcc=54.0%
  W69: 2021-07 RMSE=0.012990 DirAcc=54.7%
  W72: 2021-10 RMSE=0.015292 DirAcc=57.8%
  W75: 2022-01 RMSE=0.018711 DirAcc=45.2%
  W78: 2022-04 RMSE=0.026263 DirAcc=48.4%
  W81: 2022-07 RMSE=0.019450 DirAcc=48.4%
  W84: 2022-10 RMSE=0.025212 DirAcc=44.4%
  W87: 2023-01 RMSE=0.014922 DirAcc=35.5%
  W90: 2023-04 RMSE=0.011557 DirAcc=50.0%
  W93: 2023-07 RMSE=0.013713 DirAc

## 📊 Step 8c — Walk-Forward Performance Drift

RMSE and directional accuracy per rolling window over time. A model that degrades steadily is overfitting to the training distribution; one that is stable is generalising. Drift toward 50% DirAcc means the signal is decaying.

In [21]:
print("\n📊 STEP 8c: Walk-Forward Performance Drift")
print("=" * 55)

fig_wf, axes_wf = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig_wf.suptitle(f'{CONFIG["ticker"]} — Walk-Forward Validation Drift', fontsize=13, fontweight='bold')

wf_x = range(len(wf_results))
wf_labels = wf_results['test_start'].values if 'test_start' in wf_results.columns else [f'W{i}' for i in wf_x]

# ── Panel 1: RMSE per window with ±1σ band ───────────────────────────────
ax1 = axes_wf[0]
rmse_vals = wf_results['RMSE'].values
rmse_mean = rmse_vals.mean(); rmse_std = rmse_vals.std()
ax1.bar(wf_x, rmse_vals, color='#2196F3', alpha=0.7, label='Window RMSE')
ax1.axhline(rmse_mean, color='white',  linewidth=1.2, linestyle='--', label=f'Mean={rmse_mean:.5f}')
ax1.axhline(rmse_mean + rmse_std, color='#F44336', linewidth=0.8, linestyle=':', label=f'+1σ={rmse_mean+rmse_std:.5f}')
ax1.axhline(rmse_mean - rmse_std, color='#4CAF50', linewidth=0.8, linestyle=':')
ax1.set_ylabel('RMSE'); ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)
ax1.set_title('RMSE per Walk-Forward Window', fontsize=10)
ax1.set_xticks(list(wf_x))
ax1.set_xticklabels(wf_labels, rotation=35, ha='right', fontsize=7)

# ── Panel 2: Directional accuracy per window ──────────────────────────────
ax2 = axes_wf[1]
da_vals = wf_results['Directional Acc %'].values
da_mean = da_vals.mean()
colors_da = ['#4CAF50' if v >= 55 else ('#FFB300' if v >= 50 else '#F44336') for v in da_vals]
ax2.bar(wf_x, da_vals, color=colors_da, alpha=0.8, label='Window DirAcc %')
ax2.axhline(da_mean, color='white',  linewidth=1.2, linestyle='--', label=f'Mean={da_mean:.1f}%')
ax2.axhline(50, color='#F44336', linewidth=1.0, linestyle=':', label='50% (random)')
ax2.axhline(55, color='#4CAF50', linewidth=1.0, linestyle=':', label='55% (signal threshold)')
ax2.set_ylabel('Directional Accuracy %'); ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)
ax2.set_title('Directional Accuracy per Walk-Forward Window', fontsize=10)
ax2.set_xticks(list(wf_x))
ax2.set_xticklabels(wf_labels, rotation=35, ha='right', fontsize=7)
ax2.set_ylim(30, 80)

plt.tight_layout()
plt.savefig('walkforward_drift.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"  RMSE: mean={rmse_mean:.6f} ± {rmse_std:.6f}")
print(f"  DirAcc: mean={da_mean:.1f}% | min={da_vals.min():.1f}% | max={da_vals.max():.1f}%")
degrading = np.polyfit(range(len(da_vals)), da_vals, 1)[0]
print(f"  DirAcc trend slope: {degrading:+.3f}%/window ({'⚠️  degrading' if degrading < -0.5 else '✅  stable'})")
print("  ✅ Walk-forward drift chart saved → walkforward_drift.png")



📊 STEP 8c: Walk-Forward Performance Drift
  RMSE: mean=0.017914 ± 0.006753
  DirAcc: mean=51.1% | min=35.5% | max=65.6%
  DirAcc trend slope: -0.094%/window (✅  stable)
  ✅ Walk-forward drift chart saved → walkforward_drift.png


## 📐 Step 8b — Stationarity & Residual Autocorrelation Tests

Said & Dickey (1984), *"Testing for Unit Roots in Autoregressive-Moving Average Models of Unknown Order"*, Biometrika 71(3) — the Augmented Dickey-Fuller test. Null hypothesis: the series has a unit root (is non-stationary). We expect log-returns to reject that null; if they didn't, every downstream model would be fitting a moving target.

Ljung & Box (1978), *"On a Measure of Lack of Fit in Time Series Models"*, Biometrika 65(2) — Q-test for leftover autocorrelation in the ensemble's residuals. If residuals are still autocorrelated at p<0.05, the model has left exploitable structure on the table; if not, there's no obvious linear pattern left for *this* feature set to capture.

In [22]:
# ── Step 8b — Stationarity & Residual Autocorrelation Tests ────────────────
print("\n📊 STEP 8b: Stationarity & Autocorrelation Tests")
print("=" * 55)

from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# ── 1. ADF test on log-returns (should be stationary) ─────────────────────
log_ret_series = df_features['Log_Return'].dropna()
adf_stat, adf_p, adf_lags, _, adf_crit, _ = adfuller(log_ret_series, autolag='AIC')
print(f"\n  Augmented Dickey-Fuller on Log_Return:")
print(f"    ADF stat   = {adf_stat:.4f}")
print(f"    p-value    = {adf_p:.6f}  "
      f"{'✅ stationary (p<0.05)' if adf_p < 0.05 else '❌ unit root not rejected'}")
print(f"    Lags used  = {adf_lags}")
print(f"    Crit 1%/5% = {adf_crit['1%']:.4f} / {adf_crit['5%']:.4f}")

# ── 2. Ljung-Box on ensemble residuals ─────────────────────────────────────
residuals = y_test - y_pred_ensemble
lb_result = acorr_ljungbox(residuals, lags=[5, 10, 20], return_df=True)
print(f"\n  Ljung-Box Q-test on Ensemble residuals:")
print(f"    {'Lag':<6} {'Q-stat':>10} {'p-value':>10} {'Verdict':>14}")
for lag, row in lb_result.iterrows():
    verdict = '✅ no AC left' if row['lb_pvalue'] > 0.05 else '⚠️  AC present'
    print(f"    {int(lag):<6} {row['lb_stat']:>10.4f} {row['lb_pvalue']:>10.4f} {verdict:>14}")

# ── 3. ACF / PACF diagnostic plots ─────────────────────────────────────────
fig_diag, axes = plt.subplots(2, 2, figsize=(14, 8))
fig_diag.suptitle(f'{CONFIG["ticker"]} — Return Structure Diagnostics', fontsize=13, fontweight='bold')

axes[0,0].plot(log_ret_series.values[-250:], color='#2196F3', linewidth=0.8)
axes[0,0].set_title('Log_Return — last 250 obs'); axes[0,0].grid(alpha=0.3)

acf_vals  = acf(log_ret_series, nlags=20, fft=True)
axes[0,1].stem(range(len(acf_vals)), acf_vals)
axes[0,1].axhline(1.96/np.sqrt(len(log_ret_series)), color='r', linestyle='--', linewidth=0.8)
axes[0,1].axhline(-1.96/np.sqrt(len(log_ret_series)), color='r', linestyle='--', linewidth=0.8)
axes[0,1].set_title('ACF — Log_Return')

pacf_vals = pacf(log_ret_series, nlags=20)
axes[1,0].stem(range(len(pacf_vals)), pacf_vals)
axes[1,0].axhline(1.96/np.sqrt(len(log_ret_series)), color='r', linestyle='--', linewidth=0.8)
axes[1,0].axhline(-1.96/np.sqrt(len(log_ret_series)), color='r', linestyle='--', linewidth=0.8)
axes[1,0].set_title('PACF — Log_Return')

resid_acf = acf(pd.Series(residuals).dropna(), nlags=20, fft=True)
axes[1,1].stem(range(len(resid_acf)), resid_acf)
axes[1,1].axhline(1.96/np.sqrt(len(residuals)), color='r', linestyle='--', linewidth=0.8)
axes[1,1].axhline(-1.96/np.sqrt(len(residuals)), color='r', linestyle='--', linewidth=0.8)
axes[1,1].set_title('ACF — Ensemble Residuals')

plt.tight_layout()
plt.savefig('stationarity_diagnostics.png', dpi=130, bbox_inches='tight')
plt.show()

print("\n✅ Step 8b complete")


📊 STEP 8b: Stationarity & Autocorrelation Tests

  Augmented Dickey-Fuller on Log_Return:
    ADF stat   = -16.8186
    p-value    = 0.000000  ✅ stationary (p<0.05)
    Lags used  = 8
    Crit 1%/5% = -3.4328 / -2.8626

  Ljung-Box Q-test on Ensemble residuals:
    Lag        Q-stat    p-value        Verdict
    5         12.6752     0.0266 ⚠️  AC present
    10        16.0638     0.0978   ✅ no AC left
    20        26.8218     0.1404   ✅ no AC left

✅ Step 8b complete


## 📊 Step 8d — Residual Distribution & QQ-Plot

Fat tails in prediction errors mean your model underestimates tail risk. The QQ-plot against a normal distribution reveals skewness and kurtosis at a glance. If residuals are systematically biased (mean ≠ 0), the model has a directional lean.

In [23]:
print("\n📊 STEP 8d: Residual Distribution & QQ-Plot")
print("=" * 55)

from scipy import stats as _scipy_stats

residuals = y_test - y_pred_ensemble
resid_s   = pd.Series(residuals).dropna()

fig_res, axes_res = plt.subplots(1, 3, figsize=(16, 5))
fig_res.suptitle(f'{CONFIG["ticker"]} — Ensemble Residual Distribution', fontsize=13, fontweight='bold')

# ── Panel 1: Histogram with normal overlay ────────────────────────────────
ax1 = axes_res[0]
mu, sigma = resid_s.mean(), resid_s.std()
ax1.hist(resid_s.values, bins=50, color='#2196F3', alpha=0.7, density=True, label='Residuals')
xr = np.linspace(resid_s.min(), resid_s.max(), 200)
ax1.plot(xr, _scipy_stats.norm.pdf(xr, mu, sigma), color='#F44336', linewidth=2, label='Normal fit')
ax1.axvline(0,  color='white', linewidth=1.0, linestyle='--')
ax1.axvline(mu, color='#FFB300', linewidth=1.2, linestyle=':', label=f'Mean={mu:.5f}')
ax1.set_xlabel('Residual (log-return)'); ax1.set_ylabel('Density')
ax1.set_title('Residual Histogram'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
skew_ = float(_scipy_stats.skew(resid_s)); kurt_ = float(_scipy_stats.kurtosis(resid_s))
ax1.text(0.02, 0.97, f'Skew={skew_:.3f}  Kurt={kurt_:.3f}', transform=ax1.transAxes,
         fontsize=8, va='top', color='gray')

# ── Panel 2: QQ-plot ───────────────────────────────────────────────────────
ax2 = axes_res[1]
(osm, osr), (slope, intercept, r) = _scipy_stats.probplot(resid_s, dist='norm')
ax2.scatter(osm, osr, color='#2196F3', s=8, alpha=0.6, label='Sample quantiles')
ax2.plot(osm, slope * np.array(osm) + intercept, color='#F44336', linewidth=1.5, label='Normal line')
ax2.set_xlabel('Theoretical quantiles'); ax2.set_ylabel('Sample quantiles')
ax2.set_title(f'QQ-Plot  (R²={r**2:.4f})'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
if abs(kurt_) > 3:
    ax2.set_title(f'QQ-Plot  (R²={r**2:.4f})  ⚠️ fat tails', color='#FFB300', fontsize=9)

# ── Panel 3: Residuals over time (drift check) ────────────────────────────
ax3 = axes_res[2]
ax3.plot(test_dates[:len(residuals)], residuals, color='#2196F3', linewidth=0.7, alpha=0.7)
ax3.axhline(0, color='white', linewidth=0.8, linestyle='--')
roll_bias = pd.Series(residuals).rolling(30).mean().values
ax3.plot(test_dates[:len(roll_bias)], roll_bias, color='#F44336', linewidth=1.5, label='30d rolling mean')
ax3.set_xlabel('Date'); ax3.set_ylabel('Residual')
ax3.set_title('Residuals Over Time (bias check)'); ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('residual_distribution.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"  Mean={mu:.6f}  Std={sigma:.6f}  Skew={skew_:.4f}  Excess-Kurt={kurt_:.4f}")
print(f"  {'⚠️  Fat tails detected' if abs(kurt_) > 3 else '✅  Near-normal tails'}")
print(f"  {'⚠️  Bias: mean residual non-zero' if abs(mu) > sigma*0.1 else '✅  No systematic bias'}")
print("  ✅ Residual distribution chart saved → residual_distribution.png")



📊 STEP 8d: Residual Distribution & QQ-Plot
  Mean=0.003575  Std=0.017448  Skew=0.1418  Excess-Kurt=8.6181
  ⚠️  Fat tails detected
  ⚠️  Bias: mean residual non-zero
  ✅ Residual distribution chart saved → residual_distribution.png


## 🔍 Step 9 — SHAP

In [24]:
explainer=shap.TreeExplainer(xgb_model)
shap_values=explainer.shap_values(X_test)
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values,X_test,feature_names=feature_cols,plot_type='bar',show=False)
plt.title('SHAP Feature Importance'); plt.tight_layout()
plt.savefig('shap_summary.png',dpi=150,bbox_inches='tight'); plt.show()
plt.figure(figsize=(10,8))
shap.summary_plot(shap_values,X_test,feature_names=feature_cols,show=False)
plt.tight_layout(); plt.savefig('shap_beeswarm.png',dpi=150,bbox_inches='tight'); plt.show()
shap.waterfall_plot(shap.Explanation(values=shap_values[-1],base_values=explainer.expected_value,
    data=X_test[-1],feature_names=feature_cols),show=True)
print('✅ SHAP done')


✅ SHAP done


## 🎯 Step 10 — Confusion Matrices

In [25]:
for name,pred in predictions.items():
    if name=='LSTM': plot_confusion_matrix(y_le,pred[0],'LSTM')
    else:            plot_confusion_matrix(y_test,pred,name)


  Precision: 68.00% | Recall: 22.67%
  Precision: 69.07% | Recall: 22.33%
  Precision: 52.94% | Recall: 36.00%
  Precision: 72.31% | Recall: 15.67%
  Precision: 63.33% | Recall: 6.33%
  Precision: 65.12% | Recall: 9.33%
  Precision: 55.11% | Recall: 60.34%


## 📅 Step 11 — Multi-Step Forecasting

In [26]:
def multistep_forecast(df, feature_cols, scaler, horizons=(1,3,5,10)):
    """
    FIX (root cause of the 'only the white Actual line shows, all Pred lines
    sit flat at the base' bug): the old version plotted Actual_T1 in raw
    PRICE units (close.values[sp:], e.g. ~$150-300) on the SAME axes as
    Pred_T{h}, which were raw LOG-RETURN predictions (tiny values, typically
    +-0.001 to +-0.02). On a shared y-axis, the price line dwarfs the
    return lines so completely that the return lines are visually
    indistinguishable from zero — exactly the symptom reported.

    Fix: reconstruct each horizon's PREDICTED PRICE LEVEL as
    anchor_price * exp(predicted_log_return), and plot it against the
    REALISED price at the target date (anchor_date + h trading days) —
    not against the anchor date. That gives an apples-to-apples
    price-vs-price comparison, and price-level RMSE is reported alongside
    the original log-return RMSE.
    """
    close = df['Close'].squeeze()
    X_raw = df[feature_cols].values
    sp    = int(len(X_raw) * (1 - CONFIG['test_size']))
    Xtr   = scaler.transform(X_raw[:sp])
    Xte   = scaler.transform(X_raw[sp:])
    n_rows = len(df)

    print('\n📅 Multi-step forecast:')
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=close.values, name='Actual Close',
                              line=dict(color='white', width=2)))

    horizon_frames = {}
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    for h, color in zip(horizons, colors):
        t = np.log(close.shift(-h) / close).values
        ytr, yte = t[:sp], t[sp:]
        v = ~np.isnan(ytr)
        m = xgb.XGBRegressor(**best_xgb_params, verbosity=0).fit(Xtr[v], ytr[v])

        vt = ~np.isnan(yte)
        ph = np.full(len(Xte), np.nan)
        ph[vt] = m.predict(Xte[vt])
        rmse_logret = np.sqrt(mean_squared_error(yte[vt], ph[vt]))

        # ── Reconstruct predicted PRICE level and align to the TARGET date ──
        anchor_pos    = sp + np.where(vt)[0]            # row index of the anchor (today)
        anchor_prices = close.values[anchor_pos]
        pred_price    = anchor_prices * np.exp(ph[vt])  # price implied by predicted return

        target_pos    = anchor_pos + h                  # row index h days AHEAD (target)
        in_range      = target_pos < n_rows
        target_dates  = df.index[target_pos[in_range]]
        target_actual = close.values[target_pos[in_range]]
        pred_price_al = pred_price[in_range]

        rmse_price = np.sqrt(mean_squared_error(target_actual, pred_price_al))
        print(f'  T+{h:2d}: RMSE(log-ret)={rmse_logret:.6f}  |  RMSE(price)=${rmse_price:.3f}  (n={in_range.sum()})')

        fig.add_trace(go.Scatter(
            x=target_dates, y=pred_price_al, name=f'Pred_T{h} (price)',
            line=dict(color=color, dash='dash', width=1.6)))

        horizon_frames[f'Pred_T{h}_logret'] = pd.Series(ph, index=df.index[sp:])
        horizon_frames[f'Pred_T{h}_price']  = pd.Series(pred_price_al, index=target_dates).reindex(df.index)

    fig.update_layout(template='plotly_dark',
        title='Multi-Step Forecast — predicted PRICE level vs realised price at target date',
        xaxis_title='Date', yaxis_title=f'Price', height=480)
    fig.write_html('multistep_forecast.html'); fig.show()

    df_r = pd.DataFrame({'Date': df.index, 'Actual_Close': close.values}).set_index('Date')
    for k, v in horizon_frames.items():
        df_r[k] = v
    return df_r

multistep_df = multistep_forecast(df_features, feature_cols, scaler, CONFIG['multistep_days'])



📅 Multi-step forecast:
  T+ 1: RMSE(log-ret)=0.017641  |  RMSE(price)=$3.941  (n=536)
  T+ 3: RMSE(log-ret)=0.032379  |  RMSE(price)=$7.258  (n=534)
  T+ 5: RMSE(log-ret)=0.041981  |  RMSE(price)=$9.506  (n=532)
  T+10: RMSE(log-ret)=0.058561  |  RMSE(price)=$13.526  (n=527)


## 📊 Step 12 — Model Performance Summary

In [27]:
metrics_df=pd.DataFrame(metrics_list).set_index('Model')
print('\nMODEL PERFORMANCE SUMMARY\n'+'='*90)
print(metrics_df[['RMSE', 'RMSE_CI_95', 'MAE', 'R2',
                   'Directional Acc %', 'P-Value', 'Sig (p<0.05)']].to_string())

# MAPE excluded from all tickers:
# Target = log-returns (values ~0.001–0.02).
# Dividing by near-zero denominators produces 100x–3000x% errors
# regardless of stock. MAPE is only meaningful for price-level targets.
# Use RMSE + Directional Accuracy as primary metrics instead.
print('='*90)
print(f"Best RMSE: {metrics_df['RMSE'].idxmin()} | Best DirAcc: {metrics_df['Directional Acc %'].idxmax()}")

# ── AAPL Price Comparison Cell ─────────────────────────────────────────────
# AAPL current price on NASDAQ as of Jun 18 2026 = ~$295.95
# The model predicts LOG-RETURNS, not prices.
# Ensemble log-return * 100 ≈ predicted % change for small moves.
# To convert to price: predicted_price = current_close * exp(log_return_pred)
#
# HOW TO INTERPRET:
#   If today's close = $295.95 and Ensemble predicts log_return = +0.003
#   then tomorrow's predicted price = 295.95 * exp(0.003) ≈ $296.84 (+$0.89, +0.30%)
#
# The RMSE in log-return space (~0.010–0.015) translates to:
#   Dollar RMSE ≈ 295.95 * 0.012 ≈ $3.55 per day
#   That is a NORMAL range for a day-ahead stock predictor.
#
# DIRECTIONAL ACCURACY matters more than RMSE:
#   >55% = genuinely predictive
#   50–55% = marginal
#   <50% = worse than random

fig,axes=plt.subplots(1,3,figsize=(18,5))
fig.suptitle(f"{CONFIG['ticker']} Model Comparison",fontsize=14,fontweight='bold')
palette=['#2196F3','#9C27B0','#FF9800','#4CAF50','#F44336','#00BCD4','#FF5722']
for ax,metric in zip(axes,['RMSE','R2','Directional Acc %']):  # FIX M1: R² replaces MAPE (meaningless for log-return targets)
    vals=metrics_df[metric].sort_values(ascending=(metric!='Directional Acc %'))
    bars=ax.bar(vals.index,vals.values,color=palette[:len(vals)])
    ax.set_title(metric,fontsize=11)
    ax.set_xticklabels(vals.index,rotation=20,ha='right',fontsize=8)
    for bar,val in zip(bars,vals.values):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.001*max(vals),
                f'{val:.4f}',ha='center',va='bottom',fontsize=7)
    ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.savefig('model_comparison.png',dpi=150,bbox_inches='tight'); plt.show()



MODEL PERFORMANCE SUMMARY
                       RMSE           RMSE_CI_95       MAE       R2  Directional Acc %  P-Value Sig (p<0.05)
Model                                                                                                       
Linear Regression  0.018354  [0.016438,0.020432]  0.013236  -0.0969              50.84   0.0000        ✅ Yes
Ridge Regression   0.018333  [0.016364,0.020524]  0.013203  -0.0944              51.02   0.0000        ✅ Yes
XGBoost            0.017625  [0.015234,0.020371]  0.011883  -0.0115              46.37   0.0048        ✅ Yes
LightGBM           0.017704  [0.015603,0.020063]  0.012227  -0.0206              49.53   0.0005        ✅ Yes
Random Forest      0.018669  [0.016434,0.021291]  0.013302  -0.1349              45.62   0.0000        ✅ Yes
Ensemble           0.017795  [0.015531,0.020320]  0.012299  -0.0311              46.55   0.0000        ✅ Yes
LSTM               0.110633  [0.104792,0.116185]  0.088950 -39.3616              50.66   0.0000      

## 🧠 Step 12b — Neural Meta-Learner (Signal Stacker)

Instead of a fixed weighted average, we train a **small dense neural network** that learns *how to combine* the signals from XGBoost, LightGBM, Random Forest, and LSTM into one optimal prediction. This is called **stacked generalisation** (Wolpert 1992). The base models are already frozen — only the 3-layer MLP is trained here, using out-of-fold predictions as inputs so the stacker never sees the test set during training.

Is it overkill? Only mildly — the stacker has just ~130 parameters, trains in seconds, and replaces the brittle fixed-weight average with a learnable combination. On noisy daily returns the gain is modest but real: the stacker can discover that, e.g., RF is reliable in low-volatility regimes but XGB is better in high-volatility ones, without you having to hard-code that logic.


In [28]:

# ─────────────────────────────────────────────────────────────────────────────
# Step 12b — Neural Meta-Learner: trains a small MLP to combine model signals
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print("\n🧠 STEP 12b: Neural Meta-Learner (Signal Stacker)")
print("=" * 55)

# ── 1. Build stacking feature matrix ─────────────────────────────────────────
# Base-model test-set predictions (already computed above)
_stack_xgb = y_pred_xgb           # shape (n_test,)
_stack_lgb = y_pred_lgb
_stack_rf  = y_pred_rf

# LSTM aligns to lstm_test_dates — find the overlap with the tree-model test window
_lstm_index = pd.Index(lstm_test_dates)
_tree_index  = pd.Index(test_dates)
_common      = _tree_index.intersection(_lstm_index)

if len(_common) >= 20:
    _ti_loc = [_tree_index.get_loc(d) for d in _common]
    _li_loc = [_lstm_index.get_loc(d) for d in _common]
    _stack_mat = np.column_stack([
        _stack_xgb[_ti_loc],
        _stack_lgb[_ti_loc],
        _stack_rf[_ti_loc],
        y_pred_lstm[_li_loc],
    ])
    _stack_y_true = y_test[_ti_loc]
    _stack_dates  = _common
    _n_base_models = 4
    print(f"  Using {len(_common)} dates with all 4 signals (tree + LSTM aligned)")
else:
    # Fallback: tree models only
    _stack_mat = np.column_stack([_stack_xgb, _stack_lgb, _stack_rf])
    _stack_y_true = y_test
    _stack_dates  = test_dates
    _n_base_models = 3
    print(f"  LSTM overlap too small ({len(_common)} dates) — using 3 tree signals only")

# ── 2. Train / val split (no shuffle — time-series) ─────────────────────────
_n = len(_stack_mat)
_split = int(_n * 0.75)
_Xs_tr, _Xs_va = _stack_mat[:_split], _stack_mat[_split:]
_ys_tr, _ys_va = _stack_y_true[:_split], _stack_y_true[_split:]

# ── 3. Build neural stacker ───────────────────────────────────────────────────
meta_model = Sequential([
    Dense(32, activation='relu', input_shape=(_n_base_models,)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.1),
    Dense(1),
], name='meta_learner')

meta_model.compile(optimizer=Adam(5e-4), loss='huber', metrics=['mae'])
print(f"  Meta-learner params: {meta_model.count_params():,}")

_es_meta = EarlyStopping(patience=10, restore_best_weights=True, verbose=0)
meta_model.fit(
    _Xs_tr, _ys_tr,
    validation_data=(_Xs_va, _ys_va),
    epochs=150, batch_size=16, shuffle=False,
    callbacks=[_es_meta], verbose=0,
)

# ── 4. Evaluate stacker on held-out portion ───────────────────────────────────
y_pred_meta = meta_model.predict(_stack_mat, verbose=0).flatten()
m_meta = evaluate_model(_stack_y_true, y_pred_meta, 'Neural Stacker')
metrics_list.append(m_meta)
predictions['Neural Stacker'] = (y_pred_meta, _stack_dates)
print(f"  Neural Stacker: RMSE={m_meta['RMSE']} DirAcc={m_meta['Directional Acc %']}%  IC={m_meta['IC (Spearman)']}")
meta_model.save('meta_learner_model.keras')
print("  ✅ meta_learner_model.keras saved")



🧠 STEP 12b: Neural Meta-Learner (Signal Stacker)
  Using 531 dates with all 4 signals (tree + LSTM aligned)
  Meta-learner params: 833
  Neural Stacker: RMSE=0.019091 DirAcc=52.92%  IC=-0.0397
  ✅ meta_learner_model.keras saved


## 📊 Step 12c — Accuracy Tracking Dashboard

A persistent, at-a-glance summary of every model's accuracy metrics, including the new neural stacker. Colour-coded so you immediately see which model wins on each criterion.


In [29]:

# ─────────────────────────────────────────────────────────────────────────────
# Step 12c — Accuracy Tracking Dashboard (colour-coded summary table + chart)
# ─────────────────────────────────────────────────────────────────────────────
print("\n📊 STEP 12c: Accuracy Tracking Dashboard")
print("=" * 55)

# Rebuild metrics_df including the neural stacker
metrics_df = pd.DataFrame(metrics_list).set_index('Model')

_display_cols = ['RMSE','MAE','R2','Directional Acc %','DirAcc CI 95%','IC (Spearman)','DM-stat','Sig (p<0.05)']
_show = metrics_df[[c for c in _display_cols if c in metrics_df.columns]].copy()

# ── Console table ─────────────────────────────────────────────────────────────
print("\nMODEL ACCURACY DASHBOARD")
print("=" * 90)
print(f"{'Model':<22} {'RMSE':>10} {'MAE':>10} {'R²':>8} {'DirAcc%':>9} {'DirAcc CI':>16} {'IC':>8} {'DM-stat':>9} {'Sig':>6}")
print("-" * 90)
for mdl, row in _show.iterrows():
    best_rmse_flag = "★" if mdl == metrics_df['RMSE'].idxmin() else " "
    best_da_flag   = "★" if mdl == metrics_df['Directional Acc %'].idxmax() else " "
    print(f"{mdl:<22}"
          f" {row.get('RMSE', float('nan')):>10.6f}{best_rmse_flag}"
          f" {row.get('MAE', float('nan')):>10.6f}"
          f" {row.get('R2', float('nan')):>8.4f}"
          f" {row.get('Directional Acc %', float('nan')):>9.2f}{best_da_flag}"
          f" {str(row.get('DirAcc CI 95%','—')):>16}"
          f" {row.get('IC (Spearman)', float('nan')):>8.4f}"
          f" {row.get('DM-stat', float('nan')):>9.3f}"
          f" {row.get('Sig (p<0.05)','—'):>6}")
print("-" * 90)
print(f"  ★ = best in column | DirAcc: random = 50%, good = 55%+, strong = 60%+ | IC > 0.05 is meaningful")

# ── Visual dashboard (4-panel bar chart) ─────────────────────────────────────
_models = list(metrics_df.index)
_n = len(_models)
_xpos = np.arange(_n)

# Colour scheme: highlight Neural Stacker and Ensemble
_COLORS = []
for m in _models:
    if 'Neural' in m:    _COLORS.append('#CE93D8')   # purple — stacker
    elif 'Ensemble' in m:_COLORS.append('#FFFFFF')   # white  — weighted ensemble
    elif 'XGB' in m:     _COLORS.append('#FF9800')
    elif 'LGB' in m or 'Light' in m: _COLORS.append('#64B5F6')
    elif 'Forest' in m:  _COLORS.append('#81C784')
    elif 'LSTM' in m:    _COLORS.append('#F06292')
    else:                _COLORS.append('#90A4AE')

fig_dash, axes_d = plt.subplots(2, 2, figsize=(16, 10))
fig_dash.suptitle('Model Accuracy Dashboard — All Metrics', fontsize=14, fontweight='bold')
fig_dash.patch.set_facecolor('#0a0a14')
for ax in axes_d.flat:
    ax.set_facecolor('#0d0d1a')
    ax.tick_params(colors='#cccccc')
    ax.xaxis.label.set_color('#cccccc')
    ax.yaxis.label.set_color('#cccccc')
    ax.title.set_color('#ffffff')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a1a2e')

def _bar_panel(ax, vals, title, ylabel, highlight_min=True, ref_line=None, ref_label=None):
    bars = ax.bar(_xpos, vals, color=_COLORS, alpha=0.85, edgecolor='#0a0a14', linewidth=0.5)
    best_idx = np.nanargmin(vals) if highlight_min else np.nanargmax(vals)
    bars[best_idx].set_edgecolor('#FFD700'); bars[best_idx].set_linewidth(2.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=7.5, color='#cccccc')
    if ref_line is not None:
        ax.axhline(ref_line, color='#FFB300', linewidth=1.2, linestyle='--', label=ref_label)
        ax.legend(fontsize=8, labelcolor='#cccccc', facecolor='#0d0d1a', edgecolor='#1a1a2e')
    ax.set_xticks(_xpos); ax.set_xticklabels(_models, rotation=30, ha='right', fontsize=8, color='#cccccc')
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel); ax.grid(axis='y', alpha=0.25, color='#2a2a3a')

_rmse_vals = [metrics_df.loc[m,'RMSE'] if m in metrics_df.index else np.nan for m in _models]
_da_vals   = [metrics_df.loc[m,'Directional Acc %'] if m in metrics_df.index else np.nan for m in _models]
_r2_vals   = [metrics_df.loc[m,'R2'] if m in metrics_df.index else np.nan for m in _models]
_ic_vals   = [metrics_df.loc[m,'IC (Spearman)'] if m in metrics_df.index else np.nan for m in _models]

_bar_panel(axes_d[0,0], _rmse_vals, 'RMSE (lower = better)', 'RMSE', highlight_min=True)
_bar_panel(axes_d[0,1], _da_vals,   'Directional Accuracy % (higher = better)', 'DA %',
           highlight_min=False, ref_line=50, ref_label='Random (50%)')
_bar_panel(axes_d[1,0], _r2_vals,   'R² Score (higher = better)', 'R²', highlight_min=False)
_bar_panel(axes_d[1,1], _ic_vals,   'Information Coefficient — Spearman IC (higher = better)',
           'IC', highlight_min=False, ref_line=0.05, ref_label='IC threshold (0.05)')

plt.tight_layout()
plt.savefig('accuracy_dashboard.png', dpi=130, bbox_inches='tight', facecolor='#0a0a14')
plt.show()
print("\n  Gold border = best model per metric | Purple = Neural Stacker | White = Weighted Ensemble")
print(f"  Best RMSE     : {metrics_df['RMSE'].idxmin()}  ({metrics_df['RMSE'].min():.6f})")
print(f"  Best DirAcc   : {metrics_df['Directional Acc %'].idxmax()}  ({metrics_df['Directional Acc %'].max():.2f}%)")
print(f"  Best IC       : {metrics_df['IC (Spearman)'].idxmax()}  ({metrics_df['IC (Spearman)'].max():.4f})")
print("  ✅ accuracy_dashboard.png saved")



📊 STEP 12c: Accuracy Tracking Dashboard

MODEL ACCURACY DASHBOARD
Model                        RMSE        MAE       R²   DirAcc%        DirAcc CI       IC   DM-stat    Sig
------------------------------------------------------------------------------------------
Linear Regression        0.018354    0.013236  -0.0969     50.84       [46.6,55.0]   0.0523    -5.420  ✅ Yes
Ridge Regression         0.018333    0.013203  -0.0944     51.02       [46.8,55.2]   0.0539    -5.554  ✅ Yes
XGBoost                  0.017625★   0.011883  -0.0115     46.37       [42.2,50.6]  -0.0292    -3.398  ✅ Yes
LightGBM                 0.017704    0.012227  -0.0206     49.53       [45.3,53.8]   0.0645    -3.156  ✅ Yes
Random Forest            0.018669    0.013302  -0.1349     45.62       [41.5,49.9]   0.0963    -6.272  ✅ Yes
Ensemble                 0.017795    0.012299  -0.0311     46.55       [42.4,50.8]   0.0661    -4.129  ✅ Yes
LSTM                     0.110633    0.088950 -39.3616     50.66       [46.4,54.9

## 🎯 Step 13b — Precision Stock Picker & Neural-Stacked Prediction

Pick any ticker, choose your training window length, and get the **highest-precision prediction** by routing through the neural meta-learner instead of the fixed weighted ensemble.

**Why date range matters:** More data = more stable weights but may include stale regimes. Less data = more responsive to recent behaviour but higher variance. The picker lets you tune this directly.


In [30]:

# ─────────────────────────────────────────────────────────────────────────────
# Step 13b — Precision Stock Picker + Neural-Stacked Live Prediction
#
# Usage:  just change PICK_TICKER / PICK_LOOKBACK_DAYS and run this cell.
#         The cell retrains the base models on the chosen window and runs
#         the neural stacker on top — giving the highest-precision single
#         number the pipeline can produce.
# ─────────────────────────────────────────────────────────────────────────────

# ── USER CONTROLS ─────────────────────────────────────────────────────────────
PICK_TICKER        = CONFIG['ticker']    # e.g. 'RELIANCE.NS', 'TCS.NS', 'MSFT'
PICK_LOOKBACK_DAYS = 600                 # training window size in calendar days
                                         # 365 = 1yr, 600 = ~2yr, 1200 = ~5yr
                                         # More days → more stable but slower
PICK_PRECISION_MODE = True               # True = use neural stacker (slower, better)
                                         # False = use weighted ensemble (faster)

# ─────────────────────────────────────────────────────────────────────────────
print(f"\n🎯 STEP 13b: Precision Stock Picker")
print("=" * 55)
print(f"  Ticker       : {PICK_TICKER}")
print(f"  Lookback     : {PICK_LOOKBACK_DAYS} days")
print(f"  Precision    : {'Neural Stacker' if PICK_PRECISION_MODE else 'Weighted Ensemble'}")

_pick_end   = datetime.today().strftime('%Y-%m-%d')
_pick_start = (datetime.today() - timedelta(days=PICK_LOOKBACK_DAYS)).strftime('%Y-%m-%d')

# ── Download + engineer for chosen ticker/window ─────────────────────────────
print(f"\n  📡 Fetching {PICK_TICKER} ({_pick_start} → {_pick_end})...")
_df_pick = download_stock_data(PICK_TICKER, _pick_start, _pick_end)
_df_pick = add_market_context(_df_pick, PICK_TICKER, start=_pick_start, end=_pick_end)
_df_pick = engineer_features(_df_pick)
_df_pick = _df_pick.dropna(subset=['Target'])

# Dynamically determine feature_cols for this specific _df_pick
_picker_feature_cols = [c for c in _df_pick.columns if c not in EXCLUDE]

# Verify we have enough data
_min_rows = 120
if len(_df_pick) < _min_rows:
    print(f"  ⚠️  Only {len(_df_pick)} rows after feature engineering "
          f"(need ≥{_min_rows}). Increase PICK_LOOKBACK_DAYS.")
    raise ValueError("Insufficient data for chosen lookback window.")

_Xp, _Xp_te, _yp, _yp_te, _sp, _tp = split_data(_df_pick, _picker_feature_cols, CONFIG['test_size'])
print(f"  ✅ {len(_df_pick)} rows | Train={len(_Xp)} Test={len(_Xp_te)}")

# ── Quick-fit base models on this ticker/window ───────────────────────────────
print("  🔵 Fitting base models on this window...")
_xgb_p = xgb.XGBRegressor(**best_xgb_params, verbosity=0)
_lgb_p = lgb.LGBMRegressor(**best_lgb_params)
_rf_p  = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=5,
                                min_samples_leaf=2, max_features='sqrt',
                                random_state=CONFIG['random_state'], n_jobs=-1)
_xgb_p.fit(_Xp, _yp);  _xgb_p_pred = _xgb_p.predict(_Xp_te)
_lgb_p.fit(_Xp, _yp);  _lgb_p_pred = _lgb_p.predict(_Xp_te)
_rf_p.fit(_Xp, _yp);   _rf_p_pred  = _rf_p.predict(_Xp_te)

# ── Get live price ────────────────────────────────────────────────────────────
_live_info  = get_live_price(PICK_TICKER)
_live_price = _live_info['price'] if (_live_info.get('price', 0) > 0 and
                                       not _live_info.get('error')) else None

# ── Build latest feature vector ───────────────────────────────────────────────
_feat_vec = np.zeros((1, len(_picker_feature_cols))) # Use _picker_feature_cols
for _i, _col in enumerate(_picker_feature_cols):
    if _col in _df_pick.columns:
        _feat_vec[0, _i] = float(_df_pick[_col].iloc[-1])
_feat_scaled = _sp.transform(_feat_vec)

# Inject live price signals if available
if _live_price:
    _last_close  = float(_df_pick['Close'].iloc[-1])
    _today_open  = float(_df_pick['Open'].iloc[-1])
    _intra_ret   = (_live_price - _today_open) / _today_open if _today_open > 0 else 0
    if 'Intraday_Return' in _df_pick.columns:
        _feat_vec[0, _picker_feature_cols.index('Intraday_Return')] = _intra_ret # Use _picker_feature_cols
    _feat_scaled = _sp.transform(_feat_vec)
    _cp = _live_price
    print(f"  ✅ Live price injected: {_live_info.get('currency','')} {_live_price:.2f} "
          f"(intraday {_intra_ret*100:+.2f}%)")
else:
    _cp = float(_df_pick['Close'].iloc[-1])
    print(f"  ⚠️  No live price — using last close: {_cp:.2f}")

# ── Base model log-return forecasts ──────────────────────────────────────────
_lr_xgb = float(_xgb_p.predict(_feat_scaled)[0])
_lr_lgb = float(_lgb_p.predict(_feat_scaled)[0])
_lr_rf  = float(_rf_p.predict(_feat_scaled)[0])

# ── Neural stacker or weighted ensemble ───────────────────────────────────────
if PICK_PRECISION_MODE and 'meta_model' in dir() and meta_model is not None:
    # Stack the three signals through the neural meta-learner
    _stk_input = np.array([[_lr_xgb, _lr_lgb, _lr_rf]])
    # If meta was trained with 4 inputs (LSTM included), pad with ensemble as LSTM proxy
    if meta_model.input_shape[-1] == 4:
        _ew = CONFIG['ensemble_weights']
        _lr_ens_tmp = _lr_xgb*_ew['xgb'] + _lr_lgb*_ew['lgb'] + _lr_rf*_ew['rf']
        _stk_input = np.array([[_lr_xgb, _lr_lgb, _lr_rf, _lr_ens_tmp]])
    _lr_final = float(meta_model.predict(_stk_input, verbose=0)[0][0])
    _method   = 'Neural Stacker'
else:
    _ew = CONFIG['ensemble_weights']
    _lr_final = _lr_xgb*_ew['xgb'] + _lr_lgb*_ew['lgb'] + _lr_rf*_ew['rf']
    _method   = 'Weighted Ensemble'

# ── Convert log-return → price ─────────────────────────────────────────────
_px_xgb   = round(_cp * np.exp(_lr_xgb),   2)
_px_lgb   = round(_cp * np.exp(_lr_lgb),   2)
_px_rf    = round(_cp * np.exp(_lr_rf),    2)
_px_final = round(_cp * np.exp(_lr_final), 2)
_pct      = _lr_final * 100
_currency = _live_info.get('currency', 'INR' if '.NS' in PICK_TICKER else 'USD')
_sym      = '₹' if _currency == 'INR' else '$'
_signal   = '📈 BULLISH' if _lr_final > 0 else '📉 BEARISH'
_pred_date = (datetime.today() + pd.offsets.BDay(1)).strftime('%Y-%m-%d')

# ── Individual model test-set metrics ────────────────────────────────────────
_m_xgb = evaluate_model(_yp_te, _xgb_p_pred, f'{PICK_TICKER} XGB')
_m_lgb = evaluate_model(_yp_te, _lgb_p_pred, f'{PICK_TICKER} LGB')
_m_rf  = evaluate_model(_yp_te, _rf_p_pred,  f'{PICK_TICKER} RF')

print(f"""
{'='*60}
  PRECISION PREDICTION — {PICK_TICKER}
  Method: {_method} | Lookback: {PICK_LOOKBACK_DAYS}d
{'='*60}
  Live Price   : {_sym}{_cp:>12,.2f}
  XGBoost      : {_sym}{_px_xgb:>12,.2f}  (RMSE={_m_xgb['RMSE']:.6f} DA={_m_xgb['Directional Acc %']:.1f}%)
  LightGBM     : {_sym}{_px_lgb:>12,.2f}  (RMSE={_m_lgb['RMSE']:.6f} DA={_m_lgb['Directional Acc %']:.1f}%)
  Random Forest: {_sym}{_px_rf:>12,.2f}  (RMSE={_m_rf['RMSE']:.6f}  DA={_m_rf['Directional Acc %']:.1f}%)
{'─'*60}
  {_method:13s}: {_sym}{_px_final:>12,.2f}  ({_pct:+.3f}%)
  Signal       : {_signal}
  For date     : {_pred_date}
{'='*60}
""")


🎯 STEP 13b: Precision Stock Picker
  Ticker       : AAPL
  Lookback     : 600 days
  Precision    : Neural Stacker

  📡 Fetching AAPL (2024-11-02 → 2026-06-25)...
📡 Downloading AAPL...
  ✅ 409 days | Close: 171.51 – 315.20
  ✅ SPY: 408 rows
  ✅ QQQ: 408 rows
  ✅ XLK: 408 rows
  ✅ ^VIX: 408 rows
✅ Features: 67 cols, 209 rows
Train: 167 | 2025-08-22 → 2026-04-22
Test : 42  | 2026-04-23 → 2026-06-23
  ✅ 209 rows | Train=167 Test=42
  🔵 Fitting base models on this window...
  📡 AAPL: source=Finnhub price=USD 293.08
  ✅ Live price injected: USD 293.08 (intraday -1.50%)

  PRECISION PREDICTION — AAPL
  Method: Neural Stacker | Lookback: 600d
  Live Price   : $      293.08
  XGBoost      : $      293.40  (RMSE=0.013771 DA=52.4%)
  LightGBM     : $      292.85  (RMSE=0.016526 DA=47.6%)
  Random Forest: $      292.97  (RMSE=0.016132  DA=45.2%)
────────────────────────────────────────────────────────────
  Neural Stacker: $      293.92  (+0.287%)
  Signal       : 📈 BULLISH
  For date     : 2026

## 🔮 Step 13 — Live Inference (predict_tomorrow)

In [31]:
# FIX M3: 5-minute TTL dict cache — avoids re-downloading 600 days + ETFs on every Gradio click
_predict_cache: dict = {}
_CACHE_TTL_SECONDS = 300

def predict_tomorrow(ticker):
    """
    v3.1 (Robust) — injects live intraday data into a dedicated feature
    to avoid corrupting the daily time-series lags.

    NOTE on what 'tomorrow's price' means here: the model is trained to predict
    close[t+1] / close[t]. At inference time, when a live quote is available, the
    predicted return is applied to the LIVE price (`cp = live_price`), not to
    yesterday's official close. That's a reasonable choice (it keeps the forecast
    anchored to the most current price you'd actually be trading from) but it means
    the output is 'tomorrow's close, anchored to right-now's price' rather than
    strictly 'tomorrow's close, anchored to today's close' — those can diverge on a
    day with a large intraday move, since the model never saw 'live mid-session
    price' as a training reference point.
    """
    cache_key = ticker.upper()
    _now_ts = time.time()
    if cache_key in _predict_cache:
        _cached_result, _cached_ts = _predict_cache[cache_key]
        if _now_ts - _cached_ts < _CACHE_TTL_SECONDS:
            print(f"  ⚡ Cache hit for {ticker} ({int(_CACHE_TTL_SECONDS - (_now_ts - _cached_ts))}s remaining)")
            return _cached_result

    end   = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today() - timedelta(days=600)).strftime('%Y-%m-%d')

    # ── Download historical daily data (for feature warmup) ───────────
    df_hist = download_stock_data(ticker, start, end)
    df_hist = add_market_context(df_hist, ticker, start=start, end=end)  # FIX L2: pass rolling window, not full CONFIG dates

    df_hist = engineer_features(df_hist)

    # ── Get live price from agent ──────────────────────────────────────
    live_info  = get_live_price(ticker)
    live_price = live_info['price'] if (live_info.get('price', 0) > 0
                                        and not live_info.get('error')) else None

    # ── Inject live data into last row if available ────────────────────
    if live_price and live_price > 0:
        last_close = float(df_hist['Close'].iloc[-1])
        today_open = float(df_hist['Open'].iloc[-1])

        # Calculate live intraday return from today's open
        live_intra_ret = (live_price - today_open) / today_open if today_open > 0 else 0

        # ONLY update the dedicated Intraday feature and SMA distances.
        # DO NOT touch Return_Lag_1 or Daily_Return to keep the daily time-series pure.
        if 'Intraday_Return' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Intraday_Return')] = live_intra_ret

        if 'SMA_20' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Price_vs_SMA20')] = (
                (live_price - float(df_hist['SMA_20'].iloc[-1])) / float(df_hist['SMA_20'].iloc[-1])
            )

        if 'SMA_50' in df_hist.columns:
            df_hist.iloc[-1, df_hist.columns.get_loc('Price_vs_SMA50')] = (
                (live_price - float(df_hist['SMA_50'].iloc[-1])) / float(df_hist['SMA_50'].iloc[-1])
            )

        print(f"  ✅ Live data injected cleanly: live=${live_price:.2f} (intraday momentum: {live_intra_ret*100:+.2f}%)")
        cp = live_price   # use live as base for price conversion
    else:
        cp = float(df_hist['Close'].iloc[-1])
        print(f"  ⚠️  No live data — using last close: {cp:.2f}")

    # ── Build feature vector ───────────────────────────────────────────
    lat_full = np.zeros((1, len(feature_cols)))
    missing  = []
    for i, col in enumerate(feature_cols):
        if col in df_hist.columns:
            lat_full[0, i] = float(df_hist[col].iloc[-1])
        else:
            missing.append(col)
    if missing:
        print(f"  ⚠️  {len(missing)} features zero-filled: {missing[:3]}...")

    ls     = scaler.transform(lat_full)
    lr_xgb = float(xgb_model.predict(ls)[0])
    lr_lgb = float(lgb_model.predict(ls)[0])
    lr_rf  = float(rf_model.predict(ls)[0])
    _ew = CONFIG['ensemble_weights']
    lr_ens = lr_xgb*_ew['xgb'] + lr_lgb*_ew['lgb'] + lr_rf*_ew['rf']  # FIX M4: from CONFIG (now data-driven, Step 5b)

    px_xgb = round(cp * np.exp(lr_xgb), 2)
    px_lgb = round(cp * np.exp(lr_lgb), 2)
    px_rf  = round(cp * np.exp(lr_rf),  2)
    px_ens = round(cp * np.exp(lr_ens), 2)
    pct    = lr_ens * 100

    # ── Sentiment overlay (transparent, capped, SEPARATE from the trained ML
    # ensemble number — see CONFIG['sentiment_tilt_cap'] comment) ─────────────
    # News_Sentiment/Analyst_Score are deliberately excluded from the trained
    # models (see Step 2 / Step 3 comments — their HISTORICAL values are
    # fabricated zeros, so training on them would be meaningless). But at live
    # inference time we DO have a real, current sentiment read for whichever
    # ticker was actually requested, so we surface it as a small, capped,
    # clearly-separate adjustment rather than silently folding it into
    # 'ensemble'. Tetlock (2007), "Giving Content to Investor Sentiment: The
    # Role of Media in the Stock Market", Journal of Finance 62(3), is the
    # standard reference for short-horizon sentiment having SOME predictive
    # content — but it is a small effect, hence the tight cap.
    # NOTE: this fetches sentiment for the ACTUAL ticker passed in (not the
    # notebook-level `sentiment_report`, which only ever reflects
    # CONFIG['ticker']) — using the global AAPL-only report for an arbitrary
    # Gradio ticker would repeat the same cross-ticker bias documented for
    # the trained models themselves (see bias-analysis notes).
    try:
        _yahoo_live = scrape_yahoo_news(ticker)
        _alt_live   = (fetch_alphavantage_news(ticker) or fetch_finnhub_news(ticker)
                       or fetch_google_news_rss(ticker))
        _ns_live  = score_sentiment(_yahoo_live)
        _als_live = score_sentiment(_alt_live)
        _nz_live  = [s for s in [_ns_live, _als_live] if s != 0.0]
        live_combined_sentiment = float(np.mean(_nz_live)) if _nz_live else 0.0
    except Exception as _e:
        live_combined_sentiment = 0.0

    sentiment_tilt = float(np.clip(live_combined_sentiment, -1, 1)) * CONFIG['sentiment_tilt_cap']
    lr_ens_sent    = lr_ens + sentiment_tilt
    px_ens_sent    = round(cp * np.exp(lr_ens_sent), 2)

    currency = live_info.get('currency', 'INR' if '.NS' in ticker else 'USD')
    sym      = '₹' if currency == 'INR' else '$'

    # ── Recent history (for the Gradio per-model prediction chart) ─────────
    _hist_n = min(60, len(df_hist))
    history = {
        'dates': [str(d.date()) for d in df_hist.index[-_hist_n:]],
        'close': [round(float(c), 2) for c in df_hist['Close'].values[-_hist_n:]],
    }

    result = {
        'ticker'          : ticker,
        'current_price'   : round(cp, 2),
        'live_price'      : round(live_price or cp, 2),
        'last_data_date'  : str(df_hist.index[-1].date()),
        'currency'        : currency,
        'symbol'          : sym,
        'xgboost'         : px_xgb,
        'lightgbm'        : px_lgb,
        'random_forest'   : px_rf,
        'ensemble'        : px_ens,
        'ensemble_sentiment_adjusted': px_ens_sent,     # separate, capped, transparent
        'sentiment_score'             : round(live_combined_sentiment, 4),
        'sentiment_tilt_pct'          : round(sentiment_tilt * 100, 4),
        'pct_change'      : round(pct, 3),
        'signal'          : '📈 BULLISH' if lr_ens > 0 else '📉 BEARISH',
        'prediction_date' : (datetime.today() + pd.offsets.BDay(1)).strftime('%Y-%m-%d'),  # FIX C3: BDay skips weekends/holidays
        'live_injected'   : live_price is not None,
        'market_state'    : live_info.get('market_state', 'UNKNOWN'),
        'change_today'    : live_info.get('change', 0),
        'change_pct_today': live_info.get('change_pct', 0),
        'history'         : history,
    }

    print(f"\n{'='*55}")
    print(f"  {ticker} | Live injected: {result['live_injected']}")
    print(f"  Live Price   : {sym}{result['live_price']:.2f} "
          f"({result['change_pct_today']:+.2f}% today)")
    print(f"  Ensemble     : {sym}{px_ens:.2f} ({pct:+.2f}%)")
    print(f"  + Sentiment  : {sym}{px_ens_sent:.2f} (tilt {result['sentiment_tilt_pct']:+.3f}%, "
          f"score={result['sentiment_score']:+.3f})")
    print(f"  Signal       : {result['signal']}")
    print(f"{'='*55}")
    _predict_cache[cache_key] = (result, time.time())  # FIX M3: cache for TTL
    return result

tomorrow = predict_tomorrow(CONFIG['ticker'])

📡 Downloading AAPL...
  ✅ 409 days | Close: 171.51 – 315.20
  ✅ SPY: 408 rows
  ✅ QQQ: 408 rows
  ✅ XLK: 408 rows
  ✅ ^VIX: 408 rows
✅ Features: 67 cols, 209 rows
  📡 AAPL: source=Finnhub price=USD 293.08
  ✅ Live data injected cleanly: live=$293.08 (intraday momentum: -1.50%)
  ⚠️  4 features zero-filled: ['Fed_Funds_Rate', 'Yield_Curve', 'VIX_FRED']...

  AAPL | Live injected: True
  Live Price   : $293.08 (-0.41% today)
  Ensemble     : $292.10 (-0.34%)
  + Sentiment  : $292.14 (tilt +0.015%, score=+0.097)
  Signal       : 📉 BEARISH


In [32]:
!pip install langgraph langchain-groq -q

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

# FIX C2: guard against missing Groq key
if KEYS.get('groq'):
    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=KEYS['groq'])
else:
    llm = None
    print("\u26a0\ufe0f  GROQ_API_KEY not set \u2014 LLM agents skipped (add key to Colab Secrets)")

# ── Reducer: keeps the most recent non-empty value ─────────────────────────
def keep_latest(a, b):
    """
    If b is a real value (non-empty string, non-empty dict, non-None),
    use b. Otherwise keep a. This lets initial values flow in correctly
    AND lets nodes update their own output keys without conflict.
    """
    if b is None:
        return a
    if isinstance(b, str) and b.strip() == "":
        return a
    if isinstance(b, dict) and len(b) == 0:
        return a
    if isinstance(b, list) and len(b) == 0:
        return a
    return b

# ── State definition ────────────────────────────────────────────────────────
class TradingState(TypedDict):
    ticker:           Annotated[str,   keep_latest]
    ml_signal:        Annotated[dict,  keep_latest]
    shap_features:    Annotated[list,  keep_latest]
    news_sentiment:   Annotated[float, keep_latest]
    technical_report: Annotated[str,   keep_latest]
    sentiment_report: Annotated[str,   keep_latest]
    bull_case:        Annotated[str,   keep_latest]
    bear_case:        Annotated[str,   keep_latest]
    final_decision:   Annotated[str,   keep_latest]

# ── Agent nodes — each returns ONLY the key it writes ──────────────────────
# This is the real fix: returning {**state, 'key': value} causes every
# key to be "updated" simultaneously → InvalidUpdateError on parallel branches.
# Returning only {'key': value} tells LangGraph exactly what changed.

def technical_analyst(state: TradingState) -> dict:
    ml = state['ml_signal']
    sym        = ml.get('symbol', '$')
    ensemble   = ml.get('ensemble', 0.0)
    pct_change = ml.get('pct_change', 0.0)
    signal     = ml.get('signal', 'UNKNOWN')
    live_price = ml.get('live_price', 0.0)
    top_feats  = state.get('shap_features', [])[:5]

    prompt = f"""You are a quantitative technical analyst.
The ML pipeline produced these results for {state['ticker']}:
- Ensemble price forecast : {sym}{ensemble:.2f} ({pct_change:+.2f}%)
- Signal                  : {signal}
- Live price              : {sym}{live_price:.2f}
- Top 5 SHAP features     : {top_feats}

Write exactly 3 bullet points of technical analysis.
Be specific about what the numbers imply."""

    if llm is None:
        return {'technical_report': '[Groq key not set]'}
    report = llm.invoke(prompt).content
    return {'technical_report': report}          # ← only write what changed


def sentiment_analyst(state: TradingState) -> dict:
    prompt = f"""You are a market sentiment analyst for {state['ticker']}.
News sentiment score: {state['news_sentiment']:.4f}  (scale: -1 very negative → +1 very positive)

Write exactly 2 bullet points interpreting this sentiment score
and what it implies for short-term price action."""

    if llm is None:
        return {'sentiment_report': '[Groq key not set]'}
    report = llm.invoke(prompt).content
    return {'sentiment_report': report}          # ← only write what changed


def bull_researcher(state: TradingState) -> dict:
    prompt = f"""You are a BULLISH equity researcher for {state['ticker']}.
You have access to these reports:

TECHNICAL ANALYSIS:
{state['technical_report']}

SENTIMENT ANALYSIS:
{state['sentiment_report']}

Make the strongest possible bull case in exactly 3 points.
Focus on upside catalysts and why the stock should rise tomorrow."""

    if llm is None:
        return {'bull_case': '[Groq key not set]'}
    case = llm.invoke(prompt).content
    return {'bull_case': case}                   # ← only write what changed


def bear_researcher(state: TradingState) -> dict:
    prompt = f"""You are a BEARISH equity researcher for {state['ticker']}.
You have access to these reports:

TECHNICAL ANALYSIS:
{state['technical_report']}

SENTIMENT ANALYSIS:
{state['sentiment_report']}

Make the strongest possible bear case in exactly 3 points.
Focus on downside risks and why the stock might fall or underperform tomorrow."""

    if llm is None:
        return {'bear_case': '[Groq key not set]'}
    case = llm.invoke(prompt).content
    return {'bear_case': case}                   # ← only write what changed


def portfolio_manager(state: TradingState) -> dict:
    ml   = state['ml_signal']
    sym  = ml.get('symbol', '$')
    sig  = ml.get('signal', 'UNKNOWN')
    pct  = ml.get('pct_change', 0.0)
    ens  = ml.get('ensemble', 0.0)

    prompt = f"""You are the Portfolio Manager making the final trading decision for {state['ticker']}.

ML ENSEMBLE FORECAST : {sym}{ens:.2f} ({pct:+.2f}%) — {sig}

BULL RESEARCHER SAYS:
{state['bull_case']}

BEAR RESEARCHER SAYS:
{state['bear_case']}

Synthesize all of the above and give:
1. Decision    : BUY / SELL / HOLD
2. Position    : FULL / HALF / QUARTER
3. Rationale   : One clear paragraph explaining the decision,
                 acknowledging the strongest point from each side."""

    if llm is None:
        return {'final_decision': '[Groq key not set]'}
    decision = llm.invoke(prompt).content
    return {'final_decision': decision}          # ← only write what changed


# ── Build graph ─────────────────────────────────────────────────────────────
graph = StateGraph(TradingState)

graph.add_node("technical", technical_analyst)
graph.add_node("sentiment", sentiment_analyst)
graph.add_node("bull",      bull_researcher)
graph.add_node("bear",      bear_researcher)
graph.add_node("pm",        portfolio_manager)

graph.set_entry_point("technical")
graph.add_edge("technical", "sentiment")
graph.add_edge("technical", "bear")    # FIX C1: bear_researcher reads technical_report; needs edge from technical
graph.add_edge("sentiment", "bull")
graph.add_edge("sentiment", "bear")
graph.add_edge("bull",      "pm")
graph.add_edge("bear",      "pm")
graph.add_edge("pm",        END)

trading_graph = graph.compile()
print("✅ Trading graph compiled successfully")

✅ Trading graph compiled successfully


## 💰 Step 14 — Backtest (Fixed · Risk-Managed)

In [33]:
def backtest_strategy(prices_true, opens_true, y_pred, dates, model_name,
                      cap=10000.0, tc=0.001, sl=0.05, tp=0.10):
    """
    Fixed backtest (v2 — execution timing actually matches the claim now):
    - prices_true : actual CLOSING prices (used to mark equity / check SL-TP)
    - opens_true  : actual OPENING prices (used as the trade EXECUTION price)
    - y_pred      : predicted log-returns (signal: >0 = LONG, <=0 = FLAT)
    - No lookahead: signal at day i is formed from information available at
      close of day i (preds[i]); the order can only realistically fill at the
      *next* tradable price, which is the open of day i+1 — opens_true[i+1].
      The previous version computed the signal this way but then executed at
      prices[i] (day i's own CLOSE), i.e. the same price the signal was
      derived from — not tradable in practice. That mismatch is fixed here.
    - Stop-loss 5%, take-profit 10%, transaction cost 0.1%
    - Caveat: still daily-bar granularity, so SL/TP are checked against the
      day's close, not real intrabar prices — a true intrabar SL/TP would
      need higher-frequency data.
    """
    capital, pos, entry = cap, 0.0, 0.0
    eq, bh, trades = [], [], []
    prices = np.array(prices_true, dtype=float)
    opens  = np.array(opens_true,  dtype=float)
    preds  = np.array(y_pred,      dtype=float)

    # Validate inputs — catch the log-return / price confusion early
    if prices.mean() < 1.0:
        raise ValueError(
            f'prices_true looks like log-returns (mean={prices.mean():.4f}). '
            f'Pass actual closing prices, not y_test.')

    bhs = cap / opens[1]   # buy-and-hold benchmark also enters at a tradable open

    for i in range(len(prices) - 1):
        # Decision uses info available at close of day i (preds[i]); earliest
        # realistic fill is the OPEN of day i+1 — not the close the signal came from.
        exec_price = opens[i + 1]
        mark_price = prices[i + 1]   # day i+1 close — used to mark equity / SL-TP
        sig = 1 if preds[i] > 0 else -1

        # Check stop-loss / take-profit before considering a new signal
        if pos > 0 and entry > 0:
            r = (mark_price - entry) / entry
            if r <= -sl or r >= tp:
                capital = pos * mark_price * (1 - tc)
                trades.append({'type': 'STP/TP', 'ret': r})
                pos, entry = 0.0, 0.0

        # Open / close position — at next day's OPEN, not today's close
        if sig == 1 and pos == 0:
            pos    = (capital * (1 - tc)) / exec_price
            entry  = exec_price
            capital = 0.0
            trades.append({'type': 'BUY', 'ret': None})
        elif sig == -1 and pos > 0:
            r       = (exec_price - entry) / entry
            capital = pos * exec_price * (1 - tc)
            trades.append({'type': 'SELL', 'ret': r})
            pos, entry = 0.0, 0.0

        eq.append(capital + (pos * mark_price if pos > 0 else 0))
        bh.append(bhs * mark_price)

    # Close any open position at end
    if pos > 0:
        eq[-1] = pos * prices[-1] * (1 - tc)

    equity = np.array(eq); bha = np.array(bh)
    tr_ret = (equity[-1] - cap) / cap * 100
    bh_ret = (bha[-1]    - cap) / cap * 100

    dr     = np.diff(equity) / np.maximum(equity[:-1], 1e-10)
    sharpe = (dr.mean() / (dr.std() + 1e-10)) * np.sqrt(252)
    mdd    = ((equity - np.maximum.accumulate(equity)) /
               np.maximum.accumulate(equity)).min() * 100
    trets  = [t['ret'] for t in trades if t.get('ret') is not None]
    wr     = sum(r > 0 for r in trets) / max(len(trets), 1) * 100

    print(f'\n💰 {model_name}: Return={tr_ret:+.1f}% | B&H={bh_ret:+.1f}% | '
          f'Sharpe={sharpe:.2f} | MDD={mdd:.1f}% | '
          f'WinRate={wr:.1f}% | Trades={len(trets)}')

    fig = go.Figure()
    plot_dates = dates[1:len(equity)+1]  # equity[i] now marks day i+1 (post execution-timing fix)
    fig.add_trace(go.Scatter(x=list(plot_dates), y=equity,
                              name='Strategy', line=dict(color='#4CAF50', width=2)))
    fig.add_trace(go.Scatter(x=list(plot_dates),    y=bha,
                              name='Buy & Hold',
                              line=dict(color='#2196F3', width=2, dash='dash')))
    fig.add_hline(y=cap, line_dash='dot', line_color='white', opacity=0.5)
    fig.update_layout(
        template='plotly_dark',
        title=f'{model_name} vs Buy & Hold | Sharpe={sharpe:.2f}',
        height=420)
    fname = f"backtest_{model_name.replace(' ', '_')}.html"
    fig.write_html(fname); fig.show()

    return pd.DataFrame({
        'Date'    : plot_dates,
        'Strategy': equity,
        'BuyHold' : bha
    })


# ── Call with actual PRICES (and OPENS, for realistic execution) not y_test ──
test_prices = df_features['Close'].values[-len(y_test):]
test_opens  = df_features['Open'].values[-len(y_test):]
test_dates  = df_features.index[-len(y_test):]

backtest_df = backtest_strategy(
    prices_true = test_prices,        # ← actual closes, NOT y_test
    opens_true  = test_opens,         # ← actual opens, used for execution price
    y_pred      = y_pred_ensemble,    # ← log-return predictions (correct)
    dates       = test_dates,
    model_name  = 'Ensemble',
    cap         = CONFIG['initial_capital'],
    tc          = CONFIG['transaction_cost']
)


💰 Ensemble: Return=+73.2% | B&H=+72.3% | Sharpe=1.11 | MDD=-9.3% | WinRate=73.3% | Trades=15


## 📐 Step 14b — Portfolio Risk Metrics

Rockafellar & Uryasev (2000), *"Optimization of Conditional Value-at-Risk"*, Journal of Risk 2(3) — CVaR (Expected Shortfall) is the mean loss in the worst α% of scenarios, and unlike VaR it's a *coherent* risk measure (sub-additive — it never says a diversified portfolio is riskier than its parts). Sortino & Price (1994), *"Performance Measurement in a Downside Risk Framework"*, Journal of Investing 3(3) — penalises only downside deviation, since investors don't mind upside volatility. Calmar ratio (return / |max drawdown|) and Omega ratio (Keating & Shadwick (2002), *"A Universal Performance Measure"*, Journal of Performance Measurement 6(3)) round out the standard risk-reporting toolkit any quant strategy writeup is expected to show.

In [34]:
# ── Step 14b — Portfolio Risk Metrics ───────────────────────────────────────
print("\n📊 STEP 14b: Portfolio Risk Metrics")
print("=" * 55)

from scipy import stats as _stats

strat_returns = backtest_df['Strategy'].pct_change().dropna().values
bh_returns    = backtest_df['BuyHold'].pct_change().dropna().values

def portfolio_risk_report(daily_returns, label='Strategy'):
    r = np.array(daily_returns, dtype=float)
    r = r[~np.isnan(r)]
    if len(r) < 10:
        print(f"  {label}: insufficient data"); return {}

    ann_ret  = (1 + r.mean())**252 - 1
    ann_vol  = r.std() * np.sqrt(252)
    sharpe   = ann_ret / (ann_vol + 1e-10)

    downside = r[r < 0]
    down_dev = np.sqrt((downside**2).mean()) * np.sqrt(252) if len(downside) > 0 else 1e-10
    sortino  = ann_ret / (down_dev + 1e-10)

    var_95  = np.percentile(r, 5)
    var_99  = np.percentile(r, 1)
    cvar_95 = r[r <= var_95].mean()
    cvar_99 = r[r <= var_99].mean()

    cum_ret   = np.cumprod(1 + r)
    peak      = np.maximum.accumulate(cum_ret)
    dd_series = (cum_ret - peak) / peak
    max_dd    = dd_series.min()
    calmar    = ann_ret / (abs(max_dd) + 1e-10)

    gains  = r[r > 0].sum()
    losses = abs(r[r < 0].sum())
    omega  = gains / (losses + 1e-10)
    profit_factor = gains / (losses + 1e-10)

    skew = float(_stats.skew(r))
    kurt = float(_stats.kurtosis(r))   # excess kurtosis; > 0 = fatter-than-normal tails
    tail_ratio = abs(np.percentile(r, 95)) / (abs(np.percentile(r, 5)) + 1e-10)

    metrics = {
        'label': label, 'Ann Return %': round(ann_ret*100,2), 'Ann Vol %': round(ann_vol*100,2),
        'Sharpe': round(sharpe,3), 'Sortino': round(sortino,3), 'Calmar': round(calmar,3),
        'Omega': round(omega,3), 'Profit Factor': round(profit_factor,3),
        'VaR 95% (1d)': round(var_95*100,3), 'CVaR 95% (1d)': round(cvar_95*100,3),
        'VaR 99% (1d)': round(var_99*100,3), 'CVaR 99% (1d)': round(cvar_99*100,3),
        'Max DD %': round(max_dd*100,2), 'Skewness': round(skew,3),
        'Exc Kurtosis': round(kurt,3), 'Tail Ratio': round(tail_ratio,3),
    }
    print(f"\n  ── {label} ──")
    for k, v in metrics.items():
        if k == 'label': continue
        flag = ''
        if k == 'Sharpe':        flag = ' ✅' if v > 1   else (' ⚠️' if v > 0.5 else ' ❌')
        if k == 'Sortino':       flag = ' ✅' if v > 1.5 else (' ⚠️' if v > 0.8 else ' ❌')
        if k == 'Calmar':        flag = ' ✅' if v > 1   else (' ⚠️' if v > 0.3 else ' ❌')
        if k == 'CVaR 95% (1d)': flag = ' ✅' if v > -3  else ' ❌'
        if k == 'Exc Kurtosis':  flag = ' ⚠️ fat tails' if v > 3 else ''
        print(f"    {k:<20}: {v}{flag}")
    return metrics

risk_strat = portfolio_risk_report(strat_returns, 'Ensemble Strategy')
risk_bh    = portfolio_risk_report(bh_returns,    'Buy & Hold')

print("\n" + "="*55)
compare_keys = ['Ann Return %','Ann Vol %','Sharpe','Sortino','Calmar',
                 'VaR 95% (1d)','CVaR 95% (1d)','Max DD %','Tail Ratio']
print(f"  {'Metric':<22} {'Strategy':>12} {'Buy&Hold':>12}")
print("  " + "-"*48)
for k in compare_keys:
    print(f"  {k:<22} {str(risk_strat.get(k,'n/a')):>12} {str(risk_bh.get(k,'n/a')):>12}")

risk_summary = {'strategy': risk_strat, 'buy_hold': risk_bh}
with open('risk_metrics.json', 'w') as f:
    json.dump(risk_summary, f, indent=2)
print("\n✅ Risk metrics complete — stored in risk_summary, saved to risk_metrics.json")


📊 STEP 14b: Portfolio Risk Metrics

  ── Ensemble Strategy ──
    Ann Return %        : 33.17
    Ann Vol %           : 25.84
    Sharpe              : 1.284 ✅
    Sortino             : 1.017 ⚠️
    Calmar              : 3.573 ✅
    Omega               : 2.884
    Profit Factor       : 2.884
    VaR 95% (1d)        : 0.0
    CVaR 95% (1d)       : -0.064 ✅
    VaR 99% (1d)        : -2.064
    CVaR 99% (1d)       : -3.314
    Max DD %            : -9.28
    Skewness            : 15.945
    Exc Kurtosis        : 314.766 ⚠️ fat tails
    Tail Ratio          : 26630481.613

  ── Buy & Hold ──
    Ann Return %        : 34.14
    Ann Vol %           : 28.02
    Sharpe              : 1.219 ✅
    Sortino             : 1.23 ⚠️
    Calmar              : 1.023 ✅
    Omega               : 1.221
    Profit Factor       : 1.221
    VaR 95% (1d)        : -2.672
    CVaR 95% (1d)       : -3.918 ❌
    VaR 99% (1d)        : -4.62
    CVaR 99% (1d)       : -6.03
    Max DD %            : -33.36
    Skewn

## 📊 Step 14c — Quant Finance Charts

Three panels that complete the risk picture:
- **Drawdown curve**: when and how long the worst losses lasted (not just the max)
- **OOF RMSE → ensemble weights**: makes the data-driven weighting rationale visible
- **VaR/CVaR fan chart**: return distribution with tail risk zones

In [35]:
print("\n📊 STEP 14c: Quant Finance Charts")
print("=" * 55)

from scipy import stats as _scipy_stats

fig_q, axes_q = plt.subplots(1, 3, figsize=(18, 6))
fig_q.suptitle(f'{CONFIG["ticker"]} — Quant Risk Charts', fontsize=13, fontweight='bold')

strat_ret = backtest_df['Strategy'].pct_change().dropna().values
bh_ret    = backtest_df['BuyHold'].pct_change().dropna().values

# ── Panel 1: Drawdown curves (Strategy vs Buy & Hold) ────────────────────
ax1 = axes_q[0]
def _drawdown(returns):
    cum = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cum)
    return (cum - peak) / (peak + 1e-10) * 100

dd_strat = _drawdown(strat_ret)
dd_bh    = _drawdown(bh_ret)
dd_dates = backtest_df.index[1:]  # pct_change drops first row

ax1.fill_between(dd_dates[:len(dd_strat)], dd_strat, 0, alpha=0.4, color='#4CAF50', label='Strategy DD')
ax1.fill_between(dd_dates[:len(dd_bh)],    dd_bh,    0, alpha=0.3, color='#2196F3', label='Buy & Hold DD')
ax1.plot(dd_dates[:len(dd_strat)], dd_strat, color='#4CAF50', linewidth=0.8)
ax1.plot(dd_dates[:len(dd_bh)],    dd_bh,    color='#2196F3', linewidth=0.8, linestyle='--')
ax1.axhline(0, color='white', linewidth=0.5)
ax1.set_xlabel('Date'); ax1.set_ylabel('Drawdown %')
ax1.set_title('Drawdown Over Time', fontsize=10); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax1.text(0.02, 0.03, f'Max DD: Strat={dd_strat.min():.1f}%  B&H={dd_bh.min():.1f}%',
         transform=ax1.transAxes, fontsize=8, color='gray')

# ── Panel 2: OOF RMSE → ensemble weights bar ─────────────────────────────
ax2 = axes_q[1]
ew = CONFIG['ensemble_weights']
models_w = list(ew.keys()); weights_w = list(ew.values())
colors_w  = ['#FF9800', '#64B5F6', '#81C784']
bars2 = ax2.bar(models_w, weights_w, color=colors_w, alpha=0.85, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars2, weights_w):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.axhline(1/3, color='white', linewidth=0.8, linestyle=':', label='Equal weight (1/3)')
ax2.set_ylabel('Ensemble Weight'); ax2.set_ylim(0, max(weights_w) * 1.25)
ax2.set_title('Data-Driven Ensemble Weights\n(inverse OOF-RMSE)', fontsize=10)
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)
# Hardcoded baseline for comparison
ax2.bar([m + '_base' for m in ['xgb','lgb','rf']],
        [0.4, 0.4, 0.2], color=colors_w, alpha=0.25,
        edgecolor='white', linewidth=0.5, linestyle='--')
ax2.set_xticklabels(['XGB\n(data)', 'LGB\n(data)', 'RF\n(data)'], fontsize=9)
ax2.text(0.5, 0.98, 'Faded bars = old hardcoded weights (0.4/0.4/0.2)',
         transform=ax2.transAxes, ha='center', va='top', fontsize=7, color='gray')

# ── Panel 3: Return distribution with VaR/CVaR fan ───────────────────────
ax3 = axes_q[2]
ax3.hist(strat_ret, bins=60, color='#2196F3', alpha=0.6, density=True, label='Daily returns')
var_95 = np.percentile(strat_ret, 5);  cvar_95 = strat_ret[strat_ret <= var_95].mean()
var_99 = np.percentile(strat_ret, 1);  cvar_99 = strat_ret[strat_ret <= var_99].mean()
ax3.axvline(var_95,  color='#FFB300', linewidth=1.5, linestyle='--', label=f'VaR 95%={var_95*100:.2f}%')
ax3.axvline(cvar_95, color='#FF9800', linewidth=1.5, linestyle=':',  label=f'CVaR 95%={cvar_95*100:.2f}%')
ax3.axvline(var_99,  color='#F44336', linewidth=1.5, linestyle='--', label=f'VaR 99%={var_99*100:.2f}%')
ax3.axvline(cvar_99, color='#B71C1C', linewidth=1.5, linestyle=':',  label=f'CVaR 99%={cvar_99*100:.2f}%')
# Shade tail zones
ax3.axvspan(strat_ret.min(), var_99,  alpha=0.15, color='#F44336')
ax3.axvspan(var_99,          var_95,  alpha=0.10, color='#FF9800')
ax3.set_xlabel('Daily Return'); ax3.set_ylabel('Density')
ax3.set_title('Return Distribution with VaR/CVaR Zones', fontsize=10)
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quant_risk_charts.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"  Max drawdown  : Strategy={dd_strat.min():.1f}%  Buy&Hold={dd_bh.min():.1f}%")
print(f"  VaR 95%/99%   : {var_95*100:.3f}% / {var_99*100:.3f}%")
print(f"  CVaR 95%/99%  : {cvar_95*100:.3f}% / {cvar_99*100:.3f}%")
print(f"  Weights       : {ew}")
print("  ✅ Quant risk charts saved → quant_risk_charts.png")



📊 STEP 14c: Quant Finance Charts
  Max drawdown  : Strategy=-9.3%  Buy&Hold=-33.4%
  VaR 95%/99%   : 0.000% / -2.064%
  CVaR 95%/99%  : -0.064% / -3.314%
  Weights       : {'xgb': 0.346, 'lgb': 0.3307, 'rf': 0.3233}
  ✅ Quant risk charts saved → quant_risk_charts.png


## 📝 Step 15 — Groq LLM Report

In [36]:
def run_groq_agent(ticker,metrics_df,sentiment_report,tomorrow_pred,wf_summary):
    combined = sentiment_report.get('combined_sentiment', sentiment_report.get('alt_news_sentiment',0))
    s={
        'ticker':ticker,'best_model':metrics_df['RMSE'].idxmin(),
        'best_rmse':float(metrics_df['RMSE'].min()),
        'best_da':float(metrics_df['Directional Acc %'].max()),
        'sig':metrics_df[metrics_df['Sig (p<0.05)']=='✅ Yes'].index.tolist(),
        'wf_mu':float(wf_summary['RMSE'].mean()),
        'wf_sd':float(wf_summary['RMSE'].std()),
        'tm':tomorrow_pred,'sent':sentiment_report,
    }
    prompt=(
        f"Senior quant analyst reviewing ML stock prediction pipeline.\n"
        f"Stock:{s['ticker']} | Best:{s['best_model']}(RMSE:{s['best_rmse']:.6f}) | DirAcc:{s['best_da']:.1f}%\n"
        f"Significant models (p<0.05 vs naive): {s['sig']}\n"
        f"Walk-Forward RMSE: {s['wf_mu']:.6f} ± {s['wf_sd']:.6f}\n"
        f"Sentiment: news={s['sent']['news_sentiment']:.4f} combined={combined:.4f} analyst={s['sent']['analyst_score']:.4f}\n"
        f"Tomorrow: live={s['tm']['symbol']}{s['tm']['live_price']:.2f} "        f"ensemble={s['tm']['symbol']}{s['tm']['ensemble']:.2f} ({s['tm']['pct_change']:+.2f}%) "        f"signal={s['tm']['signal']}\n\n"
        f"Write 5-point analysis: model performance, sentiment signals, "        f"prediction confidence, key risks, one-line recommendation. <300 words, bullets."
    )
    if not KEYS.get('groq'):
        return (f"LOCAL REPORT — {ticker}\n"
                f"Best: {s['best_model']} RMSE={s['best_rmse']:.6f} DirAcc={s['best_da']:.1f}%\n"
                f"WF RMSE: {s['wf_mu']:.6f} ± {s['wf_sd']:.6f}\n"
                f"Signal: {s['tm']['signal']} | Ensemble: {s['tm']['symbol']}{s['tm']['ensemble']:.2f} ({s['tm']['pct_change']:+.2f}%)\n"
                f"Live: {s['tm']['symbol']}{s['tm']['live_price']:.2f} | Market: {s['tm']['market_state']}\n"
                f"Note: Add GROQ_API_KEY to Colab Secrets for Llama 3 narrative report.")
    try:
        from groq import Groq
        resp=Groq(api_key=KEYS['groq']).chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[{'role':'user','content':prompt}],
            max_tokens=500,temperature=0.3)
        return resp.choices[0].message.content
    except Exception as e: return f'Groq error: {e}'

llm_report=run_groq_agent(CONFIG['ticker'],metrics_df,sentiment_report,tomorrow,wf_results)
print('\n'+'='*60+'\nLLM REPORT\n'+'='*60+'\n'+llm_report)
with open('analysis_report.txt','w') as f: f.write(llm_report)



LLM REPORT
Here's a 5-point analysis of the ML stock prediction pipeline for AAPL:

* **Model Performance**: The XGBoost model outperforms other models with an RMSE of 0.017625, and the walk-forward RMSE is 0.017914 ± 0.006855, indicating a relatively stable prediction performance.
* **Sentiment Signals**: Sentiment analysis shows a mildly positive news sentiment (0.1221) and a slightly lower combined sentiment (0.0973), with no analyst sentiment (0.0000), which may indicate a neutral to slightly bullish outlook.
* **Prediction Confidence**: The ensemble prediction for tomorrow is $292.10, which is 0.34% lower than the live price, with a bearish signal (📉), indicating a moderate level of confidence in the prediction.
* **Key Risks**: The key risks include model overfitting, sentiment signal noise, and potential market volatility, which may impact the accuracy of the predictions.
* **Recommendation**: Based on the analysis, consider a **cautious short-term bearish position** on AAPL, a

## 🖥️ Step 16 — Gradio Dashboard v7 (Quant Charts + Fixed Agents)

**What's new vs v6:**
- 📊 **Quant Charts tab** — dropdown menu to pick any of 10 charts (model comparison, accuracy dashboard, walk-forward drift, HMM/GARCH regimes, residuals, multi-step forecast, quant risk/VaR, SHAP, LSTM curves, prediction). Each chart renders full-width — no Plotly toolbar overlap.
- ✅ **Agent button moved inside the Agent tab** — no more shared-header ambiguity; button is right next to its outputs and triggers only the agent accordions
- 🔧 **Prediction chart legend** moved below chart (`b=90`) so it never overlaps the toolbar zone
- 🎛️ **Chart auto-loads on dropdown change** — no need to click Load Chart, but the button is there too
- All charts re-use the ticker from the shared input box at the top — change ticker and reload to get fresh charts for any stock


In [37]:
import gradio as gr
print(gr.__version__)

# ... rest of your Gradio cell below
# ── Safety defaults: optional globals that may not exist if steps were skipped ───
_g = globals()
if 'multistep_df'    not in _g: multistep_df    = None
if 'shap_values'     not in _g: shap_values     = None
if 'history'         not in _g: history         = None
if 'meta_model'      not in _g: meta_model      = None
if 'backtest_df'     not in _g: backtest_df     = None
if 'wf_results'      not in _g: wf_results      = None
if 'metrics_df'      not in _g: metrics_df      = None
if 'df_features'     not in _g: df_features     = None
if 'y_test'          not in _g: y_test          = None
if 'y_pred_ensemble' not in _g: y_pred_ensemble = None
if 'test_dates'      not in _g: test_dates      = None
if 'feature_cols'    not in _g: feature_cols    = []
del _g

# ─────────────────────────────────────────────────────────────────────────────
# Step 16 — Gradio Dashboard v7 (QuantPlus Neural Stack)
#
# What's new vs v6:
#   1. TAB 4 "Quant Charts" — dropdown menu to pick one chart at a time,
#      each rendered full-width so nothing overlaps the Plotly toolbar
#   2. Charts re-generated on demand for whichever ticker is in the input box
#   3. Agent tab: Run Agents button is now wired INSIDE the tab, not in the
#      shared header, so clicks are unambiguous and the loading state is local
#   4. Prediction chart legend pushed below chart (b=80) so it never overlaps
#      the Plotly mode-bar area
#   5. All gr.Plot() wrappers use show_label=False + container=False so the
#      grey Gradio label box never appears on top of the chart chrome
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr
from datetime import datetime
import plotly.graph_objects as go
import numpy as np, pandas as pd

_current_ticker = {"value": CONFIG["ticker"]}

# ─────────────────────────────────────────────────────────────────────────────
# CSS
# ─────────────────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
/* ── global ── */
.gradio-container { background: #0a0a0f !important; }
.gr-block, .gr-box { background: transparent !important; }

/* ── section labels ── */
.section-label {
    font-family: 'JetBrains Mono', 'Fira Code', monospace;
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    color: #8a8aaa;
    margin: 14px 0 6px;
    padding-left: 2px;
}

/* ── price card ── */
.price-card {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 12px;
    padding: 18px 20px;
    font-family: 'JetBrains Mono', monospace;
}
.price-card.up   { border-color: #1a3a1a; }
.price-card.down { border-color: #3a1a1a; }

/* ── prediction table ── */
.pred-card {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 12px;
    padding: 18px 20px;
    font-family: 'JetBrains Mono', monospace;
}

/* ── watchlist ── */
.watch-card {
    background: #0a0a14;
    border: 1px solid #1a1a2e;
    border-radius: 12px;
    padding: 16px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13px;
}

/* ── agent sections ── */
.agent-section {
    background: #0d0d1a;
    border: 1px solid #1e1e3a;
    border-radius: 10px;
    padding: 14px 18px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13px;
    line-height: 1.8;
    color: #cccccc;
    margin-bottom: 8px;
}
.agent-section-title {
    font-size: 10px;
    font-weight: 700;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-bottom: 10px;
}

/* ── chart dropdown ── */
.chart-selector label {
    color: #8a8aaa !important;
    font-family: 'JetBrains Mono', monospace;
    font-size: 11px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.chart-selector select {
    background: #0d0d1a !important;
    border: 1px solid #1e1e3a !important;
    color: #cccccc !important;
    border-radius: 8px;
    font-family: 'JetBrains Mono', monospace;
}

/* ── fix Gradio plot wrapper chrome ── */
.gradio-plot { background: transparent !important; border: none !important; }
.gradio-plot > .wrap { padding: 0 !important; }

/* ── agent run button inside tab ── */
.agent-run-btn {
    background: #1e1e3a !important;
    border: 1px solid #3a3a6a !important;
    color: #cccccc !important;
    border-radius: 8px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 12px;
}
.agent-run-btn:hover {
    background: #2a2a4a !important;
    border-color: #6464aa !important;
}
"""

# ─────────────────────────────────────────────────────────────────────────────
# LIVE TICKER (unchanged logic, fixed CSS class)
# ─────────────────────────────────────────────────────────────────────────────
def refresh_live_ticker(ticker_input: str) -> str:
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    _current_ticker["value"] = ticker
    info = get_live_price(ticker)

    if info.get("error"):
        return f"""<div class='price-card' style='border-color:#3a1a1a;'>
  <div style='color:#888;font-size:11px;font-family:monospace;'>ERROR</div>
  <div style='color:#F44336;font-size:14px;margin-top:6px;'>{ticker}: {info["error"]}</div>
</div>"""

    sym       = "&#8377;" if info["currency"] == "INR" else "$"
    is_up     = info["change"] >= 0
    clr       = "#4CAF50" if is_up else "#F44336"
    arrow     = "&#9650;" if is_up else "&#9660;"
    sign      = "+" if is_up else ""
    card_cls  = "up" if is_up else "down"
    ms        = info.get("market_state", "")
    dot_color = "#4CAF50" if ms == "REGULAR" else ("#FFB300" if ("PRE" in ms or "POST" in ms) else "#666")
    vol       = f"{info['volume']:,}" if info.get('volume', 0) > 0 else "N/A"
    src       = info.get("source", "")
    ts        = info.get("timestamp", "")

    currency_warning = ""
    if ticker.endswith(".NS") and info["currency"] == "INR" and info["price"] < 10 and info.get('source') != 'NSE_API':
        currency_warning = "<div style='color:#FFB300;font-size:11px;margin-top:6px;'>&#9888; Possible USD data leak</div>"

    return f"""<div class='price-card {card_cls}'>
  <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px;'>
    <span style='width:8px;height:8px;border-radius:50%;background:{dot_color};display:inline-block;'></span>
    <span style='color:#aaaacc;font-size:11px;letter-spacing:0.08em;'>{ms.upper()}</span>
    <span style='color:#555;font-size:11px;'>&#124;</span>
    <span style='color:#aaa;font-size:11px;'>{ts}</span>
    <span style='color:#555;font-size:11px;'>&#124;</span>
    <span style='color:#aaa;font-size:11px;'>{src}</span>
  </div>
  <div style='font-size:15px;font-weight:600;color:#ffffff;letter-spacing:0.1em;margin-bottom:4px;'>{ticker}</div>
  <div style='font-size:38px;font-weight:700;color:white;line-height:1.1;margin-bottom:6px;'>
    {sym}{info['price']:,.2f}
  </div>
  <div style='font-size:16px;color:{clr};margin-bottom:10px;'>
    {arrow} {sign}{info['change']:.2f} &nbsp; <span style='font-size:14px;'>({sign}{info['change_pct']:.2f}%)</span>
  </div>
  <div style='display:flex;gap:20px;font-size:11px;color:#888;border-top:1px solid #1a1a2e;padding-top:8px;'>
    <span>Prev close: {sym}{info['prev_close']:,.2f}</span>
    <span>Vol: {vol}</span>
    <span>CCY: {info['currency']}</span>
  </div>
  <div style='font-size:10px;color:#555;margin-top:6px;'>Prices delayed ~15 min (yfinance free tier)</div>
  {currency_warning}
</div>"""


def _timer_tick() -> str:
    return refresh_live_ticker(_current_ticker["value"])


def _sync_state(t: str):
    _current_ticker["value"] = t.strip().upper() if t else CONFIG["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# SHARED PLOTLY LAYOUT HELPER
# ─────────────────────────────────────────────────────────────────────────────
_DARK = dict(
    template="plotly_dark",
    paper_bgcolor="#0d0d1a",
    plot_bgcolor="#0d0d1a",
    font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
    xaxis=dict(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"), title_font=dict(color="#cccccc")),
    yaxis=dict(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"), title_font=dict(color="#cccccc")),
    margin=dict(l=60, r=20, t=60, b=90),   # b=90 gives legend room below chart
    legend=dict(
        orientation="h", y=-0.22, x=0,
        font=dict(size=11, color="#cccccc"),
        bgcolor="rgba(0,0,0,0)",
    ),
    title=dict(font=dict(color="#ffffff", size=13), x=0.0, xanchor="left"),
)

def _apply_dark(fig, title="", height=500):
    layout = dict(**_DARK, height=height)
    layout["title"] = dict(text=title, font=dict(color="#ffffff", size=13), x=0.0, xanchor="left")
    fig.update_layout(**layout)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# PREDICTION CHART (full-width, legend below, no toolbar overlap)
# ─────────────────────────────────────────────────────────────────────────────
def build_prediction_chart(r: dict):
    hist       = r.get("history", {})
    hist_dates = pd.to_datetime(hist.get("dates", []))
    hist_close = hist.get("close", [])

    fig = go.Figure()
    if len(hist_dates) > 0:
        fig.add_trace(go.Scatter(
            x=hist_dates, y=hist_close, name="Actual Close",
            mode="lines", line=dict(color="#9aa0b0", width=2),
            hovertemplate="%{x|%b %d}<br>Close: %{y:.2f}<extra></extra>"))
        last_date, last_close = hist_dates[-1], hist_close[-1]
    else:
        last_date, last_close = pd.Timestamp.today(), r.get("current_price", 0)

    next_date = pd.to_datetime(r["prediction_date"])
    for name, key, color in [
        ("XGBoost",       "xgboost",       "#FF9800"),
        ("LightGBM",      "lightgbm",      "#64B5F6"),
        ("Random Forest", "random_forest", "#81C784"),
        ("Ensemble",      "ensemble",      "#FFFFFF"),
    ]:
        px = r.get(key)
        if px is None:
            continue
        fig.add_trace(go.Scatter(
            x=[last_date, next_date], y=[last_close, px],
            mode="lines+markers", name=name,
            line=dict(color=color, width=1.8, dash="dot"),
            marker=dict(size=[0, 10], color=color),
            hovertemplate=f"{name}<br>%{{x|%b %d}}: %{{y:.2f}}<extra></extra>"))

    if "ensemble_sentiment_adjusted" in r:
        fig.add_trace(go.Scatter(
            x=[last_date, next_date],
            y=[last_close, r["ensemble_sentiment_adjusted"]],
            mode="lines+markers", name="Ensemble + Sentiment",
            line=dict(color="#CE93D8", width=1.8, dash="dot"),
            marker=dict(size=[0, 10], color="#CE93D8"),
            hovertemplate="Ensemble+Sentiment<br>%{x|%b %d}: %{y:.2f}<extra></extra>"))

    return _apply_dark(fig,
        title=f"{r.get('ticker','')} — Per-Model Forecast ({r.get('prediction_date','')})",
        height=460)


# ─────────────────────────────────────────────────────────────────────────────
# PER-TICKER CHART DATA CACHE
# Runs Steps 4→6→9 for any ticker on demand so every chart reflects the
# stock the user actually typed, not just the notebook startup ticker.
# Heavy globals (wf_results, multistep_df, backtest_df, history/LSTM) are
# left as-is — those four chart types still show the startup ticker.
# ─────────────────────────────────────────────────────────────────────────────
_CHART_CACHE: dict = {}   # { "TICKER": { "metrics_df": ..., "y_test": ..., ... } }
def _build_ticker_chart_data(ticker: str) -> dict:
    """
    Downloads 600d of history for `ticker`, engineers features, scores the
    existing trained models on the new ticker's test split, and computes SHAP.
    Results are cached in _CHART_CACHE so repeated chart switches are instant.

    FIX: The original used split_data(df_t, feature_cols, ...) which called
    df_t[feature_cols] — this KeyErrors because df_t is missing FRED macro
    columns (Fed_Funds_Rate, Yield_Curve, VIX_FRED, CPI) that were only added
    to the main AAPL pipeline. Instead, we now build the feature matrix by
    zero-filling any missing feature_cols column, exactly like predict_tomorrow.
    """
    ticker = ticker.strip().upper()
    if ticker in _CHART_CACHE:
        return _CHART_CACHE[ticker]

    import shap as _shap_mod
    from datetime import datetime, timedelta

    end   = datetime.today().strftime('%Y-%m-%d')
    start = (datetime.today() - timedelta(days=600)).strftime('%Y-%m-%d')

    # ── Download + feature-engineer for this ticker ──────────────────────
    df_t = download_stock_data(ticker, start, end)
    df_t = add_market_context(df_t, ticker, start=start, end=end)
    df_t = engineer_features(df_t)

    # ── Build feature matrix aligned to training feature_cols ─────────────
    # CANNOT use df_t[feature_cols] directly — df_t may be missing FRED
    # macro columns (Fed_Funds_Rate, Yield_Curve, VIX_FRED, CPI) and any
    # other columns added only during the main pipeline's full setup.
    # Solution: build a zero matrix and fill column-by-column, same as
    # predict_tomorrow() does for live inference.
    n      = len(df_t)
    X_full = np.zeros((n, len(feature_cols)))
    missing_cols = []
    for i, col in enumerate(feature_cols):
        if col in df_t.columns:
            X_full[:, i] = df_t[col].values
        else:
            missing_cols.append(col)

    if missing_cols:
        print(f"  ⚠️  {ticker}: {len(missing_cols)} features zero-filled "
              f"(not in 600d download): {missing_cols[:5]}")

    # ── Train/test split ──────────────────────────────────────────────────
    y_full  = df_t['Target'].values
    sp      = int(n * (1 - CONFIG['test_size']))

    # Scale using fresh scaler fit on training portion only
    sc_t    = StandardScaler()
    X_tr_sc = sc_t.fit_transform(X_full[:sp])
    X_te_sc = sc_t.transform(X_full[sp:])

    y_tr_t      = y_full[:sp]
    y_te_t      = y_full[sp:]
    test_dates_t = df_t.index[sp:]

    # Drop NaN targets from test set (shift(-1) leaves last row NaN)
    mask    = ~np.isnan(y_te_t)
    X_te_sc = X_te_sc[mask]
    y_te_t  = y_te_t[mask]
    test_dates_t = test_dates_t[mask]

    if len(y_te_t) < 5:
        raise ValueError(
            f"Only {len(y_te_t)} valid test rows for {ticker} after NaN drop. "
            f"Try a ticker with more history."
        )

    # ── Score existing trained models (NO retraining) ─────────────────────
    yp_xgb_t = xgb_model.predict(X_te_sc)
    yp_lgb_t = lgb_model.predict(X_te_sc)
    yp_rf_t  = rf_model.predict(X_te_sc)
    _ew      = CONFIG['ensemble_weights']
    yp_ens_t = (yp_xgb_t * _ew['xgb'] +
                yp_lgb_t * _ew['lgb'] +
                yp_rf_t  * _ew['rf'])

    # ── Metrics DataFrame ─────────────────────────────────────────────────
    _m_list = []
    for _name, _pred in [
        ('XGBoost',       yp_xgb_t),
        ('LightGBM',      yp_lgb_t),
        ('Random Forest', yp_rf_t),
        ('Ensemble',      yp_ens_t),
    ]:
        _m_list.append(evaluate_model(y_te_t, _pred, _name))
    mdf_t = pd.DataFrame(_m_list).set_index('Model')

    # Add IC (Spearman) column if missing — accuracy dashboard uses it
    if 'IC (Spearman)' not in mdf_t.columns:
        from scipy.stats import spearmanr
        ic_vals = {}
        for _name, _pred in [
            ('XGBoost',       yp_xgb_t),
            ('LightGBM',      yp_lgb_t),
            ('Random Forest', yp_rf_t),
            ('Ensemble',      yp_ens_t),
        ]:
            ic, _ = spearmanr(y_te_t, _pred)
            ic_vals[_name] = round(float(ic), 4) if not np.isnan(ic) else 0.0
        mdf_t['IC (Spearman)'] = pd.Series(ic_vals)

    # ── SHAP values ───────────────────────────────────────────────────────
    # Use a subsample for speed (max 200 rows — full test set can be slow)
    try:
        _exp     = _shap_mod.TreeExplainer(xgb_model)
        _sample  = X_te_sc[:min(200, len(X_te_sc))]
        _shap_t  = _exp.shap_values(_sample)
    except Exception as e:
        print(f"  ⚠️  SHAP failed for {ticker}: {e}")
        _shap_t = None

    result = {
        'df_features'     : df_t,
        'metrics_df'      : mdf_t,
        'y_test'          : y_te_t,
        'y_pred_ensemble' : yp_ens_t,
        'test_dates'      : test_dates_t,
        'shap_values'     : _shap_t,
        'shap_sample_size': min(200, len(X_te_sc)),
    }
    _CHART_CACHE[ticker] = result
    print(f"  Chart data built + cached for {ticker} "
          f"({len(y_te_t)} test rows, {len(missing_cols)} cols zero-filled)")
    return result

# ─────────────────────────────────────────────────────────────────────────────
# QUANT CHART BUILDER — one function per chart type, all return go.Figure
# Each is regenerated live for the chosen ticker using pre-computed globals
# ─────────────────────────────────────────────────────────────────────────────

def _chart_model_comparison(ticker):
    """RMSE / R² / Directional Accuracy — 3-panel subplot (FIX D: each metric on its own Y axis)."""
    from plotly.subplots import make_subplots
    _res = _build_ticker_chart_data(ticker)
    _df  = _res["metrics_df"].copy()
    models = list(_df.index)
    palette = ["#2196F3","#9C27B0","#FF9800","#4CAF50","#F44336","#00BCD4","#CE93D8","#FF5722"]
    colors  = [palette[i % len(palette)] for i in range(len(models))]

    metrics = [
        ("RMSE",             True,  "RMSE (↓ better)",    "#2196F3"),
        ("R2",               False, "R² Score (↑ better)","#4CAF50"),
        ("Directional Acc %",False, "Dir Acc % (↑ better)","#FF9800"),
    ]
    # Filter to only metrics that exist
    metrics = [(col, asc, lbl, clr) for col, asc, lbl, clr in metrics if col in _df.columns]
    ncols = len(metrics)

    fig = make_subplots(rows=1, cols=ncols,
        subplot_titles=[lbl for _, _, lbl, _ in metrics],
        horizontal_spacing=0.10)

    for ci, (col, ascending, label, base_color) in enumerate(metrics, start=1):
        vals = _df[col].fillna(0)
        best = vals.idxmin() if ascending else vals.idxmax()
        bar_colors = ["#FFD700" if m == best else colors[i] for i, m in enumerate(models)]
        fig.add_trace(go.Bar(
            x=models, y=list(vals), marker_color=bar_colors,
            name=col, showlegend=False,
            hovertemplate=f"{col}: %{{y:.4f}}<extra></extra>",
        ), row=1, col=ci)
        # Reference lines
        if col == "Directional Acc %" and "Directional Acc %" in _df.columns:
            fig.add_shape(type="line", x0=-0.5, x1=len(models)-0.5, y0=50, y1=50,
                line=dict(color="#FFB300", width=1.5, dash="dot"), row=1, col=ci)
        if col == "R2":
            fig.add_shape(type="line", x0=-0.5, x1=len(models)-0.5, y0=0, y1=0,
                line=dict(color="#F44336", width=1, dash="dot"), row=1, col=ci)

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=50, r=20, t=80, b=80), height=460,
        title=dict(text=f"{ticker} — Model Comparison (RMSE · R² · Dir Acc)",
                   font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for attr in ["xaxis","xaxis2","xaxis3"]:
        ax = getattr(fig.layout, attr, None)
        if ax: ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for attr in ["yaxis","yaxis2","yaxis3"]:
        ax = getattr(fig.layout, attr, None)
        if ax: ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_accuracy_dashboard(ticker):
    """4-metric accuracy dashboard for all models."""
    _res = _build_ticker_chart_data(ticker)
    from plotly.subplots import make_subplots
    _df  = _res["metrics_df"].copy()
    models = list(_df.index)
    palette = ["#2196F3","#9C27B0","#FF9800","#4CAF50","#F44336","#00BCD4","#CE93D8","#FF5722"]
    colors  = [palette[i % len(palette)] for i in range(len(models))]

    fig = make_subplots(rows=2, cols=2,
        subplot_titles=["RMSE (↓ better)", "Directional Accuracy % (↑ better)",
                        "R² Score (↑ better)", "IC Spearman (↑ better | 0.05 threshold)"],
        vertical_spacing=0.22, horizontal_spacing=0.12)

    def _bar(row, col, metric, ascending, ref=None):
        if metric not in _df.columns:
            return
        vals = _df[metric].fillna(0)
        best = vals.idxmin() if ascending else vals.idxmax()
        bar_colors = ["#FFD700" if m == best else colors[i] for i, m in enumerate(models)]
        fig.add_trace(go.Bar(x=models, y=list(vals), marker_color=bar_colors,
            hovertemplate=f"{metric}: %{{y:.4f}}<extra></extra>", showlegend=False), row=row, col=col)
        if ref is not None:
            fig.add_shape(type="line", x0=-0.5, x1=len(models)-0.5, y0=ref, y1=ref,
                line=dict(color="#FFB300", width=1.2, dash="dot"), row=row, col=col)

    _bar(1, 1, "RMSE",             ascending=True)
    _bar(1, 2, "Directional Acc %", ascending=False, ref=50)
    _bar(2, 1, "R2",                ascending=False)
    _bar(2, 2, "IC (Spearman)",     ascending=False, ref=0.05)

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=50, r=20, t=80, b=60),
        height=600,
        title=dict(text=f"{ticker} — Accuracy Dashboard (gold = best per metric)",
                   font=dict(color="#ffffff", size=13), x=0.0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"
        ann.font.size  = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2, fig.layout.xaxis3, fig.layout.xaxis4]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2, fig.layout.yaxis3, fig.layout.yaxis4]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_walkforward(ticker):
    """Walk-forward RMSE + directional accuracy drift."""
    if 'wf_results' not in globals() or wf_results is None:
        return _empty_fig("Run Step 8 (Walk-Forward Validation) first")
    from plotly.subplots import make_subplots
    wf = wf_results.copy()
    x  = list(range(1, len(wf) + 1))
    fig = make_subplots(rows=2, cols=1,
        subplot_titles=["RMSE per Walk-Forward Window", "Directional Accuracy % per Window"],
        vertical_spacing=0.18, shared_xaxes=True)

    rmse_v = wf["RMSE"].values
    rmse_m = rmse_v.mean(); rmse_s = rmse_v.std()
    fig.add_trace(go.Bar(x=x, y=list(rmse_v), name="Window RMSE",
        marker_color="#2196F3", opacity=0.8,
        hovertemplate="Window %{x}<br>RMSE: %{y:.6f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=[rmse_m]*len(x), name=f"Mean RMSE ({rmse_m:.5f})",
        line=dict(color="#FFB300", dash="dash", width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x+x[::-1],
        y=list(rmse_v+rmse_s)+list((rmse_v-rmse_s)[::-1]),
        fill="toself", fillcolor="rgba(33,150,243,0.12)",
        line=dict(color="rgba(0,0,0,0)"), showlegend=False, hoverinfo="skip"), row=1, col=1)

    if "Directional Acc %" in wf.columns:
        da_v = wf["Directional Acc %"].values
        fig.add_trace(go.Scatter(x=x, y=list(da_v), name="Dir Acc %",
            mode="lines+markers", line=dict(color="#4CAF50", width=2),
            marker=dict(size=6, color="#4CAF50"),
            hovertemplate="Window %{x}<br>Dir Acc: %{y:.1f}%<extra></extra>"), row=2, col=1)
        fig.add_shape(type="line", x0=0.5, x1=len(x)+0.5, y0=50, y1=50,
            line=dict(color="#F44336", dash="dot", width=1.2), row=2, col=1)

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=60, r=20, t=80, b=80), height=580,
        legend=dict(orientation="h", y=-0.15, x=0, font=dict(color="#cccccc"), bgcolor="rgba(0,0,0,0)"),
        title=dict(text=f"{ticker} — Walk-Forward Performance Drift", font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_hmm_garch(ticker):
    """HMM regime overlay + GARCH volatility."""
    if 'df_features' not in globals() or df_features is None:
        return _empty_fig("Run Step 6b (HMM/GARCH) first")
    from plotly.subplots import make_subplots
    df = df_features.copy()
    fig = make_subplots(rows=3, cols=1,
        subplot_titles=["Price with HMM Regime Overlay", "HMM Regime State", "GARCH(1,1) Conditional Volatility"],
        vertical_spacing=0.10, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25])

    # Price
    fig.add_trace(go.Scatter(x=df.index, y=df["Close"].squeeze(),
        name="Close", line=dict(color="#9aa0b0", width=1.5)), row=1, col=1)

    # Regime shading
    if "HMM_Regime" in df.columns:
        regimes = df["HMM_Regime"].values
        bull_color = "rgba(76,175,80,0.12)"; bear_color = "rgba(244,67,54,0.12)"
        start_i = 0
        for i in range(1, len(regimes)):
            if regimes[i] != regimes[i-1] or i == len(regimes)-1:
                clr = bull_color if regimes[start_i] == 0 else bear_color
                fig.add_vrect(x0=df.index[start_i], x1=df.index[min(i, len(df)-1)],
                    fillcolor=clr, line_width=0, row=1, col=1)
                start_i = i
        fig.add_trace(go.Scatter(x=df.index, y=regimes.astype(float),
            name="Regime (0=Bull, 1=Bear)", mode="lines",
            line=dict(color="#FFB300", width=1.5, shape="hv")), row=2, col=1)

    # GARCH vol
    if "GARCH_Vol" in df.columns:
        gv = df["GARCH_Vol"].squeeze()
        fig.add_trace(go.Scatter(x=df.index, y=gv, name="GARCH Vol",
            line=dict(color="#CE93D8", width=1.5)), row=3, col=1)
        fig.add_trace(go.Scatter(
            x=list(df.index)+list(df.index[::-1]),
            y=list(gv)+[0]*len(gv),
            fill="toself", fillcolor="rgba(206,147,216,0.10)",
            line=dict(color="rgba(0,0,0,0)"), showlegend=False), row=3, col=1)

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=60, r=20, t=80, b=80), height=640,
        legend=dict(orientation="h", y=-0.14, x=0, font=dict(color="#cccccc"), bgcolor="rgba(0,0,0,0)"),
        title=dict(text=f"{ticker} — HMM Regime Detection + GARCH(1,1) Volatility",
                   font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2, fig.layout.xaxis3]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2, fig.layout.yaxis3]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_residuals(ticker):
    """Residual distribution + QQ plot for ensemble model."""
    _res  = _build_ticker_chart_data(ticker)
    from plotly.subplots import make_subplots
    from scipy import stats as _scs
    resid = np.array(_res["y_test"]) - np.array(_res["y_pred_ensemble"])
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=["Residual Distribution (Ensemble)", "Q-Q Plot vs Normal"],
        horizontal_spacing=0.12)

    # Histogram
    fig.add_trace(go.Histogram(x=resid, nbinsx=60, name="Residuals",
        marker_color="#2196F3", opacity=0.75,
        hovertemplate="Residual: %{x:.5f}<br>Count: %{y}<extra></extra>"), row=1, col=1)
    _xr = np.linspace(resid.min(), resid.max(), 200)
    _kde = _scs.norm.pdf(_xr, resid.mean(), resid.std()) * len(resid) * (resid.max()-resid.min())/60
    fig.add_trace(go.Scatter(x=_xr, y=_kde, name="Normal fit",
        line=dict(color="#FFB300", width=2, dash="dash")), row=1, col=1)

    # QQ
    qq = _scs.probplot(resid, dist="norm")
    theoretical = qq[0][0]; sample = qq[0][1]
    fig.add_trace(go.Scatter(x=theoretical, y=sample, mode="markers", name="QQ",
        marker=dict(color="#CE93D8", size=4, opacity=0.7),
        hovertemplate="Theoretical: %{x:.3f}<br>Sample: %{y:.5f}<extra></extra>"), row=1, col=2)
    _qmin, _qmax = theoretical.min(), theoretical.max()
    fig.add_trace(go.Scatter(x=[_qmin, _qmax], y=[_qmin*qq[1][0]+qq[1][1], _qmax*qq[1][0]+qq[1][1]],
        mode="lines", name="Perfect normal", line=dict(color="#FFB300", dash="dash", width=1.5)), row=1, col=2)

    skew = float(_scs.skew(resid)); kurt = float(_scs.kurtosis(resid))
    fig.add_annotation(text=f"mean={resid.mean():.5f} | std={resid.std():.5f}<br>skew={skew:.3f} | excess-kurt={kurt:.3f}",
        xref="paper", yref="paper", x=0, y=-0.12, showarrow=False, font=dict(color="#888", size=10))  # FIX B: was -0.18 (outside paper)

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=60, r=20, t=80, b=100), height=500,
        legend=dict(orientation="h", y=-0.28, x=0, font=dict(color="#cccccc"), bgcolor="rgba(0,0,0,0)"),
        title=dict(text=f"{ticker} — Ensemble Residual Distribution & Q-Q Plot",
                   font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        if getattr(ann, "xref", None) == "paper":  # FIX C: skip stats annotation (paper-coord) — keep its custom font
            continue
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_multistep(ticker):
    """Multi-step forecast price curves."""
    if 'multistep_df' not in globals() or multistep_df is None:
        return _empty_fig("Run Step 11 (Multi-Step Forecasting) first")
    df = multistep_df.copy()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df["Actual_Close"], name="Actual Close",
        line=dict(color="#9aa0b0", width=2),
        hovertemplate="%{x|%b %d}<br>Close: %{y:.2f}<extra></extra>"))
    colors = ["#2196F3","#4CAF50","#FF9800","#F44336"]
    for h, color in zip(CONFIG["multistep_days"], colors):
        col = f"Pred_T{h}_price"
        if col not in df.columns:
            continue
        s = df[col].dropna()
        fig.add_trace(go.Scatter(x=s.index, y=s.values, name=f"T+{h}d forecast",
            mode="lines", line=dict(color=color, dash="dash", width=1.6),
            hovertemplate=f"T+{h}d: %{{y:.2f}}<extra></extra>"))
    return _apply_dark(fig, title=f"{ticker} — Multi-Step Price Forecast (T+1/3/5/10 days)", height=500)


def _chart_quant_risk(ticker):
    """Drawdown, ensemble weights, return distribution with VaR/CVaR."""
    _res           = _build_ticker_chart_data(ticker)
    _has_backtest  = (ticker.upper() == CONFIG["ticker"].upper()) and (backtest_df is not None)  # backtest only exists for startup ticker
    _has_raw_preds = (_res["y_pred_ensemble"] is not None and _res["test_dates"] is not None
                      and _res["df_features"] is not None)
    from plotly.subplots import make_subplots

    # Prefer backtest_df returns (include SL/TP/costs); else reconstruct from raw predictions
    if _has_backtest:
        _strat_ret = backtest_df["Strategy"].pct_change().dropna().values
        _bh_ret    = backtest_df["BuyHold"].pct_change().dropna().values
        # x-axis: use Date column if present, else numeric index → converted to string
        _dd_x = (list(backtest_df["Date"].iloc[1:].values)
                 if "Date" in backtest_df.columns
                 else list(backtest_df.index[1:]))
    else:
        _prices = _res["df_features"].loc[_res["test_dates"], "Close"].squeeze().values
        _y_ens  = np.array(_res["y_pred_ensemble"])
        _min_n  = min(len(_prices)-1, len(_y_ens))
        _sig    = np.where(_y_ens[:_min_n] > 0, 1, -1)
        _close  = _prices[1:_min_n+1]; _prev = _prices[:_min_n]
        _strat_ret = (_close - _prev) / (_prev + 1e-10) * _sig
        _bh_ret    = (_close - _prev) / (_prev + 1e-10)
        _dd_x = list(_res["test_dates"][1:_min_n+1])

    fig = make_subplots(rows=3, cols=1,
        subplot_titles=["Drawdown — Strategy vs Buy & Hold",
                        "Data-Driven Ensemble Weights",
                        "Return Distribution with VaR/CVaR"],
        vertical_spacing=0.14, row_heights=[0.38, 0.24, 0.38])

    # Drawdown
    def _dd(r):
        cum = np.cumprod(1+r); peak = np.maximum.accumulate(cum)
        return (cum-peak)/(peak+1e-10)*100
    dd_s = _dd(_strat_ret); dd_b = _dd(_bh_ret)
    dd_x = _dd_x  # set above based on whether backtest_df is available
    fig.add_trace(go.Scatter(x=dd_x, y=list(dd_s), name="Strategy DD",
        fill="tozeroy", fillcolor="rgba(76,175,80,0.18)", line=dict(color="#4CAF50", width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=dd_x, y=list(dd_b), name="Buy & Hold DD",
        fill="tozeroy", fillcolor="rgba(33,150,243,0.12)", line=dict(color="#2196F3", width=1, dash="dash")), row=1, col=1)

    # Weights
    _ew = CONFIG["ensemble_weights"]
    _mnames = list(_ew.keys()); _wvals = list(_ew.values())
    _wcolors = ["#FF9800","#64B5F6","#81C784"][:len(_mnames)]
    fig.add_trace(go.Bar(x=_mnames, y=_wvals, name="Weights",
        marker_color=_wcolors, showlegend=False,
        text=[f"{w:.4f}" for w in _wvals], textposition="outside",
        textfont=dict(color="#cccccc", size=11)), row=2, col=1)
    fig.add_shape(type="line", x0=-0.5, x1=len(_mnames)-0.5, y0=1/3, y1=1/3,
        line=dict(color="#FFB300", dash="dot", width=1.2), row=2, col=1)

    # Return distribution
    _bins = np.histogram(_strat_ret, bins=60)
    fig.add_trace(go.Bar(x=list(0.5*(_bins[1][:-1]+_bins[1][1:])), y=list(_bins[0]),
        name="Returns", marker_color="#2196F3", opacity=0.55, showlegend=False), row=3, col=1)
    for pct, clr, lbl in [(5,"#FFB300","VaR 95%"),(1,"#F44336","VaR 99%")]:
        var = np.percentile(_strat_ret, pct)
        cvar = _strat_ret[_strat_ret<=var].mean()
        fig.add_shape(type="line", x0=var, x1=var, y0=0, y1=1,
            xref="x3", yref="y3 domain",
            line=dict(color=clr, dash="dash", width=1.5))
        fig.add_shape(type="line", x0=cvar, x1=cvar, y0=0, y1=1,
            xref="x3", yref="y3 domain",
            line=dict(color=clr, dash="dot", width=1.2))
        fig.add_annotation(x=var, y=0.02, xref="x3", yref="y3 domain",
            text=f"{lbl}", showarrow=False,
            font=dict(color=clr, size=9), yanchor="bottom")

    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=60, r=20, t=80, b=90), height=720,
        legend=dict(orientation="h", y=-0.12, x=0, font=dict(color="#cccccc"), bgcolor="rgba(0,0,0,0)"),
        title=dict(text=f"{ticker} — Quant Risk: Drawdown · Weights · VaR/CVaR",
                   font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2, fig.layout.xaxis3]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2, fig.layout.yaxis3]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _chart_shap(ticker):
    """SHAP feature importance as a horizontal bar chart."""
    _res = _build_ticker_chart_data(ticker)
    if _res['shap_values'] is None:
        return _empty_fig('SHAP computation failed for this ticker')

    sv        = np.array(_res['shap_values'])
    n_samples, n_feats = sv.shape

    # Guard: SHAP array columns must match feature_cols
    if n_feats != len(feature_cols):
        return _empty_fig(
            f'SHAP shape mismatch: {n_feats} SHAP cols vs '
            f'{len(feature_cols)} feature_cols. '
            f'Re-run Step 5 to retrain with current feature_cols.'
        )

    mean_abs = np.abs(sv).mean(axis=0)
    top_n    = min(30, len(mean_abs))
    idx      = np.argsort(mean_abs)[-top_n:][::-1]
    feats    = [feature_cols[i] for i in idx]
    vals     = mean_abs[idx]
    palette  = ['#CE93D8' if v == vals[0] else '#2196F3' for v in vals]

    fig = go.Figure(go.Bar(
        x=list(vals), y=feats, orientation='h',
        marker_color=palette, opacity=0.85,
        hovertemplate='%{y}<br>Mean |SHAP|: %{x:.5f}<extra></extra>',
    ))
    fig.update_layout(yaxis=dict(autorange='reversed'))
    return _apply_dark(
        fig,
        title=f'{ticker} — Top {top_n} SHAP Feature Importances (XGBoost)',
        height=max(500, top_n * 22),
    )

def _chart_lstm_training(ticker):
    """LSTM training vs validation loss curves."""
    if 'history' not in globals() or history is None:
        return _empty_fig("Run Step 7 (LSTM) first")
    from plotly.subplots import make_subplots
    ep = list(range(1, len(history.history["loss"])+1))
    best = int(np.argmin(history.history["val_loss"])) + 1
    fig = make_subplots(rows=1, cols=2,
        subplot_titles=["Train vs Validation Loss (Huber)", "Train vs Validation MAE"],
        horizontal_spacing=0.12)

    fig.add_trace(go.Scatter(x=ep, y=history.history["loss"], name="Train Loss",
        line=dict(color="#2196F3", width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=ep, y=history.history["val_loss"], name="Val Loss",
        line=dict(color="#F44336", width=2, dash="dash")), row=1, col=1)
    fig.add_shape(type="line", x0=best, x1=best, y0=0, y1=1,
        xref="x", yref="y domain",
        line=dict(color="#FFB300", dash="dot", width=1.5))
    fig.add_annotation(x=best, xref="x", y=0.95, yref="y domain",
        text=f"Best ({best})", font=dict(color="#FFB300", size=10), showarrow=False, yanchor="top")

    fig.add_trace(go.Scatter(x=ep, y=history.history["mae"], name="Train MAE",
        line=dict(color="#4CAF50", width=2)), row=1, col=2)
    fig.add_trace(go.Scatter(x=ep, y=history.history["val_mae"], name="Val MAE",
        line=dict(color="#FF9800", width=2, dash="dash")), row=1, col=2)
    fig.add_shape(type="line", x0=best, x1=best, y0=0, y1=1,
        xref="x2", yref="y2 domain",
        line=dict(color="#FFB300", dash="dot", width=1.5))

    gap = np.array(history.history["val_loss"]) - np.array(history.history["loss"])
    overfit_flag = "  ⚠️ overfit signal" if gap[-1] > gap[0] * 2 else "  ✅ ok"
    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        font=dict(color="#cccccc", family="JetBrains Mono, monospace"),
        margin=dict(l=60, r=20, t=80, b=90), height=480,
        legend=dict(orientation="h", y=-0.2, x=0, font=dict(color="#cccccc"), bgcolor="rgba(0,0,0,0)"),
        title=dict(text=f"{ticker} — LSTM Training Diagnostics{overfit_flag}",
                   font=dict(color="#ffffff", size=13), x=0),
    )
    for ann in fig.layout.annotations:
        ann.font.color = "#aaaacc"; ann.font.size = 11
    for ax in [fig.layout.xaxis, fig.layout.xaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    for ax in [fig.layout.yaxis, fig.layout.yaxis2]:
        ax.update(gridcolor="#1a1a2e", tickfont=dict(color="#aaaacc"))
    return fig


def _empty_fig(msg="No data"):
    fig = go.Figure()
    fig.update_layout(
        paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
        height=400, margin=dict(l=20, r=20, t=60, b=20),
        font=dict(color="#888"),
        annotations=[dict(text=msg, showarrow=False, font=dict(color="#888", size=13))],
    )
    return fig


# ── Chart registry: label → builder function ──────────────────────────────────
CHART_OPTIONS = [
    "Prediction Forecast",
    "Model Comparison (RMSE · R² · Dir Acc)",
    "Accuracy Dashboard (4-metric)",
    "Walk-Forward Drift",
    "HMM Regime + GARCH Volatility",
    "Residual Distribution & Q-Q Plot",
    "Multi-Step Price Forecast",
    "Quant Risk (Drawdown · VaR · Weights)",
    "SHAP Feature Importance",
    "LSTM Training Curves",
]


def build_quant_chart(chart_name: str, ticker_input: str) -> go.Figure:
    """Route a chart name to its builder. All builders safe-fail with _empty_fig."""
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    try:
        if chart_name == "Prediction Forecast":
            r = predict_tomorrow(ticker)
            return build_prediction_chart(r)
        elif chart_name == "Model Comparison (RMSE · R² · Dir Acc)":
            return _chart_model_comparison(ticker)
        elif chart_name == "Accuracy Dashboard (4-metric)":
            return _chart_accuracy_dashboard(ticker)
        elif chart_name == "Walk-Forward Drift":
            return _chart_walkforward(ticker)
        elif chart_name == "HMM Regime + GARCH Volatility":
            return _chart_hmm_garch(ticker)
        elif chart_name == "Residual Distribution & Q-Q Plot":
            return _chart_residuals(ticker)
        elif chart_name == "Multi-Step Price Forecast":
            return _chart_multistep(ticker)
        elif chart_name == "Quant Risk (Drawdown · VaR · Weights)":
            return _chart_quant_risk(ticker)
        elif chart_name == "SHAP Feature Importance":
            return _chart_shap(ticker)
        elif chart_name == "LSTM Training Curves":
            return _chart_lstm_training(ticker)
        else:
            return _empty_fig("Select a chart from the menu")
    except Exception as e:
        import traceback
        return _empty_fig(f"{e}\n{traceback.format_exc()[:300]}")


# ─────────────────────────────────────────────────────────────────────────────
# PREDICTION function (for Live & Predict tab)
# ─────────────────────────────────────────────────────────────────────────────
def gradio_predict(ticker_input: str):
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    _current_ticker["value"] = ticker

    try:
        r   = predict_tomorrow(ticker)
        sym = r["symbol"]
        is_bull      = "BULL" in r["signal"]
        signal_color = "#4CAF50" if is_bull else "#F44336"
        pct_sign     = "+" if r["pct_change"] >= 0 else ""
        signal_label = "BULLISH" if is_bull else "BEARISH"

        model_rows = ""
        for label, key, color in [
            ("XGBoost",       "xgboost",       "#FF9800"),
            ("LightGBM",      "lightgbm",      "#64B5F6"),
            ("Random Forest", "random_forest", "#81C784"),
        ]:
            diff     = r[key] - r["live_price"]
            diff_pct = diff / max(r["live_price"], 0.01) * 100
            d_sign   = "+" if diff >= 0 else ""
            d_color  = "#4CAF50" if diff >= 0 else "#F44336"
            model_rows += f"""
  <tr style='border-bottom:1px solid #1a1a2e;'>
    <td style='padding:7px 10px;color:#aaaacc;font-size:12px;'>{label}</td>
    <td style='padding:7px 10px;color:{color};font-weight:600;'>{sym}{r[key]:.2f}</td>
    <td style='padding:7px 10px;color:{d_color};font-size:11px;'>{d_sign}{diff_pct:.2f}% vs live</td>
  </tr>"""

        sentiment_row = ""
        if "ensemble_sentiment_adjusted" in r:
            sent_score = r.get("sentiment_score", 0.0)
            sent_tilt  = r.get("sentiment_tilt_pct", 0.0)
            sent_color = "#4CAF50" if sent_score >= 0 else "#F44336"
            sentiment_row = f"""
  <tr style='border-bottom:1px solid #1a1a2e;'>
    <td style='padding:7px 10px;color:#aaaacc;font-size:12px;'>+ Sentiment</td>
    <td style='padding:7px 10px;color:#CE93D8;font-weight:600;'>{sym}{r['ensemble_sentiment_adjusted']:.2f}</td>
    <td style='padding:7px 10px;color:{sent_color};font-size:11px;'>score {sent_score:+.2f}, tilt {sent_tilt:+.3f}%</td>
  </tr>"""

        live_diff     = r["ensemble"] - r["live_price"]
        live_diff_pct = live_diff / max(r["live_price"], 0.01) * 100
        ld_sign       = "+" if live_diff >= 0 else ""
        ld_color      = "#4CAF50" if live_diff >= 0 else "#F44336"

        html = f"""<div class='pred-card'>
  <div style='display:flex;align-items:baseline;gap:10px;margin-bottom:16px;flex-wrap:wrap;'>
    <span style='font-size:13px;font-weight:700;color:{signal_color};letter-spacing:0.1em;'>{signal_label}</span>
    <span style='font-size:13px;color:#888;'>&#8594;</span>
    <span style='font-size:22px;font-weight:700;color:white;'>{sym}{r['ensemble']:.2f}</span>
    <span style='font-size:14px;color:{signal_color};'>({pct_sign}{r['pct_change']:.2f}%)</span>
    <span style='font-size:11px;color:#777;margin-left:auto;'>{r['ticker']}</span>
  </div>
  <table style='width:100%;border-collapse:collapse;'>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <td style='padding:7px 10px;color:#aaaacc;font-size:12px;'>Live price</td>
      <td style='padding:7px 10px;color:white;font-weight:600;'>{sym}{r['live_price']:.2f}</td>
      <td style='padding:7px 10px;color:#F44336;font-size:11px;'>{r.get('change_pct_today',0):+.2f}% today</td>
    </tr>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <td style='padding:7px 10px;color:#aaaacc;font-size:12px;'>Last close</td>
      <td style='padding:7px 10px;color:#cccccc;'>{sym}{r['current_price']:.2f}</td>
      <td style='padding:7px 10px;color:#777;font-size:11px;'>model base</td>
    </tr>
    {model_rows}
    <tr style='background:#0a0a1a;'>
      <td style='padding:9px 10px;color:#ffffff;font-size:13px;font-weight:600;'>Ensemble</td>
      <td style='padding:9px 10px;color:{signal_color};font-size:16px;font-weight:700;'>{sym}{r['ensemble']:.2f}</td>
      <td style='padding:9px 10px;color:{ld_color};font-size:12px;'>{ld_sign}{live_diff_pct:.2f}% vs live</td>
    </tr>
    {sentiment_row}
  </table>
  <div style='margin-top:12px;font-size:10px;color:#666;border-top:1px solid #1a1a2e;padding-top:8px;'>
    Data: {r['last_data_date']} &#124; Market: {r['market_state']}
    &#124; Live injected: {r['live_injected']}
    &#124; For: {r['prediction_date']}
    &#124; Educational only
  </div>
</div>"""
        chart = build_prediction_chart(r)
        return html, chart

    except Exception as e:
        import traceback
        err_html = f"""<div class='pred-card' style='border-color:#3a1a1a;'>
  <div style='color:#F44336;'>Error: {e}</div>
  <pre style='color:#888;font-size:10px;margin-top:8px;white-space:pre-wrap;'>{traceback.format_exc()}</pre>
</div>"""
        return err_html, _empty_fig("Prediction failed")


# ─────────────────────────────────────────────────────────────────────────────
# AGENT PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def _agent_section(title: str, color: str, content: str) -> str:
    return f"""<div class='agent-section'>
  <div class='agent-section-title' style='color:{color};'>{title}</div>
  <div style='color:#cccccc;'>{content if content else '<span style="color:#555;">No output.</span>'}</div>
</div>"""


def run_agents(ticker_input: str):
    ticker = (ticker_input or "").strip().upper() or CONFIG["ticker"]
    _EMPTY = "<span style='color:#555;'>—</span>"

    if not KEYS.get("groq"):
        msg = _agent_section("NO GROQ KEY", "#FFB300",
                              "Add GROQ_API_KEY to Colab Secrets to enable agent reasoning.")
        return msg, msg, msg, msg, msg, msg

    try:
        ml_result = predict_tomorrow(ticker)
        final = trading_graph.invoke({
            "ticker":           ticker,
            "ml_signal":        ml_result,
            "shap_features":    list(zip(feature_cols, shap_values[-1].tolist())),
            "news_sentiment":   float(ml_result.get("sentiment_score", 0.0)),  # FIX: was frozen global sentiment_report (always CONFIG['ticker']); now uses per-ticker live sentiment already fetched by predict_tomorrow(ticker) above
            "technical_report": "",
            "sentiment_report": "",
            "bull_case":        "",
            "bear_case":        "",
            "final_decision":   "",
        })

        dec = final["final_decision"].upper()
        dec_color = "#4CAF50" if "BUY" in dec else ("#F44336" if "SELL" in dec else "#FFB300")
        verdict   = "BUY" if "BUY" in dec else ("SELL" if "SELL" in dec else "HOLD")

        decision_html = f"""<div class='agent-section' style='border-color:{dec_color};'>
  <div style='font-size:11px;font-weight:700;letter-spacing:0.12em;color:{dec_color};margin-bottom:10px;'>
    PORTFOLIO MANAGER — FINAL DECISION
  </div>
  <div style='font-size:28px;font-weight:700;color:{dec_color};margin-bottom:10px;'>{verdict}</div>
  <div style='color:#cccccc;font-size:13px;line-height:1.8;'>{final['final_decision']}</div>
</div>"""

        technical_html = _agent_section("&#128200; TECHNICAL ANALYST",  "#64B5F6", final.get("technical_report", _EMPTY))
        sentiment_html = _agent_section("&#128248; SENTIMENT ANALYST",  "#FFB300", final.get("sentiment_report", _EMPTY))
        bull_html      = _agent_section("&#9650; BULL RESEARCHER",      "#4CAF50", final.get("bull_case",        _EMPTY))
        bear_html      = _agent_section("&#9660; BEAR RESEARCHER",      "#F44336", final.get("bear_case",        _EMPTY))
        portfolio_html = _agent_section("&#127959; PORTFOLIO MANAGER",  "#CE93D8", final.get("final_decision",   _EMPTY))

        return decision_html, technical_html, sentiment_html, bull_html, bear_html, portfolio_html

    except Exception as e:
        import traceback
        err = _agent_section("ERROR", "#F44336",
            f"{e}<br><pre style='font-size:10px;color:#888;'>{traceback.format_exc()}</pre>")
        return err, err, err, err, err, err


# ─────────────────────────────────────────────────────────────────────────────
# NSE WATCHLIST
# ─────────────────────────────────────────────────────────────────────────────
def refresh_indian_watchlist(_=None) -> str:
    tickers = [
        "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS",
        "WIPRO.NS", "ADANIENT.NS", "BAJFINANCE.NS", "ICICIBANK.NS",
    ]
    rows = ""
    for t in tickers:
        try:
            info = get_live_price(t)
        except Exception as e:
            info = {"ticker": t, "error": f"fetch failed — {e}"}
        if info.get("error"):
            rows += f"<tr><td colspan='5' style='padding:6px 8px;color:#F44336;font-size:11px;'>{t}: {info['error']}</td></tr>"
            continue
        sign  = "+" if info["change"] >= 0 else ""
        clr   = "#4CAF50" if info["change"] >= 0 else "#F44336"
        name  = t.replace(".NS","").replace(".BO","")
        price = info["price"]
        sym   = "&#8377;"
        price_flag = " &#9888;" if (price < 50 and info["currency"] == "INR") else ""
        rows += f"""<tr style='border-bottom:1px solid #0f0f1e;'>
  <td style='padding:6px 8px;color:#cccccc;font-size:12px;'>{name}</td>
  <td style='padding:6px 8px;color:white;font-weight:600;font-size:12px;'>{sym}{price:,.2f}{price_flag}</td>
  <td style='padding:6px 8px;color:{clr};font-size:12px;'>{sign}{info['change']:.2f}</td>
  <td style='padding:6px 8px;color:{clr};font-size:11px;'>({sign}{info['change_pct']:.2f}%)</td>
  <td style='padding:6px 8px;color:#888;font-size:10px;'>{info.get("market_state","")}</td>
</tr>"""

    now = datetime.now().strftime("%H:%M:%S")
    return f"""<div class='watch-card'>
  <div style='font-size:11px;color:#8a8aaa;letter-spacing:0.1em;margin-bottom:12px;'>
    NSE LIVE &nbsp;&#124;&nbsp; auto-refresh 30s &nbsp;&#124;&nbsp; {now}
  </div>
  <table style='width:100%;border-collapse:collapse;'>
    <tr style='border-bottom:1px solid #1a1a2e;'>
      <th style='padding:5px 8px;color:#aaaacc;font-size:10px;text-align:left;letter-spacing:0.08em;'>STOCK</th>
      <th style='padding:5px 8px;color:#aaaacc;font-size:10px;text-align:left;letter-spacing:0.08em;'>PRICE</th>
      <th style='padding:5px 8px;color:#aaaacc;font-size:10px;text-align:left;letter-spacing:0.08em;'>CHG</th>
      <th style='padding:5px 8px;color:#aaaacc;font-size:10px;text-align:left;letter-spacing:0.08em;'>%</th>
      <th style='padding:5px 8px;color:#aaaacc;font-size:10px;text-align:left;letter-spacing:0.08em;'>STATUS</th>
    </tr>
    {rows}
  </table>
  <div style='font-size:10px;color:#555;margin-top:8px;'>
    Delayed ~15 min. NSE closes 15:30 IST. &#9888; = possible USD data leak.
  </div>
</div>"""


# ─────────────────────────────────────────────────────────────────────────────
# GRADIO LAYOUT v7
# ─────────────────────────────────────────────────────────────────────────────
LOADING_SPINNER = """<div style='background:#0d0d1a;border:1px solid #1e1e3a;border-radius:12px;
padding:24px;text-align:center;font-family:monospace;color:#888;'>
  ⟳ Running pipeline...
</div>"""

EMPTY_AGENT = "<div style='color:#555;font-family:monospace;font-size:12px;padding:10px;'>Click Run Agents to populate this section.</div>"

with gr.Blocks(css=CUSTOM_CSS, title="Stock ML v7 — Quant Neural Stack") as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div style='padding:20px 0 8px;'>
      <div style='font-family:monospace;font-size:11px;color:#8a8aaa;letter-spacing:0.15em;margin-bottom:6px;'>
        ML PIPELINE v7 — QUANT+ NEURAL STACK
      </div>
      <div style='font-size:22px;font-weight:700;color:#ffffff;margin-bottom:4px;'>
        Stock Price Prediction
      </div>
      <div style='font-size:12px;color:#888;font-family:monospace;'>
        XGBoost &middot; LightGBM &middot; Random Forest &middot; LSTM &middot; Neural Meta-Learner
        &nbsp;&nbsp;|&nbsp;&nbsp;
        HMM Regimes &middot; GARCH(1,1) &middot; Avellaneda-Stoikov
      </div>
    </div>
    """)

    # ── Shared ticker input ────────────────────────────────────────────────────
    with gr.Row():
        tb = gr.Textbox(
            placeholder="AAPL  |  RELIANCE.NS  |  TCS.NS  |  MSFT  |  PLTR",
            value=CONFIG["ticker"],
            label="Ticker (shared across all tabs)",
            scale=5,
        )
        predict_btn = gr.Button("▶ Run Prediction", variant="primary", scale=1)

    with gr.Tabs():

        # ── TAB 1: Live & Predict ──────────────────────────────────────────────
        with gr.Tab("📈 Live & Predict"):
            with gr.Row():
                with gr.Column(scale=1, min_width=280):
                    gr.HTML("<div class='section-label'>Live Price</div>")
                    live_html = gr.HTML()
                    gr.HTML("<div class='section-label'>Tomorrow's Forecast</div>")
                    pred_html = gr.HTML()
                with gr.Column(scale=2):
                    gr.HTML("<div class='section-label'>Prediction Chart — by Model</div>")
                    pred_chart = gr.Plot(show_label=False, container=False)

        # ── TAB 2: Quant Charts (dropdown → single full-width chart) ──────────
        with gr.Tab("📊 Quant Charts"):
            gr.HTML("""
            <div style='padding:10px 0 4px;font-family:monospace;font-size:11px;color:#8a8aaa;letter-spacing:0.1em;'>
              SELECT A CHART — rendered full-width so the Plotly toolbar never overlaps content.
              Charts use data from whichever ticker is in the input box above.
            </div>
            """)
            with gr.Row():
                chart_dropdown = gr.Dropdown(
                    choices=CHART_OPTIONS,
                    value=CHART_OPTIONS[0],
                    label="Chart",
                    elem_classes=["chart-selector"],
                    scale=3,
                )
                chart_btn = gr.Button("Load Chart", variant="secondary", scale=1)

            # Full-width single chart — no siblings to the right, so toolbar is never clipped
            quant_chart = gr.Plot(show_label=False, container=False)

        # ── TAB 3: Agent Analysis ─────────────────────────────────────────────
        with gr.Tab("🤖 Agent Analysis"):
            gr.HTML("<div class='section-label' style='margin-bottom:8px;'>AI Agent Reports</div>")

            # Run button is INSIDE the tab — unambiguous, no shared-header confusion
            agent_btn = gr.Button("▶ Run Agents (requires GROQ_API_KEY)", variant="secondary",
                                   elem_classes=["agent-run-btn"])

            gr.HTML("<div style='height:10px;'></div>")

            # Decision banner — always visible, populated after run
            agent_decision_html = gr.HTML(EMPTY_AGENT)

            with gr.Accordion("📈 Technical Analyst", open=False):
                agent_technical_html = gr.HTML(EMPTY_AGENT)

            with gr.Accordion("📰 Sentiment Analyst", open=False):
                agent_sentiment_html = gr.HTML(EMPTY_AGENT)

            with gr.Accordion("▲ Bull Researcher", open=False):
                agent_bull_html = gr.HTML(EMPTY_AGENT)

            with gr.Accordion("▼ Bear Researcher", open=False):
                agent_bear_html = gr.HTML(EMPTY_AGENT)

            with gr.Accordion("🏛 Portfolio Manager (full reasoning)", open=False):
                agent_portfolio_html = gr.HTML(EMPTY_AGENT)

        # ── TAB 4: NSE Watchlist ───────────────────────────────────────────────
        with gr.Tab("🇮🇳 NSE Watchlist"):
            gr.HTML("<div class='section-label'>NSE Live Prices</div>")
            watch_html = gr.HTML()

    gr.HTML("""
    <div style='font-size:10px;color:#555;font-family:monospace;
                border-top:1px solid #0f0f1e;margin-top:16px;padding-top:10px;'>
      Educational purposes only. Not financial advice.
      Prices delayed ~15 min (yfinance free tier).
      Sentiment overlay is capped and shown separately — not part of the trained ensemble.
    </div>
    """)

    # ── Event wiring ──────────────────────────────────────────────────────────

    # Prediction button: show spinner instantly, then run
    predict_btn.click(
        fn=lambda: (LOADING_SPINNER, None),
        inputs=None, outputs=[pred_html, pred_chart], queue=False,
    ).then(
        fn=gradio_predict, inputs=tb, outputs=[pred_html, pred_chart],
    )

    # Chart dropdown + button: load selected chart
    chart_btn.click(
        fn=build_quant_chart,
        inputs=[chart_dropdown, tb],
        outputs=quant_chart,
    )
    # Auto-load when dropdown changes
    chart_dropdown.change(
        fn=build_quant_chart,
        inputs=[chart_dropdown, tb],
        outputs=quant_chart,
    )
    # Re-render current chart when ticker changes so all charts update automatically
    tb.change(
        fn=build_quant_chart,
        inputs=[chart_dropdown, tb],
        outputs=quant_chart,
    )

    # Agent button — wired to outputs in this tab only
    agent_btn.click(
        fn=lambda: tuple([LOADING_SPINNER] * 6),
        inputs=None,
        outputs=[agent_decision_html, agent_technical_html,
                 agent_sentiment_html, agent_bull_html,
                 agent_bear_html,     agent_portfolio_html],
        queue=False,
    ).then(
        fn=run_agents,
        inputs=tb,
        outputs=[agent_decision_html, agent_technical_html,
                 agent_sentiment_html, agent_bull_html,
                 agent_bear_html,     agent_portfolio_html],
    )

    # Live ticker: update on ticker change + timer
    tb.change(fn=refresh_live_ticker, inputs=tb, outputs=live_html)
    tb.change(fn=_sync_state, inputs=tb, outputs=None, queue=False)

    try:
        timer = gr.Timer(30)
        timer.tick(fn=_timer_tick,              inputs=None, outputs=live_html)
        timer.tick(fn=refresh_indian_watchlist, inputs=None, outputs=watch_html)
        print("Timer enabled (30s auto-refresh)")
    except AttributeError:
        with gr.Row():
            gr.Button("Refresh Price").click(refresh_live_ticker, tb, live_html)
            gr.Button("Refresh Watchlist").click(refresh_indian_watchlist, None, watch_html)

    # Load defaults on startup
    demo.load(fn=lambda: refresh_live_ticker(CONFIG["ticker"]), inputs=None, outputs=live_html)
    demo.load(fn=refresh_indian_watchlist, inputs=None, outputs=watch_html)
    demo.load(fn=lambda: build_quant_chart(CHART_OPTIONS[0], CONFIG["ticker"]),
              inputs=None, outputs=quant_chart)

LAUNCH_PUBLIC_SHARE = True
print("Launching Gradio v7...")
demo.launch(share=LAUNCH_PUBLIC_SHARE, debug=False)

6.19.0
Timer enabled (30s auto-refresh)
Launching Gradio v7...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3ea10d764de866a34a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 💾 Step 17 — Save All Artifacts

In [38]:
import zipfile, shutil
import json

# Ensure 'tomorrow' is defined from Step 13
tomorrow = predict_tomorrow(CONFIG['ticker'])

# Ensure 'llm_report' is defined from Step 15
# This relies on metrics_df, sentiment_report, tomorrow, wf_results
llm_report = run_groq_agent(CONFIG['ticker'], metrics_df, sentiment_report, tomorrow, wf_results)

results_df=pd.DataFrame({'Actual':y_test,'Linear':y_pred_lr,'Ridge':y_pred_ridge,
    'XGBoost':y_pred_xgb,'LightGBM':y_pred_lgb,'Random_Forest':y_pred_rf,
    'Ensemble':y_pred_ensemble},index=test_dates)
results_df.to_csv('predictions.csv')
with open('tomorrow_prediction.json','w') as f: json.dump(tomorrow,f,indent=2)
with open('sentiment_report.json','w') as f: json.dump(sentiment_report,f,indent=2)
metrics_df.reset_index().to_json('metrics.json',orient='records',indent=2)
wf_results.to_csv('walkforward_results.csv',index=False)
with open('analysis_report.txt','w') as f: f.write(llm_report)

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    OUT='/content/drive/MyDrive/StockML_v4_Outputs/'
    os.makedirs(OUT,exist_ok=True)
    ALL_ARTIFACTS_DRIVE = [
        'predictions.csv','metrics.json','tomorrow_prediction.json',
        'sentiment_report.json','walkforward_results.csv','analysis_report.txt',
        'shap_summary.png','shap_beeswarm.png','model_comparison.png',
        'xgboost_stock_model.json','lightgbm_stock_model.pkl',
        'random_forest_stock_model.pkl','lstm_stock_model.keras',
        f"backtest_Ensemble.html", 'multistep_forecast.html',
        'avellaneda_stoikov_spread.html', 'risk_metrics.json',
        'stationarity_diagnostics.png',
        'hmm_regime_garch.png',
        'lstm_training_curve.png',
        'walkforward_drift.png',
        'residual_distribution.png',
        'quant_risk_charts.png',
    ]
    for fn in ALL_ARTIFACTS_DRIVE:  # FIX L3: consolidated list includes backtest + SHAP beeswarm
        if os.path.exists(fn): shutil.copy(fn,OUT+fn); print(f'  \u2705 {fn}')
    print(f'\n\u2705 All files saved to Google Drive: {OUT}')
except Exception as e:
    print(f'\u26a0\ufe0f  Drive backup skipped: {e}')

# Zip for download
zn=f"{CONFIG['ticker']}_ML_v4_outputs.zip"
with zipfile.ZipFile(zn,'w',zipfile.ZIP_DEFLATED) as zf:
    # FIX L3: ALL_ARTIFACTS list used by both Drive copy and ZIP
    ALL_ARTIFACTS = [
        'predictions.csv','metrics.json','tomorrow_prediction.json',
        'sentiment_report.json','walkforward_results.csv','analysis_report.txt',
        'shap_summary.png','shap_beeswarm.png','model_comparison.png',
        'xgboost_stock_model.json','lightgbm_stock_model.pkl',
        'random_forest_stock_model.pkl','lstm_stock_model.keras',
        f"backtest_Ensemble.html", 'multistep_forecast.html',
        'avellaneda_stoikov_spread.html', 'risk_metrics.json',
        'stationarity_diagnostics.png',
        'hmm_regime_garch.png',
        'lstm_training_curve.png',
        'walkforward_drift.png',
        'residual_distribution.png',
        'quant_risk_charts.png',
    ]
    for fn in ALL_ARTIFACTS:
        if os.path.exists(fn): zf.write(fn)

print(f"\n{'='*65}")
print(f"  \u2705 PIPELINE COMPLETE \u2014 {CONFIG['ticker']} v4.1")
print('='*65)
print(f"  Best RMSE    : {metrics_df['RMSE'].idxmin()} = {metrics_df['RMSE'].min():.6f}")
print(f"  Best DirAcc  : {metrics_df['Directional Acc %'].idxmax()} = {metrics_df['Directional Acc %'].max():.1f}%")
print(f"  Signal       : {tomorrow['signal']}")
print(f"  Ensemble     : {tomorrow['symbol']}{tomorrow['ensemble']:.2f} ({tomorrow['pct_change']:+.2f}%)")
print(f"  Live Price   : {tomorrow['symbol']}{tomorrow['live_price']:.2f} ({tomorrow['market_state']})")
print('='*65)
try:
    from google.colab import files; files.download(zn)
except: print(f'Download manually: {zn}')

  ⚡ Cache hit for AAPL (272s remaining)
  ⚡ Cache hit for AAPL (260s remaining)
  📡 AAPL: source=Finnhub price=USD 293.08
  📡 RELIANCE.NS: source=yfinance_intraday price=INR 1326.2
  📡 TCS.NS: source=yfinance_intraday price=INR 2141.0
  📡 INFY.NS: source=yfinance_intraday price=INR 1054.7
  📡 HDFCBANK.NS: source=yfinance_intraday price=INR 798.2
  📡 WIPRO.NS: source=yfinance_intraday price=INR 174.45
  📡 ADANIENT.NS: source=yfinance_intraday price=INR 3083.2
  📡 BAJFINANCE.NS: source=yfinance_intraday price=INR 996.4
  📡 ICICIBANK.NS: source=yfinance_intraday price=INR 1388.4
  📡 AAPL: source=Finnhub price=USD 293.08
  📡 RELIANCE.NS: source=yfinance_intraday price=INR 1326.3
  📡 TCS.NS: source=yfinance_intraday price=INR 2140.1
  📡 INFY.NS: source=yfinance_intraday price=INR 1054.1
  📡 HDFCBANK.NS: source=yfinance_intraday price=INR 798.1
  📡 WIPRO.NS: source=yfinance_intraday price=INR 174.34
  📡 ADANIENT.NS: source=yfinance_intraday price=INR 3082.8
  📡 BAJFINANCE.NS: source=yfinance

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  📡 AAPL: source=Finnhub price=USD 293.08
